# Ad Concept Generation

In [12]:
brand_specific_information="""


"""

In [13]:
from typing import List, Optional, Dict, Any
from utils.llm import get_llm_model
from pydantic import BaseModel, Field
import json
import os
from dotenv import load_dotenv


load_dotenv()

llm_client = get_llm_model("gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))


class AdConcept(BaseModel):
    """Individual Ad Concept with all required fields"""
    title: str = Field(description="Catchy title for the ad concept")
    one_line_summary: str = Field(description="Brief one-line summary of the ad concept")
    story: str = Field(description="Detailed narrative of the ad in paragraph form")
    visual_flow:str = Field(
        description="Dictionary containing visual sequences like Opening, Sequence, Location scenes, Close-up, etc."
    )
    tagline: str = Field(description="Memorable tagline for the ad")
    key_message: str = Field(description="Core message the ad conveys")
    key_features: List[str] = Field(description="List of product features highlighted in the ad")
    tone: str = Field(description="Tone of the ad (e.g., inspirational, humorous, emotional)")


class AdConceptsResponse(BaseModel):
    """Collection of ad concepts"""
    concepts: List[AdConcept] = Field(description="List of generated ad concepts")


class AdConceptGenerator:
    def __init__(self):
        self.llm = llm_client
        
    def create_ad_concepts_system_prompt(self, brand_info: Dict[str, Any]) -> str:
        """Create system prompt for generating ad concepts"""
        return f"""You are an expert creative director specializing in advertisement concept creation.

Your task is to generate compelling ad concepts based on brand information provided.

Each ad concept MUST include:
1. **Title**: A catchy, memorable title
2. **One-Line Summary**: Brief encapsulation of the concept
3. **Story**: Detailed narrative in paragraph form (5-6 sentences) breifly describing the overall ads concept
4. **Visual Flow**: Key visual sequences broken down as:
   - Opening: Initial scene/shot
   - Sequence: Main action sequences
   - Additional scenes: Location-specific shots
   - Close-up: Important detail shots

5. **Tagline**: Memorable brand tagline
6. **Key Message**: Core message/value proposition
7. **Key Features**: Product features to highlight (list)
8. **Tone**: Overall tone of the ad

Create concepts that are:
- Visually compelling and easy to execute
- Authentic to the brand's voice and values
- Emotionally resonant with the target audience
- Clear in their messaging
- Diverse in approach (different angles, tones, scenarios)

Brand Information:
{json.dumps(brand_info, indent=2)}

Generate creative, diverse concepts that showcase different angles and emotional appeals while staying true to the brand."""

    def generate_ad_concepts(
        self, 
        brand_info: Dict[str, Any], 
        num_concepts: int = 5,
        duration:int=15
    ) -> List[AdConcept]:
        """
        Generate multiple ad concepts based on brand information
        
        Args:
            brand_info: Dictionary containing brand details like:
                - brand_name: str
                - product_name: str
                - product_description: str
                - target_audience: str
                - key_features: List[str]
                - brand_values: List[str]
                - tone_preferences: str
                - campaign_objective: str
                - celebrity_endorser: Optional[str]
                - reference_style: Optional[str]
            num_concepts: Number of ad concepts to generate (default 5)
            
        Returns:
            List of AdConcept objects
        """
        system_prompt = self.create_ad_concepts_system_prompt(brand_info)
        
        user_prompt = f"""Generate {num_concepts} diverse ad concepts for this brand with duration of {duration} sec.

Each concept should take a different creative approach adpting to brand and the sector

Ensure each concept is complete with all required fields and ready for production consideration.

Return the response in valid JSON format matching the AdConceptsResponse schema."""

        try:
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
            ]
            self.llm=self.llm.with_structured_output(AdConceptsResponse)
            ad_concepts_response = self.llm.invoke(messages)

            
            print(f"✓ Successfully generated {len(ad_concepts_response.concepts)} ad concepts")
            return ad_concepts_response.concepts
            
        except Exception as e:
            print(f"Error generating ad concepts: {str(e)}")
            raise

    def save_ad_concepts(
        self, 
        ad_concepts: List[AdConcept], 
        output_file: str, 
        output_dir: str = "projects_data"
    ):
        """Save ad concepts to JSON file"""
        concepts_dict = {
            "ad_concepts": [concept.model_dump() for concept in ad_concepts],
            "total_concepts": len(ad_concepts)
        }
        
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(concepts_dict, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Ad concepts saved to {file_path}")
        return file_path

    def load_ad_concepts(self, file_path: str) -> List[AdConcept]:
        """Load ad concepts from JSON file"""
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        return [AdConcept(**concept) for concept in data['ad_concepts']]

    def display_concepts_summary(self, ad_concepts: List[AdConcept]):
        """Display a summary of generated concepts"""
        print("\n" + "="*80)
        print("GENERATED AD CONCEPTS SUMMARY")
        print("="*80 + "\n")
        
        for idx, concept in enumerate(ad_concepts, 1):
            print(f"\n--- Concept {idx}: {concept.title} ---")
            print(f"Summary: {concept.one_line_summary}")
            print(f"Tone: {concept.tone}")
            print(f"Key Message: {concept.key_message}")
            print(f"Tagline: {concept.tagline}")
            print(f"Story of the Ad {concept.story}")
            print(f"Visual Flow {concept.visual_flow}")
            print("-" * 80)



# if __name__ == "__main__":
#     # Example brand information
#     brand_info = {
#         "brand_name": "Deconstruct",
#         "product_name": "Gel Sunscreen SPF 55",
#         "product_description": "Lightweight, matte finish sunscreen with high SPF protection",
#         "target_audience": "Active individuals, sports enthusiasts, ages 25-45",
#         "key_features": [
#             "SPF 55 protection",
#             "Matte finish - no white cast",
#             "Sweat-resistant",
#             "Non-greasy formula",
#             "Suitable for all skin types"
#         ],
#         "brand_values": ["Performance", "Quality", "Trust", "Innovation"],
#         "tone_preferences": "Confident, aspirational, authentic",
#         "campaign_objective": "Increase brand awareness and position as premium sunscreen choice",
#         "celebrity_endorser": "MS Dhoni",
#         "reference_style": "Documentary-style, authentic moments"
#     }
    
#     # Initialize generator
#     generator = AdConceptGenerator()
    
#     # Generate concepts
#     concepts = generator.generate_ad_concepts(brand_info, num_concepts=4, duration=15)
    
#     # Display summary
#     generator.display_concepts_summary(concepts)
    
#     # Save to file
#     generator.save_ad_concepts(concepts, "ad_concepts_deconstruct.json")

# Shot Level Script Generation:

In [1]:
from typing import List, Optional, Dict, Any
from utils.llm import get_llm_model
from pydantic import BaseModel, Field
import json
import os
from dotenv import load_dotenv


load_dotenv()




class CharacterInfo(BaseModel):
    """Character information extracted from the script"""
    name: str = Field(description="Character name (lowercase)")
    age: Optional[int] = Field(default=None, description="Approximate age of the character")
    role: Optional[str] = Field(default=None, description="Role in the ad (e.g., 'protagonist', 'friend', 'colleague')")
    gender: Optional[str] = Field(default=None, description="Gender of the character")
    overall_description: Optional[str] = Field(default=None, description="Detailed physical and personality description")
    image_path: Optional[str] = None
    reference_description: Optional[str] = None

class LocationInfo(BaseModel):
    """Character information extracted from the script"""
    name: str = Field(description="Location Name")
    overall_description: Optional[str] = Field(default=None, description="Detailed visual Description of the location how it looks what all things it has and key information of the location in detail")
    image_path: Optional[str] = None

class CharacterOutfitInfo(BaseModel):
    outfit: str= Field(description="name of the outfit in lowercase")
    outfit_description :str=Field(description="Detailed description of the outfit")
    image_path: Optional[str] = None

class OutfitMapping(BaseModel):
    character_name: str= Field(description="name of the character in lowercase")
    outfit_name :str=Field(description="name of the outfit in lowercase")


class Shot(BaseModel):
    """Individual shot details with image prompt"""
    shot_no: int = Field(description="Shot number in sequence")
    duration: str = Field(description="Duration of the shot (e.g., '2 seconds', '3 seconds')")
    time_stamp: str = Field(description="Time range in format MM:SS-MM:SS")
    location: str = Field(description="Detailed location description")
    location_name: str = Field(description="Name of the location used in this shot (lowercase, must match location_info)")
    camera_angle: str = Field(description="Camera angle and shot type (e.g., 'Medium close-up', 'Wide shot')")
    visual_description: str = Field(description="Detailed visual description of what's in frame")
    action: str = Field(description="Specific actions happening in the shot")
    objects_props_involved:str=Field(description="Detailed description of the objects involved")
    audio_sfx: str = Field(description="Audio and sound effects")
    dialogue: Optional[str] = Field(default="None", description="Any spoken dialogue in the shot")
    voice_over: Optional[str] = Field(default="None", description="Voice over narration")
    text_overlay: Optional[str] = Field(default="None", description="If present On-screen text or graphics")
    key_focus: str = Field(description="Primary focus or goal of this shot")
    product_image_required: bool = Field(default=False, description="Does this shot require product image")
    
    characters_involved: List[str] = Field(
        default_factory=list, 
        description="List of character names involved in this shot (all lowercase)"
    )
    outfit_character_mapping:List[OutfitMapping]=Field(
        description="List of character names and their outfit mapping"
    )
    
    image_prompt: Optional[str] = Field(default=None, description="Detailed prompt for keyframe image generation")


class ShotScript(BaseModel):
    """Complete shot-level script"""
    ad_title: str = Field(description="Title of the ad concept")
    total_duration: str = Field(description="Total ad duration")
    shots: List[Shot] = Field(description="List of all shots in sequence")
    characters_info: List[CharacterInfo] = Field(
        default_factory=list,
        description="List of all characters with detailed descriptions"
    )
    location_info:List[LocationInfo]=Field(
        default_factory=list,
        description="List of all Location with detailed descriptions"
    )
    character_outfit_info:List[CharacterOutfitInfo]=Field(
     description="List of detailed description of oufit and its name in lowercase"
     )


class ShotScriptGenerator:
    def __init__(self):
        self.llm = get_llm_model("gpt-5", api_key=os.getenv("OPENAI_API_KEY"))
        
    def create_shot_script_system_prompt(self) -> str:
        """Create system prompt for generating shot-level scripts"""
        return """You are an expert film director and scriptwriter specializing in commercial advertisements.

Your task is to convert ad concepts into detailed, shot-by-shot production scripts.

Each shot MUST include these specific columns this is for single shot information:
1. **Shot No**: Sequential number
2. **Duration**: Length of shot (e.g., "2 seconds", "3 seconds")
3. **Time stamp**: Time range (e.g., "00:00-00:02")
4. **Location**: Detailed location description with lighting/atmosphere
5. **Location Name**: EXACT name of the location from location_info list (lowercase) - this will be used to match the location reference image
6. **Camera Angle**: Shot type and angle (e.g., "Medium close-up", "Wide shot", "POV", "Over-the-shoulder")
7. **Visual Description**: Detailed description of what's visible in frame give as many inforamtion as in clear in detailed way
8. **objects_props_involved**:Detailed description of what all objects are present,(for example if bag is present give detailed description of the bag, like CSK cricket kit with blue and yellow in color with logo)
9. **Action**: Specific actions/movements happening
10. **Audio/SFX**: Sound effects and ambient audio
11. **Dialogue**: Any spoken words (use "None" if no dialogue)
12. **Voice Over**: VO narration (use "None" if no VO in this shot)
13. **Text Overlay**: On-screen text/graphics (use "None" if no text)
14. **Key Focus**: Primary purpose/goal of the shot
15. **Product image** Boolean say True when product image required or False if this shot does not contain product image in focus
16. **Characters Involved** (list of character names in lowercase)
17 **Character outfit mapping** (list of character and their outfit mapping)

 example for character outfit mapping ["priya": "priya_casual_wear"] for single shot like wise if we had multiple character include their outfit also ["priya": "priya_office_formal" ,"jay": "jay_casual_wear"]


Guidelines for shot creation:
- Each shot should be 2-5 seconds typically
- Ensure smooth flow and narrative progression
- Include establishing shots, action shots, product shots, and closing shots
- Always End the Video with Product Showcase shot where at the final shot you need to generate with text overlay describing with keyfeatures and tagline or keymessage in this ad
- Timestamps must be sequential and accurate
- Be specific about camera movements, angles, and framing
- Include relevant audio design (SFX, ambient sounds, music cues)
- For Object and Props give as many detailed description in detail and pass all nuance
- Distribute voice over strategically across shots
- Identify key product showcase moments
- Consider pacing and emotional beats
- Ensure visual variety and dynamic composition

Guidelines for outfit generaion and mapping on shot level

    1. **Decide outfit per shot**:
    - For every shot that contains characters, determine what outfit they are wearing.
    - If the same outfit continues across multiple shots, keep the same outfit name.
    - If the outfit changes, assign a *new outfit name*.
    2. **Outfit Naming Convention** (STRICT):
    <character_name_in_lowercase>_<type_of_wear_or_scene_keyword>
    Examples:
    - priya_casual_wear
    - priya_bathroom_morning_fit
    - genie_modern_magic_fit

    3. **Output Format** *(VERY IMPORTANT)*:
    ### A. List of CharacterOutfitInfo (unique outfits only):
    [
    {
    "outfit": "<outfit_name>",
    "outfit_description": "<detailed outfit description>"
    },
    ...
    ]

    ### B. For each shot, provide Outfit Mapping only:
    "outfit_character_mapping": [
    {"character_name": "<character>", "outfit_name": "<outfit_name>"},
    ...
    ]

    4. DO NOT repeat full descriptions inside shot objects — only reference the outfit name.
    5. Ensure consistency: The same outfit name must have the same description everywhere.



 **Character Extraction**:
   - Identify ALL unique characters in the script
   - Provide detailed descriptions including:
     * Name (lowercase)
     * Approximate age
     * Role in the ad
     * Gender
     * Description in description you should mention ethinicity,skin tone,face structure,hair,outfit choose mostly used from script,pose always standing upright, full body view and background white
   - Characters should be consistent across shots

HERE IS THE EXAMPLE OF CHARACTER INFO:
    name:ajay
    age:20
    role:hero
    gender:male
    overall_description: Ethnicity: South Indian, Skin tone: Warm medium brown, golden undertones, healthy complexion,Face: Oval face, expressive almond-shaped dark brown eyes, slightly arched brows, medium lips, natural look,Hair :Long, dark brown hair, Outfit & Styling: Use common in the script, Pose & Expression :Standing straight, front-facing, full body, Lighting & Background :Neutral indoor studio lighting (soft, flattering) Clean white background (AI consistency, no distractions)

**Location Extraction**:
   - Identify ALL unique location in the script
   - Provide detailed descriptions of the location:
     * Name (lowercase)
     * description detailed description of the location in detail like( Location Type: [What place is it]
                                        Design Style & Mood: [Modern / Minimal / Classic / Luxury, emotional feel]
                                        Key Architectural Elements: [Describe surfaces, fixtures, layout]
                                        Color Palette & Materials: [Dominant tones + key materials]
                                        Props & Visual Details: [List clearly visible objects]
                                        Lighting: [Natural or artificial + brightness + source direction]
                                        Atmosphere & Vibe: [Emotional tone or story feeling]
                                        Camera Framing Note: [Wide / Medium / Close / OTS / POV]
                                        )

HERE IS THE EXAMPLE OF LOCATION INFO:
name: hotel(lowercase)
overall_description: Location Type: Hotel Room (Morning),Design Style & Mood: Minimal modern Indian, soft and calm mood,Key Architectural Elements: Light wooden furniture, soft beige curtains, neutral wall tones, Color Palette & Materials: Warm beige, soft white, natural wood, cotton bedding,Props & Visual Details: Cricket duffel bag by bedside, water bottle, sports shoes, framed art on wall,Lighting: Soft natural morning light filtering through curtains, warm gentle shadows


HERE IS THE EXAMPLE OF CHARACTER OUTFIT INFO:
Format for outfit information for the character
outfit_name:priya_casual_outfit(lowercase)
outfit_description:[Outfit Type], Top: [Item, Color, Fabric, Fit], Bottom: [Item, Color, Fabric, Fit], Footwear: [Item, Color, Style], Accessories: [List], Style Inspiration: [Mood/Reference], Condition: [New/Worn], Context: [Scene Usage]

Example Description:
Athleisure casual, Top: performance t-shirt, muted charcoal gray, lightweight stretch fabric, athletic fit; Bottom: navy training joggers, soft knit, slim tapered fit; Footwear: white running shoes, clean minimal design; Accessories: black sports watch; Style Inspiration: disciplined minimal sports aesthetic; Condition: clean and well-kept; Context: early morning pre-match indoor preparation.



Shot types to consider:
- Establishing shots (wide)
- Close-ups (product, face, details)
- Medium shots (action, interaction)
- POV shots (perspective)
- Over-the-shoulder
- Tracking/following shots
- Static vs. dynamic camera work

Your Response should include:

ad_title: title of the ad
total_duration:
shot: list of shots
characters_info:list of detailed description of the characters in this script
location_info: list of detailed information of the location in this script
character_outfit_info list of detailed description of the outfit for all unique outfit
"""

    def generate_shot_script(
        self, 
        ad_concept: Dict[str, Any],
        brand_info:Optional,
        duration: int= 15,
        
    ) -> ShotScript:
        """
        Generate detailed shot-level script from ad concept
        
        Args:
            ad_concept: Dictionary containing ad concept details (can be AdConcept.model_dump())
            target_duration: Target duration for the ad
            
        Returns:
            ShotScript object with complete shot breakdown
        """
        system_prompt = self.create_shot_script_system_prompt()
        
        user_prompt = f"""Convert the following ad concept into a detailed shot-by-shot script.

Ad Concept:
{json.dumps(ad_concept, indent=2)}

Target Duration: {duration} sec

this is the Brand info {json.dumps(brand_info, indent=2)}

Requirements:
- Break down the story into 8-15 shots
- Each shot should advance the narrative
- Include proper establishing shots
- Showcase product features clearly
- Build to the tagline/message
- Ensure timestamps are sequential and accurate
- Be specific about every detail

Follow the visual flow from the concept and expand it into precise, producible shots.
Make sure each shot has ALL required fields filled out properly.

Return a complete shot script in valid JSON format matching the ShotScript schema."""

        try:
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
            ]
            self.llm=self.llm.with_structured_output(ShotScript)
            shot_script = self.llm.invoke(messages)

            print(shot_script)
            
            
            print(f"✓ Successfully generated {len(shot_script.shots)} shots for '{shot_script.ad_title}'")
            return shot_script
            
        except Exception as e:
            print(f"Error generating shot script: {str(e)}")
            raise

    def save_shot_script_json(
        self, 
        shot_script: ShotScript, 
        output_file: str, 
        output_dir: str = "projects_data"
    ):
        """Save shot script to JSON file"""
        script_dict = shot_script.model_dump()
        
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(script_dict, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Shot script saved to {file_path}")
        return file_path

   

    def load_shot_script(self, file_path: str) -> ShotScript:
        """Load shot script from JSON file"""
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        return ShotScript(**data)

    def display_shot_script(self, shot_script: ShotScript):
        """Display formatted shot script"""
        print("\n" + "="*100)
        print(f"SHOT SCRIPT: {shot_script.ad_title}")
        print(f"Total Duration: {shot_script.total_duration}")
        print("="*100 + "\n")

        print("-" * 100)
        print("CHARACTERS:")
        print("-" * 100)
        for char in shot_script.characters_info:
            print(f"\n{char.name.upper()}")
            print(f"  Role: {char.role}")
            print(f"  Age: {char.age}, Gender: {char.gender}")
            print(f"  Description: {char.overall_description}")

        print("LOCATION:")
        print("-" * 100)
        for char in shot_script.characters_info:
            print(f"\n{char.name.upper()}")
            print(f"  Description: {char.overall_description}")
        
        
        for shot in shot_script.shots:
            print(f"\n{'─'*100}")
            print(f"SHOT {shot.shot_no} | {shot.duration} | {shot.time_stamp}")
            print(f"{'─'*100}")
            print(f"📍 Location: {shot.location}")
            print(f"🎥 Camera: {shot.camera_angle}")
            print(f"👁️  Visual: {shot.visual_description}")
            print(f"🎬 Action: {shot.action}")
            print(f"🔊 Audio/SFX: {shot.audio_sfx}")
            if shot.dialogue and shot.dialogue != "None":
                print(f"💬 Dialogue: {shot.dialogue}")
            if shot.voice_over and shot.voice_over != "None":
                print(f"🎙️  Voice Over: {shot.voice_over}")
            if shot.text_overlay and shot.text_overlay != "None":
                print(f"📝 Text Overlay: {shot.text_overlay}")
            print(f"🎯 Key Focus: {shot.key_focus}")
        
        print("\n" + "="*100 + "\n")

    



# if __name__ == "__main__":
#     # Example: Load a previously generated ad concept
#     ad_concept_example = {
#         "title": "The Captain's Pre-Match Ritual",
#         "one_line_summary": "Dhoni's calm morning routine - applying sunscreen is as essential as checking his bat",
#         "story": "In the quiet hours before a crucial match, MS Dhoni follows his meticulous routine. He checks his bat, packs his kit, and applies Deconstruct gel sunscreen with the same calm focus he brings to captaincy. As he walks out to the toss under harsh stadium lights, his face shows no sweat shine - just confidence and preparation.",
#         "visual_flow": {
#             "Opening": "Dhoni in hotel room, early morning light",
#             "Sequence": "Checking bat → packing kit → Applying Deconstruct gel sunscreen calmly",
#             "Stadium": "Walking out to toss under harsh sun, confident and protected",
#             "Close-up": "Face showing no sweat shine, matte finish despite heat"
#         },
#         "voice_over": "Champions prepare for everything. Even the sun.",
#         "tagline": "Dhoni's choice. Captain Cool stays protected.",
#         "key_message": "Preparation and attention to detail, like Dhoni's captaincy style",
#         "key_features": [
#             "SPF 55 protection",
#             "Matte finish",
#             "Sweat-resistant",
#             "No white cast"
#         ],
#         "tone": "Inspirational, authentic",
#     }
    
#     brand_info = {
#         "brand_name": "Deconstruct",
#         "product_name": "Gel Sunscreen SPF 55",
#         "product_description": "Lightweight, matte finish sunscreen with high SPF protection",
#         "target_audience": "Active individuals, sports enthusiasts, ages 25-45",
#         "key_features": [
#             "SPF 55 protection",
#             "Matte finish - no white cast",
#             "Sweat-resistant",
#             "Non-greasy formula",
#             "Suitable for all skin types"
#         ],
#         "brand_values": ["Performance", "Quality", "Trust", "Innovation"],
#         "tone_preferences": "Confident, aspirational, authentic",
#         "campaign_objective": "Increase brand awareness and position as premium sunscreen choice",
#         "celebrity_endorser": "MS Dhoni",
#         "reference_style": "Documentary-style, authentic moments"
#     }
#     # Initialize generator
#     generator = ShotScriptGenerator()
    
#     # Generate shot script
#     shot_script = generator.generate_shot_script(ad_concept_example,brand_info)
    
#     # Display the script
#     generator.display_shot_script(shot_script)
    
#     # Save as JSON
#     generator.save_shot_script_json(shot_script, "shot_script_generated.json")
    


# Outfit Generator

In [2]:
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
import json
import os
from PIL import Image
from google import genai
from google.genai import types
from io import BytesIO
from dotenv import load_dotenv

load_dotenv()


class CharacterOutfitInfo(BaseModel):
    """Outfit information for characters"""
    outfit: str = Field(description="Name of the outfit in lowercase")
    outfit_description: str = Field(description="Detailed description of the outfit")
    image_path: Optional[str] = Field(default=None, description="Path to generated outfit image")


class FullOutfit(BaseModel):
    """Full outfit details for image generation"""
    outfit: str
    outfit_description: str
    image_path: Optional[str] = None


class OutfitGenerator:
    def __init__(self, output_dir: str = "outfit_images"):
        """
        Initialize Outfit Generator
        
        Args:
            output_dir: Directory to store generated outfit images
        """
        
        self.client = genai.Client()
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def create_outfit_prompt(self, outfit: dict) -> str:
        """
        Create detailed prompt for outfit image generation
        
        Args:
            outfit: Dictionary with 'outfit' (name) and 'outfit_description'
        """
        outfit_name = outfit.get('outfit', 'Outfit')
        description = outfit.get('outfit_description', '')
        
        prompt = f"""
Create a highly realistic, professional photograph of a complete outfit on a white background.

OUTFIT NAME: {outfit_name}

OUTFIT DETAILS:
{description}

PHOTOGRAPHY REQUIREMENTS:
- Style: Professional fashion photography, catalog-style
- Layout: Full outfit displayed flat lay OR on invisible mannequin
- Background: Pure white, clean, no shadows
- Lighting: Even, soft studio lighting from multiple angles
- Focus: Crystal clear, every detail visible
- Perspective: Straight-on, front view
- Quality: High resolution, commercial photography standard
- Context: Ready-to-wear presentation

OUTFIT PRESENTATION:
- All clothing items arranged neatly and proportionally
- Natural fabric drape and texture visible
- Colors accurate and vibrant
- Accessories positioned appropriately
- Professional styling that shows how items work together

IMPORTANT:
- NO human model, just the outfit itself
- Clean presentation suitable for e-commerce or fashion catalog
- Show the complete outfit as described
- Maintain realistic fabric textures and colors
- Indian fashion aesthetic where applicable

Create a clean, professional outfit photograph suitable for commercial use.
"""
        return prompt.strip()
    
    def generate_image(self, prompt: str, outfit_id: str) -> Optional[str]:
        """
        Generate outfit image using Gemini
        
        Args:
            prompt: Detailed prompt for image generation
            outfit_id: Unique identifier for the outfit
            
        Returns:
            File path to generated image or None if failed
        """
        try:
            print(f"🎨 Generating outfit image for {outfit_id} using Gemini...")
            
            response = self.client.models.generate_content(
                model="gemini-2.5-flash-image",
                contents=prompt,
                config=types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=types.ImageConfig(
                    aspect_ratio="16:9",
                )
            )
            )
            
            image_saved = False
            filename = f"{outfit_id}_outfit.png"
            filepath = os.path.join(self.output_dir, filename)
            
            if hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        image_data = part.inline_data.data
                        image = Image.open(BytesIO(image_data))
                        image.save(filepath)
                        image_saved = True
                        print(f"✓ Saved outfit image: {filepath}")
                        break
                    elif hasattr(part, 'text') and part.text:
                        print(f"📝 Gemini response: {part.text[:100]}...")
            
            if not image_saved:
                print(f"⚠ No image data generated for {outfit_id}")
                return None
                
            return filepath
            
        except Exception as e:
            print(f"✗ Error generating outfit image for {outfit_id}: {e}")
            return None
    
    def create_placeholder_image(self, outfit_id: str, outfit_name: str) -> str:
        """Create a placeholder image when AI generation fails"""
        try:
            filename = f"{outfit_id}_outfit.png"
            filepath = os.path.join(self.output_dir, filename)
            
            img = Image.new('RGB', (600, 800), color='lightgray')
            
            try:
                from PIL import ImageDraw, ImageFont
                draw = ImageDraw.Draw(img)
                
                try:
                    font = ImageFont.truetype("arial.ttf", 24)
                except:
                    font = ImageFont.load_default()
                
                text = f"{outfit_name}\n(Outfit Placeholder)"
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                
                x = (600 - text_width) // 2
                y = (800 - text_height) // 2
                
                draw.text((x, y), text, fill='black', font=font)
            except:
                pass
            
            img.save(filepath)
            print(f"✓ Created placeholder image: {filepath}")
            return filepath
            
        except Exception as e:
            print(f"✗ Error creating placeholder: {e}")
            return None
    
    def generate_outfit_image(self, outfit: FullOutfit, outfit_id: str = None) -> FullOutfit:
        """
        Generate image for a single outfit
        
        Args:
            outfit: FullOutfit object
            outfit_id: Optional custom outfit ID
            
        Returns:
            FullOutfit object with image_path populated
        """
        if not outfit_id:
            outfit_id = outfit.outfit.lower().replace(' ', '_')
        
        print(f"\nGenerating image for outfit: {outfit.outfit} ({outfit_id})")
        
        outfit_dict = outfit.model_dump()
        
        # Generate outfit image
        outfit_prompt = self.create_outfit_prompt(outfit_dict)
        print(f"Outfit prompt preview: {outfit_prompt[:150]}...")
        outfit_image_path = self.generate_image(outfit_prompt, outfit_id)
        
        if not outfit_image_path:
            print(f"Trying simpler prompt for {outfit.outfit}...")
            simple_prompt = f"Professional catalog photograph of {outfit.outfit}, flat lay on white background, realistic, high quality"
            outfit_image_path = self.generate_image(simple_prompt, outfit_id)
        
        if not outfit_image_path:
            print(f"Creating placeholder for {outfit.outfit}...")
            outfit_image_path = self.create_placeholder_image(outfit_id, outfit.outfit)
        
        # Update outfit with image path
        outfit_dict['image_path'] = outfit_image_path
        outfit_with_image = FullOutfit(**outfit_dict)
        
        return outfit_with_image
    
    def generate_images_for_all_outfits(self, outfits: List[FullOutfit]) -> List[FullOutfit]:
        """
        Generate images for all outfits
        
        Args:
            outfits: List of FullOutfit objects
            
        Returns:
            List of FullOutfit objects with image_path populated
        """
        outfits_with_images = []
        
        print(f"\n{'='*80}")
        print(f"Starting image generation for {len(outfits)} outfits...")
        print(f"{'='*80}\n")
        
        for i, outfit in enumerate(outfits, 1):
            print(f"\n--- Outfit {i}/{len(outfits)} ---")
            outfit_id = f"outfit_{i:03d}_{outfit.outfit.lower().replace(' ', '_')}"
            outfit_with_image = self.generate_outfit_image(outfit, outfit_id)
            outfits_with_images.append(outfit_with_image)
        
        print(f"\n{'='*80}")
        print(f"✓ Image generation complete for all outfits!")
        print(f"{'='*80}\n")
        
        return outfits_with_images
    
    def save_outfits_with_images(self, outfits: List[FullOutfit], filename: str):
        """
        Save outfits with image paths to JSON file
        
        Args:
            outfits: List of FullOutfit objects
            filename: Output JSON filename
        """
        outfits_dict = {
            "outfits": [outfit.model_dump() for outfit in outfits],
            "total_outfits": len(outfits)
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(outfits_dict, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Outfits with image paths saved to {filename}")
    
    def display_outfits_summary(self, outfits: List[FullOutfit]):
        """Display summary of outfits"""
        print("\n" + "="*100)
        print("OUTFITS SUMMARY")
        print("="*100 + "\n")
        
        for idx, outfit in enumerate(outfits, 1):
            print(f"\n--- Outfit {idx}: {outfit.outfit.upper()} ---")
            print(f"Description: {outfit.outfit_description[:150]}..." if len(outfit.outfit_description) > 150 else f"Description: {outfit.outfit_description}")
            if outfit.image_path:
                print(f"Image: {outfit.image_path}")
            print("-" * 100)

# Character Generator

In [3]:

from openai import OpenAI
import base64
import os
from typing import Optional, Dict
from pydantic import BaseModel, Field
from PIL import Image
from google import genai
from google.genai import types
from io import BytesIO
import io
from typing import List
import datetime
import PIL.Image as PILImage
import os
import tempfile
from dotenv import load_dotenv

load_dotenv()


class FullCharacter(BaseModel):
    name: str
    age: Optional[int] = None
    role: Optional[str] = None
    gender: Optional[str] = None
    overall_description: Optional[str] = None
    image_path: Optional[str] = None
    reference_description: Optional[str] = None


class CharacterGenerator:
    def __init__(self, output_dir: str = "projects_data"):
        # Configure Google Generative AI

        self.client = genai.Client()
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def describe_character_appearance(self, image_path: str) -> str:
        """
        Generate a concise reference description of the character image
        for use in scene generation prompts
        
        Args:
            image_path: Path to character image
            
        Returns:
            Short description for reference identification
        """
        try:
            img = Image.open(image_path)
            
            prompt = """
            Please analyze this character image and provide a SHORT, CONCISE description (2-3 lines maximum) including:
            1. Gender
            2. Key outfit details (main colors, type of clothing)
            3. ONE distinctive facial or physical feature that helps identify them from other similar people

            Format: "[Gender], wearing [outfit color and type], [one distinctive feature]"

            Example: "Female, wearing green chudi with gold border, has an oval face with expressive eyes"

            Keep it brief and focused on visual identification."""
            
            response = self.client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[prompt, img]
            )
            
            description = response.text.strip()
            print(f"  📝 Generated reference description: {description[:100]}...")
            return description
        except Exception as e:
            print(f"  ⚠️  Error describing character appearance: {e}")
            return "character in reference image"

        
    def create_front_facing_prompt(self, character: dict) -> str:
        # Extract character traits
        name = character.get('name', 'Character')
        age = character.get('age', 25)
        gender = character.get('gender', 'person')
        role = character.get('role', 'character')
        
        # Create a safe, persona-based description
        safe_description = self._create_safe_description(character.get('overall_description', ''))
        
        # Generate persona-based outfit
        # outfit_description = self._generate_persona_outfit(character)
        
        base_prompt = f"""
        Create a realistic portrait photograph of a {age}-year-old {gender} named {name}.
        
        Character Details:
        - Age: {age} years old
        - Gender: {gender}
        - Role: {role}
        - Personality: {safe_description}
        
        Outfit & Appearance:
        Based on the personality and role of the character, create a outfit and appearance that is authentic to their role and personality in more detailed way.

        IMPORTANT REQUIREMENTS:
        Always create character in indian style
        
        Photography Style:
        - Realistic portrait style
        - Front-facing, looking directly at camera
        - Natural, authentic appearance
        - Soft, natural lighting
        - Neutral background (white or light gray)
        - High resolution, detailed
        - Expression that matches personality 
        -Standing Still passport style
        : {safe_description}
        
        Focus on creating a character that looks authentic to their role and personality, not generic or overly professional.
        """
        return base_prompt.strip()
    
    def _create_safe_description(self, description: str) -> str:
        """Create a safe description by removing potentially problematic content"""
        if not description:
            return "friendly and professional"
        
        # Remove crime/violence related words
        problematic_words = [
            'crime', 'murder', 'kill', 'death', 'violence', 'gun', 'weapon', 
            'interrogation', 'police', 'detective', 'suspect', 'guilt', 'fear',
            'betrayal', 'reckless', 'aggressive', 'tough', 'commanding'
        ]
        
        safe_desc = description.lower()
        for word in problematic_words:
            safe_desc = safe_desc.replace(word, '')
        
        # Clean up and create a positive description
        safe_desc = safe_desc.strip()
        if not safe_desc or len(safe_desc) < 10:
            return "friendly and professional"
        
        # Add positive traits
        positive_traits = ["confident", "friendly", "professional", "approachable"]
        return f"{safe_desc[:100]}, {', '.join(positive_traits[:2])}"
    
    
    
   
    
    def generate_image(self, prompt: str, image_type: str, character_id: str) -> Optional[str]:
        try:
            print(f"Generating {image_type} image for {character_id} using Gemini...")
            
            # Use the correct Google Generative AI API with image generation
            response = self.client.models.generate_content(
                model="gemini-2.5-flash-image",
                contents=prompt,
                config=types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=types.ImageConfig(
                    aspect_ratio="16:9",
                )
            )
            )
            
            image_saved = False
            filename = f"{character_id}_{image_type}.png"
            filepath = os.path.join(self.output_dir, filename)
            
            # Check if response has images
            if hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        image_data = part.inline_data.data
                        image = Image.open(BytesIO(image_data))
                        image.save(filepath)
                        image_saved = True
                        print(f" Saved {image_type} image: {filepath}")
                        break
                    elif hasattr(part, 'text') and part.text:
                        print(f"Gemini response: {part.text[:100]}...")
            
            if not image_saved:
                print(f" No image data generated for {character_id} {image_type}")
                return None
                
            return filepath
            
        except Exception as e:
            print(f" Error generating {image_type} image for {character_id}: {str(e)}")
            return None

    
    

    def generate_character_images(self, character: FullCharacter) -> FullCharacter:
        
        print(f"\nGenerating image for {character.name} )")

        character_dict = character.model_dump()

        # Generate character image
        character_prompt = self.create_front_facing_prompt(character_dict)
        print(f"Character prompt: {character_prompt[:100]}...")
        character_image_path = self.generate_image(character_prompt, "character", character.name)

        if not character_image_path:
            print(f"Trying simpler prompt for {character.name}...")
            simple_prompt = f"Professional portrait of a {character.age}-year-old {character.gender} named {character.name}, friendly expression, neutral background"
            character_image_path = self.generate_image(simple_prompt, "character", character.name)
        
        if not character_image_path:
            print(f"Creating placeholder for {character.name}...")
            character_image_path = self.create_placeholder_image(character.name)

        # Update character with image path
        reference_description = None
        if character_image_path and os.path.exists(character_image_path):
            reference_description = self.describe_character_appearance(character_image_path)

        # Update character with image path and reference description
        character_dict['image_path'] = character_image_path
        character_dict['reference_description'] = reference_description  # NEW
        character_with_images = FullCharacter(**character_dict)

        return character_with_images

    def create_placeholder_image(self, character_name: str) -> str:
        """Create a placeholder image when AI generation fails"""
        try:
            # Create a simple placeholder image
            filename = f"{character_name}_character.png"
            filepath = os.path.join(self.output_dir, filename)
            
            # Create a simple colored rectangle as placeholder
            img = Image.new('RGB', (400, 400), color='lightgray')
            
            # Add text (if PIL supports it)
            try:
                from PIL import ImageDraw, ImageFont
                draw = ImageDraw.Draw(img)
                
                # Try to use a default font
                try:
                    font = ImageFont.truetype("arial.ttf", 20)
                except:
                    font = ImageFont.load_default()
                
                # Add character name
                text = f"{character_name}\n(Placeholder)"
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                
                x = (400 - text_width) // 2
                y = (400 - text_height) // 2
                
                draw.text((x, y), text, fill='black', font=font)
            except:
                # If text fails, just save the colored rectangle
                pass
            
            img.save(filepath)
            print(f" Created placeholder image: {filepath}")
            return filepath
            
        except Exception as e:
            print(f" Error creating placeholder: {str(e)}")
            return None

    def generate_images_for_all_characters(self, characters: List[FullCharacter]) -> List[FullCharacter]:
        characters_with_images = []

        print(f"Starting image generation for {len(characters)} characters...")

        for i, character in enumerate(characters, 1):
            print(f"\n--- Character {i}/{len(characters)} ---")
            character_with_images = self.generate_character_images(character)
            characters_with_images.append(character_with_images)

        print(f"\nImage generation complete for all characters!")
        return characters_with_images

    
    
    
    def save_characters_with_images(self, characters: list[FullCharacter], filename: str):
        """Save characters with image paths and reference descriptions"""
        import json
        
        characters_dict = {
            "characters": [char.model_dump() for char in characters]
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(characters_dict, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Characters with image paths and reference descriptions saved to {filename}")



In [10]:
with open("/Users/sanjail/Akaike/Internal_project/ads_poc/projects_data/shot_script_generated.json","r") as f:
    data=json.load(f)

characters_info=data["characters_info"]
characters_data=[]

for character in characters_info:
    full_character_info=FullCharacter(
        name=character["name"],
        age=40,
        role=character["role"],
        gender=character["gender"],
    )
    characters_data.append(full_character_info)

character_generator=CharacterGenerator()

characters_with_images=character_generator.generate_images_for_all_characters(characters_data)

character_generator.save_characters_with_images(characters_with_images,"demo_image.png")


Starting image generation for 1 characters...

--- Character 1/1 ---

Generating image for dhoni )
Character prompt: Create a realistic portrait photograph of a 40-year-old male named dhoni.
        
        Character...
Generating character image for dhoni using Gemini...
Gemini response: Here's a realistic portrait of Dhoni, incorporating all your specifications: ...
 Saved character image: projects_data/dhoni_character.png

Image generation complete for all characters!
💾 Characters with image paths saved to demo_image.png


# Location Generator

In [4]:
from openai import OpenAI
import base64
import os
from typing import Optional, Dict, List
from pydantic import BaseModel, Field
from PIL import Image
from google import genai
from google.genai import types
from io import BytesIO
import io
import datetime
import PIL.Image as PILImage
import tempfile

from dotenv import load_dotenv

load_dotenv()


class LocationInfo(BaseModel):
    """Location information extracted from the script"""
    name: str = Field(description="Location Name (lowercase)")
    overall_description: Optional[str] = Field(
        default=None, 
        description="Detailed visual description of the location"
    )
    image_path: Optional[str] = Field(default=None, description="Path to generated location image")


class FullLocation(BaseModel):
    """Full location details for image generation"""
    name: str
    overall_description: Optional[str] = None
    image_path: Optional[str] = None


class LocationGenerator:
    def __init__(self, output_dir: str = "location_images"):
        # Configure Google Generative AI
        self.client = genai.Client()
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
    def create_location_prompt(self, location: dict) -> str:
        """
        Create detailed prompt for location image generation
        
        Args:
            location: Dictionary with 'name' and 'overall_description'
        """
        name = location.get('name', 'Location')
        description = location.get('overall_description', '')
        
        # Parse the description to extract structured details
        location_details = self._parse_location_description(description)
        
        base_prompt = f"""
Create a highly realistic, cinematic photograph of a {name}.

{description}

DETAILED SPECIFICATIONS:

Location Type: {location_details.get('location_type', name)}

Design Style & Mood:
{location_details.get('design_style', 'Modern and realistic')}

Key Architectural Elements:
{location_details.get('architectural_elements', 'Authentic architectural details appropriate to the location type')}

Color Palette & Materials:
{location_details.get('color_palette', 'Natural, realistic color palette')}

Props & Visual Details:
{location_details.get('props', 'Appropriate props and details that make the space feel lived-in and authentic')}

Lighting:
{location_details.get('lighting', 'Natural, cinematic lighting that enhances the mood')}

Atmosphere & Vibe:
{location_details.get('atmosphere', 'Realistic and inviting atmosphere')}

PHOTOGRAPHY REQUIREMENTS:
- Camera: {location_details.get('camera_framing', 'Wide shot showing the full space')}
- High resolution, photorealistic quality
- Professional cinematography style suitable for commercial advertising
- Perfect for film production reference
- No people in the frame - empty location shot
- Sharp focus with appropriate depth of field
- Color graded for commercial/cinematic look
- Indian context and aesthetic where applicable

IMPORTANT:
- Create an EMPTY location (no people)
- Focus on the environment and atmosphere
- Realistic, not CGI or cartoon-like
- Professional photography quality
- Suitable for use as a filming location reference

Style: Cinematic photography, commercial advertisement quality, realistic and detailed.
"""
        return base_prompt.strip()
    
    def _parse_location_description(self, description: str) -> Dict[str, str]:
        """
        Parse the structured location description into components
        
        Args:
            description: Structured description string with format:
                Location Type: ..., Design Style & Mood: ..., etc.
        
        Returns:
            Dictionary with parsed components
        """
        if not description:
            return {}
        
        components = {
            'location_type': '',
            'design_style': '',
            'architectural_elements': '',
            'color_palette': '',
            'props': '',
            'lighting': '',
            'atmosphere': '',
            'camera_framing': ''
        }
        
        # Parse each component
        lines = description.split(',')
        current_key = None
        
        for line in lines:
            line = line.strip()
            
            # Check for component headers
            if 'Location Type:' in line:
                components['location_type'] = line.split('Location Type:')[1].strip()
            elif 'Design Style & Mood:' in line or 'Design Style and Mood:' in line:
                components['design_style'] = line.split(':')[1].strip() if ':' in line else line
            elif 'Key Architectural Elements:' in line:
                components['architectural_elements'] = line.split('Key Architectural Elements:')[1].strip()
            elif 'Color Palette & Materials:' in line or 'Color Palette and Materials:' in line:
                components['color_palette'] = line.split(':')[1].strip() if ':' in line else line
            elif 'Props & Visual Details:' in line or 'Props and Visual Details:' in line:
                components['props'] = line.split(':')[1].strip() if ':' in line else line
            elif 'Lighting:' in line:
                components['lighting'] = line.split('Lighting:')[1].strip()
            elif 'Atmosphere & Vibe:' in line or 'Atmosphere and Vibe:' in line:
                components['atmosphere'] = line.split(':')[1].strip() if ':' in line else line
            elif 'Camera Framing Note:' in line:
                components['camera_framing'] = line.split('Camera Framing Note:')[1].strip()
            else:
                # Continuation of previous component
                if current_key and components[current_key]:
                    components[current_key] += ', ' + line
        
        # If parsing failed, use the entire description as location_type
        if not any(components.values()):
            components['location_type'] = description[:200]
        
        return components
    
    def generate_image(self, prompt: str, location_id: str) -> Optional[str]:
        """
        Generate location image using Gemini
        
        Args:
            prompt: Detailed prompt for image generation
            location_id: Unique identifier for the location
            
        Returns:
            File path to generated image or None if failed
        """
        try:
            print(f"Generating location image for {location_id} using Gemini...")
            
            # Use Gemini model for image generation
            
            
            response = self.client.models.generate_content(
                model="gemini-2.5-flash-image",
                contents=prompt,
                config=types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=types.ImageConfig(
                    aspect_ratio="16:9",
                )
            )
            )
            
            image_saved = False
            filename = f"{location_id}_location.png"
            filepath = os.path.join(self.output_dir, filename)
            
            # Check if response has images
            if hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        image_data = part.inline_data.data
                        image = Image.open(BytesIO(image_data))
                        image.save(filepath)
                        image_saved = True
                        print(f"✓ Saved location image: {filepath}")
                        break
                    elif hasattr(part, 'text') and part.text:
                        print(f"Gemini response: {part.text[:100]}...")
            
            if not image_saved:
                print(f"⚠ No image data generated for {location_id}")
                return None
                
            return filepath
            
        except Exception as e:
            print(f"✗ Error generating location image for {location_id}: {str(e)}")
            return None
    
    def generate_location_image(self, location: FullLocation, location_id: str = None) -> FullLocation:
        """
        Generate image for a single location
        
        Args:
            location: FullLocation object
            location_id: Optional custom location ID
            
        Returns:
            FullLocation object with image_path populated
        """
        if not location_id:
            location_id = location.name.lower().replace(' ', '_')
        
        print(f"\nGenerating image for location: {location.name} ({location_id})")
        
        location_dict = location.model_dump()
        
        # Generate location image
        location_prompt = self.create_location_prompt(location_dict)
        print(f"Location prompt preview: {location_prompt[:150]}...")
        location_image_path = self.generate_image(location_prompt, location_id)
        
        if not location_image_path:
            print(f"Trying simpler prompt for {location.name}...")
            simple_prompt = f"Professional cinematic photograph of a {location.name}, empty location, realistic, high quality, suitable for film production"
            location_image_path = self.generate_image(simple_prompt, location_id)
        
        if not location_image_path:
            print(f"Creating placeholder for {location.name}...")
            location_image_path = self.create_placeholder_image(location_id, location.name)
        
        # Update location with image path
        location_dict['image_path'] = location_image_path
        location_with_image = FullLocation(**location_dict)
        
        return location_with_image
    
    def create_placeholder_image(self, location_id: str, location_name: str) -> str:
        """Create a placeholder image when AI generation fails"""
        try:
            filename = f"{location_id}_location.png"
            filepath = os.path.join(self.output_dir, filename)
            
            # Create a simple placeholder image
            img = Image.new('RGB', (800, 600), color='lightblue')
            
            # Add text
            try:
                from PIL import ImageDraw, ImageFont
                draw = ImageDraw.Draw(img)
                
                try:
                    font = ImageFont.truetype("arial.ttf", 24)
                except:
                    font = ImageFont.load_default()
                
                text = f"{location_name}\n(Location Placeholder)"
                bbox = draw.textbbox((0, 0), text, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
                
                x = (800 - text_width) // 2
                y = (600 - text_height) // 2
                
                draw.text((x, y), text, fill='black', font=font)
            except:
                pass
            
            img.save(filepath)
            print(f"✓ Created placeholder image: {filepath}")
            return filepath
            
        except Exception as e:
            print(f"✗ Error creating placeholder: {str(e)}")
            return None
    
    def generate_images_for_all_locations(self, locations: List[FullLocation]) -> List[FullLocation]:
        """
        Generate images for all locations
        
        Args:
            locations: List of FullLocation objects
            
        Returns:
            List of FullLocation objects with image_path populated
        """
        locations_with_images = []
        
        print(f"\n{'='*80}")
        print(f"Starting image generation for {len(locations)} locations...")
        print(f"{'='*80}\n")
        
        for i, location in enumerate(locations, 1):
            print(f"\n--- Location {i}/{len(locations)} ---")
            location_id = f"loc_{i:03d}_{location.name.lower().replace(' ', '_')}"
            location_with_image = self.generate_location_image(location, location_id)
            locations_with_images.append(location_with_image)
        
        print(f"\n{'='*80}")
        print(f"✓ Image generation complete for all locations!")
        print(f"{'='*80}\n")
        
        return locations_with_images
    
    def save_locations_with_images(self, locations: List[FullLocation], filename: str):
        """
        Save locations with image paths to JSON file
        
        Args:
            locations: List of FullLocation objects
            filename: Output JSON filename
        """
        import json
        
        locations_dict = {
            "locations": [loc.model_dump() for loc in locations],
            "total_locations": len(locations)
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(locations_dict, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Locations with image paths saved to {filename}")
    
    def display_locations_summary(self, locations: List[FullLocation]):
        """Display summary of locations"""
        print("\n" + "="*100)
        print("LOCATIONS SUMMARY")
        print("="*100 + "\n")
        
        for idx, location in enumerate(locations, 1):
            print(f"\n--- Location {idx}: {location.name.upper()} ---")
            print(f"Description: {location.overall_description[:150]}..." if location.overall_description else "No description")
            if location.image_path:
                print(f"Image: {location.image_path}")
            print("-" * 100)


# Example usage
# if __name__ == "__main__":
#     # Example locations
#     example_locations = [
#         FullLocation(
#             name="hotel room",
#             overall_description="Location Type: Hotel Room (Morning), Design Style & Mood: Minimal modern Indian, soft and calm mood, Key Architectural Elements: Light wooden furniture, soft beige curtains, neutral wall tones, Color Palette & Materials: Warm beige, soft white, natural wood, cotton bedding, Props & Visual Details: Cricket duffel bag by bedside, water bottle, sports shoes, framed art on wall, Lighting: Soft natural morning light filtering through curtains, warm gentle shadows, Atmosphere & Vibe: Peaceful morning preparation atmosphere, Camera Framing Note: Wide shot"
#         ),
#         FullLocation(
#             name="cricket stadium",
#             overall_description="Location Type: Cricket Stadium Exterior (Day), Design Style & Mood: Modern Indian sports venue, energetic and vibrant, Key Architectural Elements: Large concrete structure, steel railings, stadium seating visible, Color Palette & Materials: Blue stadium seats, white concrete, green field visible in background, Props & Visual Details: Stadium signage, flags, crowd barriers, Lighting: Bright harsh daylight from above, strong shadows, Atmosphere & Vibe: Match day excitement and anticipation, Camera Framing Note: Wide establishing shot"
#         ),
#         FullLocation(
#             name="stadium entrance",
#             overall_description="Location Type: Stadium Entrance Gate, Design Style & Mood: Industrial modern, busy and crowded feel, Key Architectural Elements: Metal gates, ticket counters, security checkpoints, Color Palette & Materials: Steel grey, concrete, glass panels, Props & Visual Details: Ticket scanners, security barriers, directional signs, sponsor banners, Lighting: Mix of natural daylight and artificial overhead lights, Atmosphere & Vibe: Pre-match buzz and crowd energy, Camera Framing Note: Medium shot"
#         )
#     ]
    
#     # Initialize generator
#     generator = LocationGenerator(output_dir="location_images")
    
#     # Generate images for all locations
#     locations_with_images = generator.generate_images_for_all_locations(example_locations)
    
#     # Display summary
#     generator.display_locations_summary(locations_with_images)
    
#     # Save to file
#     generator.save_locations_with_images(
#         locations_with_images, 
#         "locations_with_images.json"
#     )

# Scene Description generator

In [5]:
from typing import List, Optional, Dict, Any
from utils.llm import get_llm_model
from pydantic import BaseModel, Field
import json
import os
from dotenv import load_dotenv


load_dotenv()

llm_client = get_llm_model("gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))


class Shot(BaseModel):
    """Individual shot details with image prompt"""
    shot_no: int = Field(description="Shot number in sequence")
    duration: str = Field(description="Duration of the shot (e.g., '2 seconds', '3 seconds')")
    time_stamp: str = Field(description="Time range in format MM:SS-MM:SS")
    location: str = Field(description="Detailed location description")
    location_name: str = Field(description="Name of the location used in this shot (lowercase, must match location_info)")
    camera_angle: str = Field(description="Camera angle and shot type (e.g., 'Medium close-up', 'Wide shot')")
    visual_description: str = Field(description="Detailed visual description of what's in frame")
    action: str = Field(description="Specific actions happening in the shot")
    objects_props_involved:str=Field(description="Detailed description of the objects involved")
    audio_sfx: str = Field(description="Audio and sound effects")
    dialogue: Optional[str] = Field(default="None", description="Any spoken dialogue in the shot")
    voice_over: Optional[str] = Field(default="None", description="Voice over narration")
    text_overlay: Optional[str] = Field(default="None", description="If present On-screen text or graphics")
    key_focus: str = Field(description="Primary focus or goal of this shot")
    product_image_required: bool = Field(default=False, description="Does this shot require product image")
    
    characters_involved: List[str] = Field(
        default_factory=list, 
        description="List of character names involved in this shot (all lowercase)"
    )
    outfit_character_mapping:List[OutfitMapping]=Field(
        description="List of character names and their outfit mapping"
    )
    
    image_prompt: Optional[str] = Field(default=None, description="Detailed prompt for keyframe image generation")


class SceneDescription(BaseModel):
    """Complete scene description with all shots and prompts"""
    ad_title: str = Field(description="Title of the ad")
    total_shots: int = Field(description="Total number of shots")
    shots: List[Shot] = Field(description="List of shots with image prompts")


class SceneDescriptionResponse(BaseModel):
    """Complete scene description with all shots and prompts"""
    image_prompt: str= Field(description="Detailed Image prompts")


class SceneDescriptionGenerator:
    def __init__(self):
        self.llm = llm_client
    
    def create_scene_prompt_system_prompt(
    self,
    brand_info: Dict[str, Any],
    ad_concept: Optional[Dict[str, Any]] = None,
    characters_info: Optional[List[Dict[str, Any]]] = None,
    locations_info: Optional[List[Dict[str, Any]]] = None,
    outfits_info: Optional[List[Dict[str, Any]]] = None  # NEW
) -> str:
        """Create system prompt for generating scene image prompts"""
        
        characters_context = ""
        if characters_info:
            characters_context = "\n\nCHARACTERS IN THIS AD:\n"
            for idx, char in enumerate(characters_info, 1):
                char_name = char.get('name', 'Character')
                gender = char.get('gender', 'Unknown')
                ref_desc = char.get('reference_description', '')
                if ref_desc:
                    characters_context += f"{idx}. {char_name} ({gender}): Reference image shows - {ref_desc}\n"
                else:
                    characters_context += f"{idx}. {char_name} ({gender}): {char.get('overall_description', 'No description')[:100]}\n"
                
        
        locations_context = ""
        if locations_info:
            locations_context = "\n\nLOCATIONS IN THIS AD:\n"
            for idx, loc in enumerate(locations_info, 1):
                locations_context += f"{idx}. {loc.get('name', 'Location')}: {loc.get('overall_description', 'No description')[:200]}\n"
        
        # NEW: Outfits context
        outfits_context = ""
        if outfits_info:
            outfits_context = "\n\nOUTFITS IN THIS AD:\n"
            for idx, outfit in enumerate(outfits_info, 1):
                outfits_context += f"{idx}. {outfit.get('outfit', 'Outfit')}: {outfit.get('outfit_description', 'No description')[:200]}\n"
        
        ad_concept_context = ""
        if ad_concept:
            ad_concept_context = f"\n\nAD CONCEPT OVERVIEW:\n{json.dumps(ad_concept, indent=2)}\n"
        
        return f"""You are an expert AI image prompt engineer specializing in commercial advertisement and cinematic photography.

Your task is to create HIGHLY DETAILED, SPECIFIC image generation prompts for each shot in a commercial ad.

Over Ad Information:
{json.dumps(brand_info, indent=2)}
{locations_context}
{ad_concept_context}

Character Refernce Information:
{characters_context}

Character Outfit Info in this Shot:
{outfits_context}

CRITICAL REQUIREMENTS FOR IMAGE PROMPTS:

1. **Character & Outfit References**:
- When characters are present, reference their outfit from the outfit_character_mapping
- Use outfit descriptions provided in the outfits list
- Describe character positioning and outfit details clearly
- For same-gender characters, use "first person", "second person" OR descriptive identifiers
- For different-gender characters, use gender pronouns

2. **Outfit Integration**:
- Match each character to their assigned outfit from outfit_character_mapping
- Include specific outfit details in the character description
- Reference outfit colors, style, and key features
- Ensure outfit is appropriate for the scene context


3. **Product Integration**:
   - If product_image_required is True, mention: "[Product Name] and [Description of product] is placed at [specific location in frame]"
   - Describe product placement: "center", "right side", "held in hand", "on table", etc.
   - For product shots, describe lighting on product specifically

3. **Standard Shot Prompt Structure**:
```
Create a realistic  image of [scene subject name and action].
Setting: [Detailed location description with environmental elements, props, spatial layout]
Characters & Appearance Consistency:
Describe each character clearly:
- Identify character by name and reference source (e.g., “Priya (the first girl in the reference image and provide keyfeatures to differentiate In this format [Gender], wearing this dress in the reference image [outfit color and type], [one distinctive feature])”).
- Always say: **keep facial features, skin tone, hairstyle, and body shape consistent with the reference image.**
- If multiple characters, specify their distance, facing direction, and interaction.
- If male/female mix, clarify by naming: e.g., *Arjun (male), Priya (female).

in this format:
 Priya (protagonist):
- Reference: The first girl in the character reference images, female wearing pale yellow t-shirt
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Change the Outfit to: Casual chic evening attire – light beige silk blouse with delicate buttons, soft pastel pink midi skirt with subtle pleats as shown in the reference image
- Expression: Expressive, slightly embarrassed small smile, warm and genuine
- Action: Gently touching her right cheek with fingertips in a shy gesture
- Position: Seated in the foreground, body slightly angled toward camera, facing 3/4 view

Action: [Specific movements and interactions happening]
Camera & Framing: [Camera angle, shot type, lens perspective]
Lighting: [Type, direction, intensity, mood - be very specific]
Mood & Atmosphere: [Emotional tone, energy level]
Key Focus: [What should be the visual center of attention]
[If product_image_required]: Product Placement: [Product name][what product] is visible/placed at [location], [how it's integrated into scene]
[If text_overlay present]: Note: Image should have space for text overlay that will say "[text content]" positioned at [location]
Cinematic details: [Depth of field, color grading, material reflections, realistic details]
```
---IMPORTANT----

**Character Identification Pattern:**
```
Single character: "Priya (the girl in the reference image, wearing this dress in the reference image [outfit details from reference the])"
Multiple Character(same gender): "Priya (the first girl in reference image [[Gender], wearing this dress in reference image [outfit color and type], [one distinctive feature]), Ria (the second girl in reference image [[Gender], wearing this dress in this reference image [outfit color and type], [one distinctive feature])"
Multiple Character(Different gender): "Arjun ([[Gender], wearing this dress in reference image [outfit color and type], [one distinctive feature]), Ria ( [[Gender], wearing this dress in this reference image [outfit color and type], [one distinctive feature])"


**IMPORTANT RULES:**
- Every character mention MUST reference their order in the reference images
- Every character MUST include "Key reference from the character reference image and keep features consistent with reference image"
- Use reference_description details to differentiate between similar characters
- Never just say "the character" - always identify which reference image

EXAMPLE PROMPT FOR STANDARD SCENE GENERATION:

 ═══════════════════════════════════════════════════════════════════════════════
EXAMPLE 1: TWO CHARACTERS (SAME GENDER)
═══════════════════════════════════════════════════════════════════════════════

Create a realistic, high-quality image of two women, Priya and Janani, sitting at an outdoor cafe.

Characters & Appearance Consistency:

First woman - Priya (protagonist):
- Reference: The first girl in the character reference images, female wearing pale yellow t-shirt
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Change the Outfit to: Casual chic evening attire – light beige silk blouse with delicate buttons, soft pastel pink midi skirt with subtle pleats as shown in the reference image
- Expression: Expressive, slightly embarrassed small smile, warm and genuine
- Action: Gently touching her right cheek with fingertips in a shy gesture
- Position: Seated in the foreground, body slightly angled toward camera, facing 3/4 view

Second woman - Janani (friend):
- Reference: The second girl in the character reference images, female wearing black chudi
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Change the Outfit to: Casual smart outfit – light blue denim jacket over crisp white t-shirt, beige slim-fit trousers as shown in the reference image
- Expression: Warm, friendly smile, engaged in conversation
- Action: Leaning slightly toward Priya, holding a coffee cup
- Position: Seated slightly behind or beside Priya, creating natural depth

Setting & Location:
- Outdoor cafe setting, same as the location reference image
- Wooden cafe table between them with two ceramic coffee cups (one cappuccino, one latte), small plates with pastries
- Background: Soft-focus cafe environment with other patrons barely visible, lush green trees, hanging fairy lights
- Ground: Textured stone flooring typical of outdoor cafes

Lighting & Atmosphere:
- Golden hour lighting (late afternoon, around 5-6 PM)
- Warm, natural sunlight filtering through tree leaves, creating dappled light patterns
- Soft highlights on faces, especially catching Priya's cheekbones and Janani's hair
- Gentle rim lighting on their shoulders from backlight
- Color temperature: ~4500K (warm golden)

Camera & Composition:
- Medium shot, captured at eye level, slight 3/4 angle
- 50mm lens equivalent, f/2.8 for natural depth of field
- Priya in sharp focus (foreground), Janani slightly softer but still clear
- Background with beautiful bokeh – blurred cafe details and tree lights creating soft circular highlights
- Composition follows rule of thirds with Priya positioned on left third

Mood & Vibe:
- Light-hearted, comedic moment frozen in time
- Natural human interaction and genuine friendship
- Golden hour warmth evoking comfort and joy
- Indian urban cafe culture context – modern yet relatable
- Relaxed postures, authentic body language

Technical Details:
- Photorealistic quality, commercial photography standard
- Natural color grading with enhanced warm tones
- Subtle vignette to draw focus to subjects
- Sharp details on faces and clothing textures
- Realistic fabric draping and material reflections

═══════════════════════════════════════════════════════════════════════════════
EXAMPLE 2: SINGLE CHARACTER WITH PRODUCT
═══════════════════════════════════════════════════════════════════════════════

Create a realistic, high-quality image of Priya standing in a bright modern bathroom, applying sunscreen in front of a mirror.

Character & Appearance Consistency:

Priya (protagonist):
- Reference: The girl in the character reference image, female wearing blue chudi
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Change the Outfit to: White cotton crop top (relaxed fit, showing natural comfort) and light blue denim jeans (high-waisted, casual fit) as shown in the reference image
- Expression: Focused yet relaxed, slight satisfied smile as she cares for her skin, eyes looking at her reflection
- Action: Gently applying clear gel sunscreen onto her left cheek using fingertips of her right hand, natural dabbing motion
- Hair: Loosely tied back or down naturally, casual morning styling
- Position: Standing in front of bathroom mirror, body at 3/4 angle to camera, face visible both directly and in mirror reflection

Product Integration:

Deconstruct Gel Sunscreen:
- Reference: Same bottle design as shown in product reference image
- Position: Held in her left hand near the sink counter, clearly visible with label facing camera
- Details: White and blue packaging with "Deconstruct" branding visible, SPF 55 text readable
- Lighting on product: Soft highlight on the bottle surface, making it look premium and clean
- Placement: Bottle positioned at mid-frame right, easy to see without dominating the shot

Setting & Location:
- Modern Indian bathroom, same aesthetic as location reference image
- Mirror: Large frameless wall mirror with clean edges, showing Priya's reflection clearly
- Sink area: White ceramic sink with chrome fixtures, marble or quartz countertop (light beige/white)
- Window: Frosted glass window on the left side, allowing diffused morning sunlight
- Background elements: Neatly arranged - small potted succulent, soap dispenser, minimal clutter

Lighting & Atmosphere:
- Primary light: Soft morning sunlight streaming through frosted window from left side
- Quality: Diffused, gentle light creating a fresh morning feel
- Color temperature: ~5500K (natural daylight, slightly cool-warm balanced)
- Face lighting: Even illumination with soft shadows, flattering and natural
- Mirror reflection: Slightly brighter, catching natural window light
- Product lighting: Gentle highlight making the bottle surfaces gleam subtly

Camera & Composition:
- Medium close-up shot, captured at slight upward tilt toward mirror reflection
- 35mm lens equivalent, f/2.4 for sharp subject with slightly soft background
- Framing: Priya's upper body (from waist up) centered with slight room at top for mirror reflection
- Mirror creates interesting double perspective - seeing both her direct profile and her face in reflection
- Product visible in lower third to mid-frame area

Action & Storytelling:
- Natural skincare routine moment, authentic and relatable
- Captures the motion of application - fingers gently touching cheek with product
- Shows care and attention to skin health
- Morning self-care ritual, peaceful and mindful moment

Mood & Atmosphere:
- Fresh morning energy, calm and peaceful
- Clean, minimalist aesthetic typical of modern Indian urban homes
- Self-care and wellness vibe
- Natural, unposed authenticity
- Bright, airy, and inviting space

Technical Details:
- Photorealistic quality, lifestyle photography standard
- Natural color grading emphasizing whites, soft blues, and warm skin tones
- Sharp focus on face and product, soft bokeh on distant background elements
- Realistic material textures: cotton fabric, denim, ceramic, glass, skin
- Subtle depth of field creating professional look
- No harsh shadows, even and flattering lighting throughout


 ═══════════════════════════════════════════════════════════════════════════════
EXAMPLE 3: MULTIPLE CHARACTERS (MIXED GENDER) WITH PRODUCT
═══════════════════════════════════════════════════════════════════════════════

Create a realistic, high-quality image of two people, Arjun (male) and Priya (female), at a cricket stadium entrance, with Arjun holding Deconstruct sunscreen.

Characters & Appearance Consistency:

First person - Arjun (male protagonist):
- Reference: The first person in character reference images, male wearing grey sports t-shirt
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Current outfit: CSK (Chennai Super Kings) official jersey in yellow and blue, navy athletic track pants, white sports shoes as shown in the reference image
- Expression: Confident smile, excited for the match, energetic
- Action: Holding Deconstruct Gel Sunscreen bottle in right hand, showing it to Priya, left hand gesturing toward stadium
- Position: Standing on the right side, body facing 3/4 toward camera and Priya

Second person - Priya (female friend):
- Reference: The second person in character reference images, female wearing green kurta
- **Keep facial features, skin tone, hairstyle, and body shape exactly consistent with the reference image**
- Change the Outfit to: Casual cricket fan attire – CSK team t-shirt in yellow, blue jeans, comfortable sneakers, wearing sunglasses on head as shown in the reference image
- Expression: Interested, nodding approvingly, slight smile
- Action: Looking at the sunscreen bottle Arjun is showing, carrying a small backpack
- Position: Standing on the left side, facing toward Arjun, creating natural interaction

Product Integration:

Deconstruct Gel Sunscreen:
- Reference: Same as product reference image – white and blue packaging
- Position: Held prominently in Arjun's right hand at chest level, label clearly visible facing camera
- Details: "Deconstruct" branding and "SPF 55" text readable, bottle catching sunlight
- Context: Shown as Arjun's essential item before entering stadium under harsh sun
- Message: Smart preparation for sun exposure at outdoor event

Setting & Location:
- Cricket stadium entrance/gate area, same as location reference image
- Background: Modern stadium architecture with large pillars, glass panels, "Gate 4" signage visible
- Crowd elements: Blurred other cricket fans in yellow jerseys in far background
- Ground: Concrete walkway with marked lines
- Banners: IPL promotional banners and sponsor boards (slightly out of focus)

Lighting & Atmosphere:
- Bright, harsh midday sunlight typical of cricket match timing (1-2 PM)
- Strong overhead sun creating defined shadows on ground
- Direct sunlight on characters' faces (demonstrating need for sun protection)
- Color temperature: ~5800K (bright daylight)
- Slight lens flare from sun in top corner, adding authentic outdoor feel
- Product catches highlight, making it stand out

Camera & Composition:
- Medium wide shot, captured at eye level
- 50mm lens equivalent, f/3.5 for sharp subjects with soft background
- Both characters clearly visible with stadium structure behind
- Arjun and product positioned following rule of thirds
- Negative space in background showing stadium context

Mood & Atmosphere:
- Energetic pre-match excitement
- Sunny outdoor sports event vibe
- Friendship and shared enthusiasm
- Modern urban Indian youth culture
- Sun protection awareness in casual, natural way

Action & Interaction:
- Natural conversation moment about sun protection
- Arjun recommending/showing the product to Priya
- Body language showing genuine friendship and trust
- Unforced product integration into real scenario

GUIDLINES:

1.Always for character, location or product mention same as the reference image if we have multiple character in that case give a way to diffrentiate like Priya(protognist the first girl in the reference image)
 similarly for location , product also give reference and breif description of reference to differentiate the prompt should be more detailed


2. Always Mention the word Keep the facial and body features same as the reference image whennever saying about the characters


4. **Final Product Showcase Shot** (ONLY if it's the LAST shot AND has product showcase in the script):
```
Product Showcase for [Product Name]:
Product Positioning: [Product] prominently displayed at [specific position - center, right, left] of the frame, [angle - front-facing, slight tilt, 3/4 view]
Background Scene: [Detailed background that connects to the ad story - location, atmosphere, blurred elements]
Lighting Setup:
- Main light: [Direction, color temperature, intensity]
- Rim lighting: [Details for product edges]
- Ambient: [Overall scene lighting]
- Special effects: [Lens flares, god rays, highlights, shadows]
Color Palette: [Specific colors and gradients - be very detailed]
Environmental Details: [Background elements, depth, blur, reflections]

Text Overlay Layout (to be added in post):
[For each text element, specify:]
- Text: "[Actual text]"
- Position: [Relative to product - left, right, above, below]
- Font style: [Bold, serif, sans-serif, script]
- Color: [Specific color]
- Size relative to product: [Large, medium, small]
- Special effects: [Shadow, glow, gradient]

Atmosphere: [Overall mood, energy, feeling - modern, fresh, energetic, premium, etc.]
Technical specs: [Resolution feel, depth of field, material reflections, professional photography quality]
Visual centerpiece: Product should be the main focus with background supporting the story
```

--IMPORTANT ---
 EXAMPLE PROMPT FOR PRODUCT SHOWCASE(THIS IS HOW YOU SHOULD GENERATE PRODUCT SHOWCASE IMAGE PROMPT):

   Showcase the Deconstruct Gel Sunscreen tube prominently on the center-right of the frame. Background is a modern, bright IPL stadium scene with warm sunlight streaming in from the top-left, creating lens flares and soft golden highlights. Include blurred cheering crowd and stadium details, with subtle reflections on the ground for realism. Use modern gradient colors like yellow, orange, and soft white to evoke energy and freshness.
     Text overlay:
      “IPL Season Begins. Stay Protected.” — split into two parts:
       First part (“IPL Season Begins.”) positioned left of the bottle, bold, energetic font in yellow, slightly italicized.
       Second part (“Stay Protected.”) positioned right of the bottle, bold black font with soft shadow, modern sans serif.
      “Deconstruct” — black thin serif (Didot or Bodoni), spaced wide, positioned just above the product.
      “Gel Sunscreen” — modern sans serif in light gray, subtle underline, below brand name.
      “SPF 55+ | PA+++” — monospaced metallic silver font, aligned bottom-right.
   Add dynamic sunlight flares, warm rim lighting, soft shadows under the product, and a modern, editorial feel. Keep the product as the visual centerpiece, with the background conveying IPL excitement and sunny protection.

5. **Key Guidelines**:
   - Be extremely specific about positions, colors, lighting
   - Use cinematic/photography terms
   - Ensure prompts are 150-300 words for standard shots
   - Product showcase prompts can be 300-500 words
   - Always maintain realism and commercial quality
   - Consider Indian context where relevant
   - Describe spatial relationships clearly
   - Include atmospheric and mood details
   - Specify depth of field and focus areas

6. **Text Overlay Handling**:
   - If text_overlay is present, mention it should have space for text
   - For product showcase, describe exact text layout in detail
   - Include font suggestions, colors, positions
   - Note: Actual text will be added in post-production

REMEMBER: The image prompt should be so detailed that an AI image generator can create the exact scene without any ambiguity."""

    def generate_image_prompt_for_shot(
        self,
        shot: Shot,
        brand_info: Dict[str, Any],
        characters_info: Optional[List[Dict[str, Any]]] = None,
        is_last_shot: bool = False,
        has_product_showcase: bool = False,
        ad_concept: Optional[Dict[str, Any]] = None,
        outfits_info:Optional[List[Dict[str, Any]]] = None 
    ) -> str:
        """
        Generate detailed image prompt for a single shot
        
        Args:
            shot: Shot object with all details
            brand_info: Brand information including product details
            characters_info: List of character information
            is_last_shot: Whether this is the final shot
            has_product_showcase: Whether this shot is a product showcase
            ad_concept: Optional ad concept for context
            
        Returns:
            Detailed image generation prompt
        """
        system_prompt = self.create_scene_prompt_system_prompt(
            brand_info=brand_info,
            ad_concept=ad_concept,
            characters_info=characters_info,
            outfits_info=outfits_info
        )
        
        # Build character context for this specific shot
        shot_characters = []
        if characters_info:
            shot_characters = [
                char for char in characters_info 
                if char.get('name', '').lower() in [c.lower() for c in shot.characters_involved]
            ]
        
        user_prompt = f"""Generate a detailed image generation prompt for the following shot:

SHOT DETAILS:
- Shot Number: {shot.shot_no}
- Duration: {shot.duration}
- Location: {shot.location}
- Camera Angle: {shot.camera_angle}
- Visual Description: {shot.visual_description}
- Action: {shot.action}
- Key Focus: {shot.key_focus}
- Characters Involved: {', '.join(shot.characters_involved) if shot.characters_involved else 'None'}
- Product Image Required: {shot.product_image_required}
- Text Overlay: {shot.text_overlay}

CHARACTERS IN THIS SHOT:
{json.dumps(shot_characters, indent=2) if shot_characters else 'No characters'}

SPECIAL CONDITIONS:
- Is Last Shot: {is_last_shot}
- Is Product Showcase: {has_product_showcase}

INSTRUCTIONS:
1. If this is a STANDARD SHOT (not final product showcase):
   - Create a detailed cinematic scene description
   - Use positional references for same-gender characters
   - Include all visual elements, lighting, mood
   - Mention product placement if product_image_required is True
   - Note text overlay space if text_overlay is present

2. If this is the FINAL PRODUCT SHOWCASE SHOT:
   - Create an elaborate product showcase prompt
   - Detail product positioning, lighting, background
   - Include complete text overlay layout specifications
   - Use gradients, colors, atmospheric effects
   - Make product the visual centerpiece

Generate the complete, detailed image prompt now."""

        try:
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
            ]
            structured_llm = self.llm.with_structured_output(SceneDescriptionResponse)
            response = structured_llm.invoke(messages)

            print(response)
            
            image_prompt = response.image_prompt
            return image_prompt
            
        except Exception as e:
            print(f"Error generating image prompt for shot {shot.shot_no}: {str(e)}")
            raise

    def generate_scene_descriptions(
        self,
        shots: List[Shot],
        brand_info: Dict[str, Any],
        ad_title: str,
        characters_info: Optional[List[Dict[str, Any]]] = None,
        locations_info: Optional[List[Dict[str, Any]]] = None,
        ad_concept: Optional[Dict[str, Any]] = None,
        outfits_info: Optional[List[Dict[str, Any]]] = None
    ) -> SceneDescription:
        """
        Generate image prompts for all shots
        
        Args:
            shots: List of Shot objects
            brand_info: Brand information
            ad_title: Title of the ad
            characters_info: List of character information
            locations_info: List of location information
            ad_concept: Optional ad concept overview
            
        Returns:
            SceneDescription with all shots containing image prompts
        """
        print("\n" + "="*100)
        print(f"GENERATING SCENE DESCRIPTIONS - {ad_title}")
        print("="*100 + "\n")
        
        total_shots = len(shots)
        shots_with_prompts = []
        
        for idx, shot in enumerate(shots):
            print(f"\n[Shot {shot.shot_no}/{total_shots}] Generating image prompt...")
            
            # Determine if this is the last shot and if it's a product showcase
            is_last_shot = (idx == total_shots - 1)
            has_product_showcase = is_last_shot and shot.product_image_required and shot.text_overlay != "None"
            
            # Generate image prompt
            image_prompt = self.generate_image_prompt_for_shot(
                shot=shot,
                brand_info=brand_info,
                characters_info=characters_info,
                is_last_shot=is_last_shot,
                has_product_showcase=has_product_showcase,
                ad_concept=ad_concept
            )
            
            # Update shot with image prompt
            shot_dict = shot.model_dump()
            shot_dict['image_prompt'] = image_prompt
            updated_shot = Shot(**shot_dict)
            shots_with_prompts.append(updated_shot)
            
            print(f"✓ Image prompt generated ({len(image_prompt)} characters)")
            print(f"  Preview: {image_prompt[:100]}...")
        
        scene_description = SceneDescription(
            ad_title=ad_title,
            total_shots=total_shots,
            shots=shots_with_prompts
        )
        
        print("\n" + "="*100)
        print(f"✓ Scene descriptions complete for all {total_shots} shots!")
        print("="*100 + "\n")
        
        return scene_description

    def save_scene_descriptions(
        self,
        scene_description: SceneDescription,
        output_file: str,
        output_dir: str = "projects_data"
    ):
        """Save scene descriptions to JSON file"""
        scene_dict = scene_description.model_dump()
        
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(scene_dict, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Scene descriptions saved to {file_path}")
        return file_path

    def load_scene_descriptions(self, file_path: str) -> SceneDescription:
        """Load scene descriptions from JSON file"""
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        return SceneDescription(**data)

    def display_scene_descriptions_summary(self, scene_description: SceneDescription):
        """Display summary of scene descriptions"""
        print("\n" + "="*100)
        print(f"SCENE DESCRIPTIONS SUMMARY - {scene_description.ad_title}")
        print("="*100 + "\n")
        
        print(f"Total Shots: {scene_description.total_shots}\n")
        
        for shot in scene_description.shots:
            print(f"\n{'─'*100}")
            print(f"SHOT {shot.shot_no} ({shot.time_stamp}) - {shot.duration}")
            print(f"{'─'*100}")
            print(f"Location: {shot.location}")
            print(f"Camera: {shot.camera_angle}")
            print(f"Characters: {', '.join(shot.characters_involved) if shot.characters_involved else 'None'}")
            print(f"Product Required: {shot.product_image_required}")
            print(f"\nIMAGE PROMPT:")
            print(f"{shot.image_prompt}\n")

    def export_prompts_only(
        self,
        scene_description: SceneDescription,
        output_file: str,
        output_dir: str = "projects_data"
    ):
        """Export only the image prompts to a text file for easy review"""
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(f"IMAGE PROMPTS FOR: {scene_description.ad_title}\n")
            f.write("="*100 + "\n\n")
            
            for shot in scene_description.shots:
                f.write(f"SHOT {shot.shot_no} ({shot.time_stamp})\n")
                f.write(f"Location: {shot.location}\n")
                f.write(f"Camera: {shot.camera_angle}\n")
                f.write(f"Characters: {', '.join(shot.characters_involved)}\n")
                f.write("-"*100 + "\n")
                f.write(f"{shot.image_prompt}\n")
                f.write("="*100 + "\n\n")
        
        print(f"✓ Image prompts exported to {file_path}")
        return file_path


# Example usage
# if __name__ == "__main__":
#     # Example brand info
#     brand_info = {
#         "brand_name": "Deconstruct",
#         "product_name": "Gel Sunscreen SPF 55",
#         "product_description": "Lightweight, matte finish sunscreen",
#         "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
#     }
    
#     # Example characters
#     characters_info = [
#         {
#             "name": "arjun",
#             "age": 28,
#             "gender": "male",
#             "role": "protagonist",
#             "overall_description": "Athletic build, short black hair, confident demeanor, wearing CSK jersey"
#         },
#         {
#             "name": "priya",
#             "age": 25,
#             "gender": "female",
#             "role": "friend",
#             "overall_description": "Cheerful personality, long black hair, wearing casual sports wear"
#         }
#     ]
    
#     # Example shots
#     example_shots = [
#         Shot(
#             shot_no=1,
#             duration="3 seconds",
#             time_stamp="00:00-00:03",
#             location="Outside cricket stadium - bright daylight",
#             camera_angle="Wide shot",
#             visual_description="Two friends standing excitedly outside stadium entrance",
#             action="Arjun and Priya walking towards stadium, chatting excitedly",
#             audio_sfx="Stadium crowd noise in background",
#             dialogue="None",
#             voice_over="None",
#             text_overlay="None",
#             key_focus="Establish setting and characters",
#             product_image_required=False,
#             characters_involved=["arjun", "priya"]
#         ),
#         Shot(
#             shot_no=10,
#             duration="5 seconds",
#             time_stamp="00:27-00:32",
#             location="Clean white background with stadium blur",
#             camera_angle="Center product shot",
#             visual_description="Product showcase with text overlays",
#             action="Static product display with dynamic lighting",
#             audio_sfx="Upbeat music crescendo",
#             dialogue="None",
#             voice_over="Stay protected. Stay confident.",
#             text_overlay="IPL Season Begins. Stay Protected. | Deconstruct | Gel Sunscreen | SPF 55+ | PA+++",
#             key_focus="Product showcase and brand message",
#             product_image_required=True,
#             characters_involved=[]
#         )
#     ]
    
#     # Initialize generator
#     generator = SceneDescriptionGenerator()
    
#     # Generate scene descriptions
#     scene_descriptions = generator.generate_scene_descriptions(
#         shots=example_shots,
#         brand_info=brand_info,
#         ad_title="The Captain's Pre-Match Ritual",
#         characters_info=characters_info
#     )
    
#     # Display summary
#     generator.display_scene_descriptions_summary(scene_descriptions)
    
#     # Save to files
#     generator.save_scene_descriptions(scene_descriptions, "scene_descriptions.json")
#     generator.export_prompts_only(scene_descriptions, "image_prompts.txt")

# Scene Image Generator

In [6]:
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
import json
import os
from dotenv import load_dotenv
from PIL import Image
from google import genai
from google.genai import types
from io import BytesIO
import time


load_dotenv()


class SceneImageGenerator:
    def __init__(self, output_dir: str = "scene_images"):
        """
        Initialize Scene Image Generator
        
        Args:
            output_dir: Directory to store generated scene images
        """
        self.client = genai.Client()
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        self.generation_progress = {
            "total_shots": 0,
            "generated_images": {},
            "failed_generations": []
        }
    
    def load_reference_images(
        self, 
        character_refs: List[Dict], 
        location_ref: Optional[Dict],
        product_ref: Optional[Dict] = None
    ) -> tuple:
        """
        Load reference images for characters, location, and product
        
        Args:
            character_refs: List of character reference dicts with 'name' and 'image_path'
            location_ref: Location reference dict with 'name' and 'image_path'
            product_ref: Product reference dict with 'name' and 'image_path'
            
        Returns:
            Tuple of (prepared_char_refs, prepared_location_ref, prepared_product_ref)
        """
        prepared_char_refs = []
        prepared_location_ref = None
        prepared_product_ref = None
        
        # Load character images
        for char_ref in character_refs:
            char_name = char_ref.get('name', 'unknown')
            char_image_path = char_ref.get('image_path')
            
            if char_image_path and os.path.exists(char_image_path):
                try:
                    loaded_image = Image.open(char_image_path)
                    prepared_char_refs.append({
                        'character_name': char_name,
                        'loaded_image': loaded_image,
                        'image_path': char_image_path
                    })
                    print(f"  ✓ Loaded character image: {char_name}")
                except Exception as e:
                    print(f"  ✗ Failed to load character image {char_name}: {e}")
            else:
                print(f"  ⚠ Character image not found: {char_name}")
        
        # Load location image
        if location_ref:
            loc_name = location_ref.get('name', 'unknown')
            loc_image_path = location_ref.get('image_path')
            
            if loc_image_path and os.path.exists(loc_image_path):
                try:
                    loaded_image = Image.open(loc_image_path)
                    prepared_location_ref = {
                        'location_name': loc_name,
                        'loaded_image': loaded_image,
                        'image_path': loc_image_path
                    }
                    print(f"  ✓ Loaded location image: {loc_name}")
                except Exception as e:
                    print(f"  ✗ Failed to load location image {loc_name}: {e}")
            else:
                print(f"  ⚠ Location image not found: {loc_name}")
        
        # Load product image
        if product_ref:
            prod_name = product_ref.get('name', 'product')
            prod_image_path = product_ref.get('image_path')
            
            if prod_image_path and os.path.exists(prod_image_path):
                try:
                    loaded_image = Image.open(prod_image_path)
                    prepared_product_ref = {
                        'product_name': prod_name,
                        'loaded_image': loaded_image,
                        'image_path': prod_image_path
                    }
                    print(f"  ✓ Loaded product image: {prod_name}")
                except Exception as e:
                    print(f"  ✗ Failed to load product image {prod_name}: {e}")
            else:
                print(f"  ⚠ Product image not found: {prod_name}")
        
        return prepared_char_refs, prepared_location_ref, prepared_product_ref
    
    def generate_scene_image(
    self,
    shot: Dict[str, Any],
    image_prompt: str,
    character_refs: List[Dict],
    location_ref: Optional[Dict],
    product_ref: Optional[Dict],
    outfit_refs: Optional[List[Dict]] = None,
    outfit_character_map: Optional[Dict[str, Dict]] = None,  # NEW: Maps character name to outfit
    aspect_ratio: str = "16:9",
    project_id: str = "project"
) -> Optional[str]:
        """Generate scene image with proper character-outfit mapping"""
        
        shot_no = shot.get('shot_no', 'unknown')
        
        print(f"\n{'─'*80}")
        print(f"🎨 Generating scene image for Shot {shot_no}")
        print(f"{'─'*80}")
        
        try:
            # Load reference images
            print("📂 Loading reference images...")
            prepared_char_refs, prepared_location_ref, prepared_product_ref = \
                self.load_reference_images(character_refs, location_ref, product_ref)
            
            # Load outfit references
            prepared_outfit_refs = []
            if outfit_refs:
                for outfit_ref in outfit_refs:
                    outfit_name = outfit_ref.get('outfit', 'unknown')
                    outfit_image_path = outfit_ref.get('image_path')
                    
                    if outfit_image_path and os.path.exists(outfit_image_path):
                        try:
                            loaded_image = Image.open(outfit_image_path)
                            prepared_outfit_refs.append({
                                'outfit_name': outfit_name,
                                'loaded_image': loaded_image,
                                'image_path': outfit_image_path
                            })
                            print(f"  ✅ Loaded outfit reference: {outfit_name}")
                        except Exception as e:
                            print(f"  ❌ Failed to load outfit image {outfit_name}: {e}")
                    else:
                        print(f"  ⚠️  Outfit image not found: {outfit_name}")
            
            print("outfit")
            print(prepared_outfit_refs)
            enhanced_prompt = self.create_enhanced_prompt_with_mapping(
                base_prompt=image_prompt,
                aspect_ratio=aspect_ratio,
                character_refs=prepared_char_refs,
                location_ref=prepared_location_ref,
                product_ref=prepared_product_ref,
                outfit_refs=prepared_outfit_refs,
                outfit_character_map=outfit_character_map  # NEW
            )
            
            # Prepare content for Gemini
            contents = [enhanced_prompt]
            
            # Add character reference images
            for char_ref in prepared_char_refs:
                if char_ref.get('loaded_image'):
                    contents.append(char_ref['loaded_image'])
                    print(f"  📸 Added character reference: {char_ref['character_name']}")
            
            # Add outfit reference images
            for outfit_ref in prepared_outfit_refs:
                if outfit_ref.get('loaded_image'):
                    contents.append(outfit_ref['loaded_image'])
                    print(f"  👔 Added outfit reference: {outfit_ref['outfit_name']}")
            
            # Add location reference image
            if prepared_location_ref and prepared_location_ref.get('loaded_image'):
                contents.append(prepared_location_ref['loaded_image'])
                print(f"  🏢 Added location reference: {prepared_location_ref['location_name']}")
            
            # Add product reference image
            if prepared_product_ref and prepared_product_ref.get('loaded_image'):
                contents.append(prepared_product_ref['loaded_image'])
                print(f"  📦 Added product reference: {prepared_product_ref['product_name']}")
            
            print(f"\n🎨 Generating with {len(contents)} items (1 prompt + {len(contents)-1} references)")
            
            

            response = self.client.models.generate_content(
                model="gemini-2.5-flash-image",
                contents=contents,
                config=types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=types.ImageConfig(
                    aspect_ratio="16:9",
                )
            )
            )
            
            # Save generated image
            filename = f"{project_id}_shot_{shot_no:03d}_scene.png"
            filepath = os.path.join(self.output_dir, filename)
            
            image_saved = False
            if hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'inline_data') and part.inline_data:
                        image_data = part.inline_data.data
                        image = Image.open(BytesIO(image_data))
                        image.save(filepath)
                        image_saved = True
                        print(f"✅ Saved scene image: {filepath}")
                        break
                    elif hasattr(part, 'text') and part.text:
                        print(f"📝 Gemini response: {part.text[:100]}...")
            
            if not image_saved:
                print(f"❌ No image data generated for Shot {shot_no}")
                self.generation_progress["failed_generations"].append({
                    "shot_no": shot_no,
                    "error": "No image data in response"
                })
                return None
            
            # Update progress tracking
            self.generation_progress["generated_images"][f"shot_{shot_no}"] = {
                "filepath": filepath,
                "shot_no": shot_no,
                "status": "success"
            }
            
            return filepath
            
        except Exception as e:
            print(f"❌ Error generating scene image for Shot {shot_no}: {e}")
            self.generation_progress["failed_generations"].append({
                "shot_no": shot_no,
                "error": str(e)
            })
            return None

    def create_enhanced_prompt_with_mapping(
        self,
        base_prompt: str,
        aspect_ratio: str,
        character_refs: List[Dict],
        location_ref: Optional[Dict],
        product_ref: Optional[Dict],
        outfit_refs: Optional[List[Dict]] = None,
        outfit_character_map: Optional[Dict[str, Dict]] = None
    ) -> str:
        """Create enhanced prompt with explicit character-outfit mapping"""
        
        enhanced_prompt = f"""
    ASPECT RATIO: {aspect_ratio}

    REFERENCE IMAGES PROVIDED:
    """
        
        # Add character reference info with outfit mapping
        if character_refs:
            enhanced_prompt += f"\nCHARACTER REFERENCES ({len(character_refs)} provided):\n"
            for idx, char_ref in enumerate(character_refs, 1):
                char_name = char_ref.get('character_name', 'Character')
                enhanced_prompt += f"- Character Reference {idx}: {char_name}\n"
                enhanced_prompt += f"  Use this for {char_name}'s facial features, body type, and overall appearance\n"
                
                # Add outfit info if mapped
                if outfit_character_map and char_name.lower() in outfit_character_map:
                    outfit_data = outfit_character_map[char_name.lower()]
                    outfit_name = outfit_data.get('outfit', 'outfit')
                    enhanced_prompt += f"  **IMPORTANT**: {char_name} is wearing the '{outfit_name}' - match this outfit exactly from the outfit reference images\n"
        
        # Add outfit reference info with character mapping
        if outfit_refs:
            enhanced_prompt += f"\nOUTFIT REFERENCES ({len(outfit_refs)} provided):\n"
            for idx, outfit_ref in enumerate(outfit_refs, 1):
                outfit_name = outfit_ref.get('outfit_name', 'Outfit')
                enhanced_prompt += f"- Outfit Reference {idx}: {outfit_name}\n"
                
                # Find which character wears this outfit
                if outfit_character_map:
                    wearing_chars = [char for char, outfit in outfit_character_map.items() 
                                if outfit.get('outfit', '').lower() == outfit_name.lower()]
                    if wearing_chars:
                        enhanced_prompt += f"  This outfit is worn by: {', '.join(wearing_chars)}\n"
                
                enhanced_prompt += f"  Match colors, style, fabric, and all details exactly from this reference\n"
        
        # Add location reference info
        if location_ref:
            loc_name = location_ref.get('location_name', 'Location')
            enhanced_prompt += f"\nLOCATION REFERENCE: {loc_name}\n"
            enhanced_prompt += f"- Use this as reference for the setting, environment, and atmosphere\n"
            enhanced_prompt += f"- Match architectural elements, color palette, lighting from this reference\n"
        
        # Add product reference info
        if product_ref:
            prod_name = product_ref.get('product_name', 'Product')
            enhanced_prompt += f"\nPRODUCT REFERENCE: {prod_name}\n"
            enhanced_prompt += f"- Use this for accurate product appearance and packaging\n"
        
        enhanced_prompt += f"""

    CHARACTER-OUTFIT MAPPING FOR THIS SHOT:
    """
        if outfit_character_map:
            for char_name, outfit_data in outfit_character_map.items():
                outfit_name = outfit_data.get('outfit', 'outfit')
                enhanced_prompt += f"- {char_name} wears '{outfit_name}'\n"
        else:
            enhanced_prompt += "- No outfit mapping provided\n"
        
        enhanced_prompt += f"""

    SCENE DESCRIPTION:
    {base_prompt}

    CRITICAL INSTRUCTIONS:
    1. CHARACTER FACES: Use the character reference images to maintain facial consistency, expressions, and body type
    2. OUTFITS: Match each character's outfit EXACTLY from their assigned outfit reference image:
    - Colors and patterns must be identical
    - Style, cut, and fit must match
    - Fabric texture and appearance must be accurate
    - All accessories and details must be included
    3. LOCATION: Use the location reference for environment, atmosphere, and spatial layout
    4. PRODUCT: If present, match the product reference exactly
    5. COMPOSITION: Integrate all references naturally into the scene
    7. QUALITY: Cinematic, commercial-grade photography

    MOST IMPORTANT: Each character MUST wear their assigned outfit from the outfit reference images. Do not mix up outfits between characters.
    """
        
        return enhanced_prompt

    def validate_shot_mapping(
    self,
    shot: Dict[str, Any],
    char_lookup: Dict[str, Any],
    loc_lookup: Dict[str, Any],
    outfit_lookup: Dict[str, Any]
) -> Dict[str, List[str]]:
        """
        Validate that all references in the shot can be found in lookups
        
        Returns:
            Dictionary with 'errors' and 'warnings' lists
        """
        errors = []
        warnings = []
        
        shot_no = shot.get('shot_no', 'unknown')
        
        # Validate characters
        for char_name in shot.get('characters_involved', []):
            if char_name.lower() not in char_lookup:
                errors.append(f"Shot {shot_no}: Character '{char_name}' not found in character lookup")
        
        # Validate location
        if hasattr(shot, 'location_name') and shot.get('location_name'):
            if shot['location_name'].lower() not in loc_lookup:
                errors.append(f"Shot {shot_no}: Location '{shot['location_name']}' not found in location lookup")
        else:
            warnings.append(f"Shot {shot_no}: No location_name field, will use fuzzy matching")
        
        # Validate outfit mappings
        if shot.get('outfit_character_mapping'):
            for mapping in shot['outfit_character_mapping']:
                char_name = mapping.get('character_name', '').lower()
                outfit_name = mapping.get('outfit_name', '').lower()
                
                # Check character exists
                if char_name not in [c.lower() for c in shot.get('characters_involved', [])]:
                    errors.append(f"Shot {shot_no}: Outfit mapping for '{char_name}' but character not in shot")
                
                # Check outfit exists
                if outfit_name not in outfit_lookup:
                    errors.append(f"Shot {shot_no}: Outfit '{outfit_name}' not found in outfit lookup")
        else:
            if shot.get('characters_involved'):
                warnings.append(f"Shot {shot_no}: Has characters but no outfit mappings")
        
        return {"errors": errors, "warnings": warnings}
    
    def generate_all_scene_images(
        self,
        scene_description: 'SceneDescription',
        characters_info: List[Dict[str, Any]],
        locations_info: List[Dict[str, Any]],
        outfits_info: List[Dict[str, Any]],
        product_info: Optional[Dict[str, Any]] = None,
        aspect_ratio: str = "16:9",
        project_id: str = "project",
        delay_between_shots: float = 2.0,
        validate_before_generation: bool = True 
    ) -> Dict[str, Any]:
        """Generate scene images for all shots with proper outfit and location mapping"""
    
        print("\n" + "="*100)
        print(f"GENERATING SCENE IMAGES - {scene_description.ad_title}")
        print(f"Aspect Ratio: {aspect_ratio}")
        print("="*100 + "\n")
        
        self.generation_progress["total_shots"] = len(scene_description.shots)
        
        # Create character lookup
        char_lookup = {char['name'].lower(): char for char in characters_info}
        print(f"📋 Character lookup created: {list(char_lookup.keys())}")
        
        # Create location lookup
        loc_lookup = {loc['name'].lower(): loc for loc in locations_info}
        print(f"📋 Location lookup created: {list(loc_lookup.keys())}")
        
        # Create outfit lookup
        outfit_lookup = {outfit['outfit'].lower(): outfit for outfit in outfits_info}
        print(f"📋 Outfit lookup created: {list(outfit_lookup.keys())}")

        print("Outfit")
        print(outfit_lookup)

        if validate_before_generation:
            print("\n" + "─"*100)
            print("VALIDATING SHOT MAPPINGS")
            print("─"*100)
            
            all_valid = True
            for shot in scene_description.shots:
                validation = self.validate_shot_mapping(
                    shot.model_dump(),
                    char_lookup,
                    loc_lookup,
                    outfit_lookup
                )
                
                if validation['errors']:
                    all_valid = False
                    print(f"\n❌ Shot {shot.shot_no} has ERRORS:")
                    for error in validation['errors']:
                        print(f"   • {error}")
                
                if validation['warnings']:
                    print(f"\n⚠️  Shot {shot.shot_no} has WARNINGS:")
                    for warning in validation['warnings']:
                        print(f"   • {warning}")
            
            if not all_valid:
                print("\n❌ Validation failed! Please fix errors before generating images.")
                return {
                    "error": "Validation failed",
                    "validation_errors": "See console output above"
                }
        else:
            print("\n✅ All shot mappings validated successfully!")
        
        for idx, shot in enumerate(scene_description.shots, 1):
            print(f"\n{'='*100}")
            print(f"[{idx}/{len(scene_description.shots)}] Processing Shot {shot.shot_no}")
            print(f"{'='*100}")
            
            # Get character references for this shot
            character_refs = []
            for char_name in shot.characters_involved:
                char_name_lower = char_name.lower()
                if char_name_lower in char_lookup:
                    character_refs.append(char_lookup[char_name_lower])
                    print(f"  ✅ Character reference: {char_name}")
                else:
                    print(f"  ⚠️  Character not found in lookup: {char_name}")
            
            # Get location reference using location_name field (more accurate)
            location_ref = None
            if hasattr(shot, 'location_name') and shot.location_name:
                location_name_lower = shot.location_name.lower()
                if location_name_lower in loc_lookup:
                    location_ref = loc_lookup[location_name_lower]
                    print(f"  ✅ Location reference: {shot.location_name}")
                else:
                    print(f"  ⚠️  Location '{shot.location_name}' not found in lookup")
                    print(f"      Available locations: {list(loc_lookup.keys())}")
            else:
                # Fallback to fuzzy matching if location_name not provided
                print(f"  ⚠️  No location_name field, using fuzzy match on location description")
                shot_location = shot.location.lower()
                for loc_name, loc_data in loc_lookup.items():
                    if loc_name in shot_location:
                        location_ref = loc_data
                        print(f"  ✅ Location matched (fuzzy): {loc_name}")
                        break
                if not location_ref:
                    print(f"  ❌ No location match found for: {shot.location}")
            
            # Get outfit references with proper character mapping
            outfit_refs = []
            outfit_character_map = {}  # Maps character name to outfit

            print("Outfit Mapping")
            print(shot.outfit_character_mapping)
            
            if hasattr(shot, 'outfit_character_mapping') and shot.outfit_character_mapping:
                print(f"  👔 Processing {len(shot.outfit_character_mapping)} outfit mappings:")
                
                for mapping in shot.outfit_character_mapping:
                    char_name = mapping.character_name.lower()
                    outfit_name = mapping.outfit_name.lower()
                    
                    print(f"     • Mapping: {char_name} → {outfit_name}")
                    
                    # Check if character is actually in this shot
                    if char_name not in [c.lower() for c in shot.characters_involved]:
                        print(f"       ⚠️  Character '{char_name}' not in shot characters_involved")
                        continue
                    
                    # Get outfit from lookup
                    if outfit_name in outfit_lookup:
                        outfit_data = outfit_lookup[outfit_name]
                        outfit_refs.append(outfit_data)
                        outfit_character_map[char_name] = outfit_data
                        print(f"       ✅ Outfit '{outfit_name}' loaded for {char_name}")
                    else:
                        print(f"       ❌ Outfit '{outfit_name}' not found in lookup")
                        print(f"          Available outfits: {list(outfit_lookup.keys())}")
            else:
                print(f"  ⚠️  No outfit_character_mapping provided for this shot")
            
            # Verify all characters have outfits
            for char_name in shot.characters_involved:
                char_name_lower = char_name.lower()
                if char_name_lower not in outfit_character_map:
                    print(f"  ⚠️  WARNING: Character '{char_name}' has no outfit assigned!")
            
            # Get product reference if needed
            product_ref = None
            if shot.product_image_required and product_info:
                product_ref = product_info
                print(f"  📦 Product reference: {product_info.get('name', 'product')}")
            
            # Summary of references
            print(f"\n  📊 Reference Summary for Shot {shot.shot_no}:")
            print(f"     • Characters: {len(character_refs)}")
            print(f"     • Outfits: {len(outfit_refs)}")
            print(f"     • Location: {'Yes' if location_ref else 'No'}")
            print(f"     • Product: {'Yes' if product_ref else 'No'}")

            print(" the prompt for this is")
            print(shot.image_prompt)
            
            # Generate scene image
            image_path = self.generate_scene_image(
                shot=shot.model_dump(),
                image_prompt=shot.image_prompt,
                character_refs=character_refs,
                location_ref=location_ref,
                product_ref=product_ref,
                outfit_refs=outfit_refs,
                outfit_character_map=outfit_character_map,  # NEW: Pass the mapping
                aspect_ratio=aspect_ratio,
                project_id=project_id
            )
            
            # Add delay to avoid rate limiting
            if idx < len(scene_description.shots):
                print(f"\n⏳ Waiting {delay_between_shots}s before next generation...")
                time.sleep(delay_between_shots)
        
        # Print summary
        print("\n" + "="*100)
        print("GENERATION COMPLETE")
        print("="*100)
        print(f"✅ Successfully generated: {len(self.generation_progress['generated_images'])}/{self.generation_progress['total_shots']}")
        print(f"❌ Failed: {len(self.generation_progress['failed_generations'])}")
        
        if self.generation_progress['failed_generations']:
            print("\nFailed shots:")
            for failed in self.generation_progress['failed_generations']:
                print(f"  - Shot {failed['shot_no']}: {failed['error']}")
        
        return self.generation_progress
    
    def save_generation_report(self, output_file: str, output_dir: str = "projects_data"):
        """Save generation progress report to JSON"""
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(self.generation_progress, f, indent=2, ensure_ascii=False)
        
        print(f"📊 Generation report saved to {file_path}")
        return file_path








# Complete Pipeline so far

In [47]:
class CompleteAdProductionPipeline:
    """
    Complete end-to-end pipeline from ad concept to scene images
    """
    def __init__(self, project_id: str, base_output_dir: str = "projects_data"):
        self.project_id = project_id
        self.base_output_dir = base_output_dir
        
        # Create directory structure
        self.dirs = {
            "base": base_output_dir,
            "project": os.path.join(base_output_dir, project_id),
            "characters": os.path.join(base_output_dir, project_id, "character_images"),
            "locations": os.path.join(base_output_dir, project_id, "location_images"),
            "products": os.path.join(base_output_dir, project_id, "product_images"),
            "scenes": os.path.join(base_output_dir, project_id, "scene_images"),
            "scripts": os.path.join(base_output_dir, project_id, "scripts"),
            "prompts": os.path.join(base_output_dir, project_id, "prompts")
        }
        
        for dir_path in self.dirs.values():
            os.makedirs(dir_path, exist_ok=True)
        
        print(f"✅ Created project structure for: {project_id}")
    
    def run_complete_pipeline(
        self,
        ad_concept: Dict[str, Any],
        brand_info: Dict[str, Any],
        product_image_path: Optional[str] = None,
        target_duration: str = "30 seconds",
        aspect_ratio: str = "16:9",
        generate_character_images: bool = True,
        generate_location_images: bool = True,
        generate_scene_images: bool = True
    ) -> Dict[str, Any]:
        """
        Run complete pipeline from ad concept to scene images
        
        Args:
            ad_concept: Ad concept dictionary
            brand_info: Brand information dictionary
            product_image_path: Path to product image (optional)
            target_duration: Target ad duration
            aspect_ratio: Aspect ratio for scene images
            generate_character_images: Whether to generate character images
            generate_location_images: Whether to generate location images
            generate_scene_images: Whether to generate scene images
            
        Returns:
            Dictionary with all generated assets and file paths
        """
        results = {
            "project_id": self.project_id,
            "directories": self.dirs,
            "files": {},
            "assets": {}
        }
        
        print("\n" + "="*100)
        print(f"🎬 STARTING COMPLETE AD PRODUCTION PIPELINE")
        print(f"Project: {self.project_id}")
        print("="*100 + "\n")
        
        # STEP 1: Generate Shot Script
        print("\n" + "─"*100)
        print("STEP 1: SHOT SCRIPT GENERATION")
        print("─"*100)
        
   
        
        shot_generator = ShotScriptGenerator()
        shot_script = shot_generator.generate_shot_script(
            ad_concept=ad_concept,
            brand_info=brand_info
        )
        
        # Save shot script
        shot_script_json = shot_generator.save_shot_script_json(
            shot_script,
            f"{self.project_id}_shot_script.json",
            self.dirs["scripts"]
        )
        
        
        results["files"]["shot_script_json"] = shot_script_json
  
        results["assets"]["shot_script"] = shot_script
        
        # STEP 2: Generate Character Images
        characters_with_images = None
        if generate_character_images and shot_script.characters_info:
            print("\n" + "─"*100)
            print("STEP 2: CHARACTER IMAGE GENERATION")
            print("─"*100)
            
            
            
            # Convert to FullCharacter format
            full_characters = []
            for idx, char in enumerate(shot_script.characters_info, 1):
                full_char = FullCharacter(
                    name=char.name,
                    age=char.age,
                    role=char.role,
                    gender=char.gender,
                    overall_description=char.overall_description
                )
                full_characters.append(full_char)
            
            # Generate images
            char_generator = CharacterGenerator(output_dir=self.dirs["characters"])
            characters_with_images = char_generator.generate_images_for_all_characters(full_characters)
            
            # Save character info
            char_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_characters.json")
            char_generator.save_characters_with_images(characters_with_images, char_json_path)
            
            results["files"]["characters_json"] = char_json_path
            results["assets"]["characters"] = [char.model_dump() for char in characters_with_images]
            
            # Update shot script with image paths
            char_image_map = {char.name.lower(): char.image_path for char in characters_with_images}
            for char_info in shot_script.characters_info:
                if char_info.name.lower() in char_image_map:
                    char_info.image_path = char_image_map[char_info.name.lower()]
        
        # STEP 3: Generate Location Images
        locations_with_images = None
        if generate_location_images and shot_script.location_info:
            print("\n" + "─"*100)
            print("STEP 3: LOCATION IMAGE GENERATION")
            print("─"*100)
            
            
            
            # Convert to FullLocation format
            full_locations = []
            for loc in shot_script.location_info:
                full_loc = FullLocation(
                    name=loc.name,
                    overall_description=loc.overall_description
                )
                full_locations.append(full_loc)
            
            # Generate images
            loc_generator = LocationGenerator(output_dir=self.dirs["locations"])
            locations_with_images = loc_generator.generate_images_for_all_locations(full_locations)
            
            # Save location info
            loc_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_locations.json")
            loc_generator.save_locations_with_images(locations_with_images, loc_json_path)
            
            results["files"]["locations_json"] = loc_json_path
            results["assets"]["locations"] = [loc.model_dump() for loc in locations_with_images]
            
            # Update shot script with image paths
            loc_image_map = {loc.name.lower(): loc.image_path for loc in locations_with_images}
            for loc_info in shot_script.location_info:
                if loc_info.name.lower() in loc_image_map:
                    loc_info.image_path = loc_image_map[loc_info.name.lower()]
        
        # Save updated shot script with all image paths
        shot_script_json = shot_generator.save_shot_script_json(
            shot_script,
            f"{self.project_id}_shot_script_complete.json",
            self.dirs["scripts"]
        )
        results["files"]["shot_script_complete"] = shot_script_json
        
        # STEP 4: Generate Scene Descriptions (Image Prompts)
        print("\n" + "─"*100)
        print("STEP 4: SCENE DESCRIPTION GENERATION")
        print("─"*100)
        
       
        
        scene_desc_generator = SceneDescriptionGenerator()
        scene_descriptions = scene_desc_generator.generate_scene_descriptions(
            shots=shot_script.shots,
            brand_info=brand_info,
            ad_title=shot_script.ad_title,
            characters_info=[char.model_dump() for char in shot_script.characters_info],
            locations_info=[loc.model_dump() for loc in shot_script.location_info],
            ad_concept=ad_concept
        )
        
        # Save scene descriptions
        scene_desc_json = scene_desc_generator.save_scene_descriptions(
            scene_descriptions,
            f"{self.project_id}_scene_descriptions.json",
            self.dirs["prompts"]
        )
        scene_prompts_txt = scene_desc_generator.export_prompts_only(
            scene_descriptions,
            f"{self.project_id}_image_prompts.txt",
            self.dirs["prompts"]
        )
        
        results["files"]["scene_descriptions"] = scene_desc_json
        results["files"]["image_prompts"] = scene_prompts_txt
        results["assets"]["scene_descriptions"] = scene_descriptions
        
        # STEP 5: Generate Scene Images
        if generate_scene_images:
            print("\n" + "─"*100)
            print("STEP 5: SCENE IMAGE GENERATION")
            print("─"*100)
            
            scene_image_generator = SceneImageGenerator(output_dir=self.dirs["scenes"])
            
            # Prepare product info
            product_info = None
            if product_image_path and os.path.exists(product_image_path):
                product_info = {
                    "name": brand_info.get("product_name", "product"),
                    "image_path": product_image_path
                }
            
            # Generate all scene images
            generation_results = scene_image_generator.generate_all_scene_images(
                scene_description=scene_descriptions,
                characters_info=results["assets"].get("characters", []),
                locations_info=results["assets"].get("locations", []),
                product_info=product_info,
                aspect_ratio=aspect_ratio,
                project_id=self.project_id
            )
            
            # Save generation report
            report_path = scene_image_generator.save_generation_report(
                f"{self.project_id}_scene_generation_report.json",
                self.dirs["scripts"]
            )
            
            results["files"]["generation_report"] = report_path
            results["assets"]["scene_images"] = generation_results
        
        # Final Summary
        print("\n" + "="*100)
        print("🎉 PIPELINE COMPLETE!")
        print("="*100)
        print(f"\n📁 Project Directory: {self.dirs['project']}")
        print(f"\n📄 Generated Files:")
        for key, path in results["files"].items():
            print(f"  - {key}: {path}")
        
        return results

In [48]:
# Example usage
if __name__ == "__main__":
    # Example ad concept
    ad_concept = {
        "title": "The Captain's Pre-Match Ritual",
        "one_line_summary": "Dhoni's calm morning routine - applying sunscreen is as essential as checking his bat",
        "story": "MS Dhoni prepares for a crucial match with his signature calm demeanor.",
        "visual_flow": {
            "Opening": "Dhoni in hotel room, early morning light",
            "Sequence": "Checking bat → packing kit → Applying Deconstruct gel sunscreen calmly",
            "Stadium": "Walking out to toss under harsh sun"
        },
        "voice_over": "Champions prepare for everything. Even the sun.",
        "tagline": "Dhoni's choice. Captain Cool stays protected.",
        "key_message": "Preparation and attention to detail",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Example brand info
    brand_info = {
        "brand_name": "Deconstruct",
        "product_name": "Gel Sunscreen SPF 55",
        "product_description": "Lightweight, matte finish sunscreen",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Initialize and run complete pipeline
    pipeline = CompleteAdProductionPipeline(project_id="deconstruct_dhoni_001")
    
    results = pipeline.run_complete_pipeline(
        ad_concept=ad_concept,
        brand_info=brand_info,
        product_image_path="/Users/sanjail/Akaike/Internal_project/ads_poc/product_image.png",  # Optional
        target_duration="45 seconds",
        aspect_ratio="16:9",
        generate_character_images=True,
        generate_location_images=True,
        generate_scene_images=True
    )
    
    print("\n✅ All assets generated successfully!")

✅ Created project structure for: deconstruct_dhoni_001

🎬 STARTING COMPLETE AD PRODUCTION PIPELINE
Project: deconstruct_dhoni_001


────────────────────────────────────────────────────────────────────────────────────────────────────
STEP 1: SHOT SCRIPT GENERATION
────────────────────────────────────────────────────────────────────────────────────────────────────
ad_title="The Captain's Pre-Match Ritual" total_duration='15 seconds' shots=[Shot(shot_no=1, duration='3 seconds', time_stamp='00:00-00:03', location='Hotel Room (Morning)', camera_angle='Wide shot', visual_description='The early morning light fills a minimalistic hotel room with soft beige walls, a neatly made bed with light beige bedding, and a cricket bat resting against the wall. The atmosphere is calm and serene, hinting at a focused morning ritual.', action='MS Dhoni stands at the window, gazing out, taking a deep breath.', audio_sfx='Soft morning ambient sounds mixed with faint bird chirping.', dialogue='None', voice_ove

In [65]:
import os
import json
from typing import Dict, Any, Optional, List
import time


class CompleteAdProductionPipeline:
    """Complete end-to-end pipeline with outfit generation"""
    
    def __init__(self, project_id: str, base_output_dir: str = "projects_data"):
        self.project_id = project_id
        self.base_output_dir = base_output_dir
        
        # Create directory structure
        self.dirs = {
            "base": base_output_dir,
            "project": os.path.join(base_output_dir, project_id),
            "characters": os.path.join(base_output_dir, project_id, "character_images"),
            "locations": os.path.join(base_output_dir, project_id, "location_images"),
            "outfits": os.path.join(base_output_dir, project_id, "outfit_images"),  # NEW
            "products": os.path.join(base_output_dir, project_id, "product_images"),
            "scenes": os.path.join(base_output_dir, project_id, "scene_images"),
            "scripts": os.path.join(base_output_dir, project_id, "scripts"),
            "prompts": os.path.join(base_output_dir, project_id, "prompts")
        }
        
        for dir_path in self.dirs.values():
            os.makedirs(dir_path, exist_ok=True)
        
        # Define expected file paths for each stage
        self.stage_files = {
            "shot_script": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script.json"),
            "shot_script_complete": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script_complete.json"),
            "characters": os.path.join(self.dirs["scripts"], f"{project_id}_characters.json"),
            "locations": os.path.join(self.dirs["scripts"], f"{project_id}_locations.json"),
            "outfits": os.path.join(self.dirs["scripts"], f"{project_id}_outfits.json"),  # NEW
            "scene_descriptions": os.path.join(self.dirs["prompts"], f"{project_id}_scene_descriptions.json"),
            "generation_report": os.path.join(self.dirs["scripts"], f"{project_id}_scene_generation_report.json")
        }
        
        print(f"✅ Initialized project: {project_id}")
        self._check_existing_stages()
        
    def _check_existing_stages(self):
        """Check which stages have already been completed"""
        print("\n📋 Checking existing stages...")
        self.completed_stages = {}
        
        for stage, filepath in self.stage_files.items():
            exists = os.path.exists(filepath)
            self.completed_stages[stage] = exists
            status = "✅ COMPLETED" if exists else "❌ PENDING"
            print(f"  {stage}: {status}")
    def _stage_completed(self, stage: str) -> bool:
        """Check if a stage is completed"""
        return self.completed_stages.get(stage, False)
    
    def _mark_stage_completed(self, stage: str):
        """Mark a stage as completed"""
        self.completed_stages[stage] = True
    
    def _load_json(self, filepath: str) -> Optional[Dict]:
        """Load JSON file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"⚠ Error loading {filepath}: {e}")
            return None
    
    def load_existing_data(self) -> Dict[str, Any]:
        """Load all existing data from previous stages"""
        print("\n" + "="*100)
        print("📂 LOADING EXISTING DATA")
        print("="*100 + "\n")
        
        loaded_data = {
            "shot_script": None,
            "characters": None,
            "locations": None,
            "outfits": None,  # NEW
            "scene_descriptions": None
        }
        
        # Load shot script
        shot_script_path = self.stage_files.get("shot_script_complete") or self.stage_files.get("shot_script")
        if os.path.exists(shot_script_path):
            print(f"📄 Loading shot script from: {shot_script_path}")
            shot_script_data = self._load_json(shot_script_path)
            if shot_script_data:
            
                loaded_data["shot_script"] = ShotScript(**shot_script_data)
                print(f"  ✅ Loaded {len(loaded_data['shot_script'].shots)} shots")
        
        # Load characters
        if os.path.exists(self.stage_files["characters"]):
            print(f"📄 Loading characters from: {self.stage_files['characters']}")
            char_data = self._load_json(self.stage_files["characters"])
            if char_data:
                loaded_data["characters"] = char_data.get("characters", [])
                print(f"  ✅ Loaded {len(loaded_data['characters'])} characters")
        
        # Load locations
        if os.path.exists(self.stage_files["locations"]):
            print(f"📄 Loading locations from: {self.stage_files['locations']}")
            loc_data = self._load_json(self.stage_files["locations"])
            if loc_data:
                loaded_data["locations"] = loc_data.get("locations", [])
                print(f"  ✅ Loaded {len(loaded_data['locations'])} locations")
        
        # NEW: Load outfits
        if os.path.exists(self.stage_files["outfits"]):
            print(f"📄 Loading outfits from: {self.stage_files['outfits']}")
            outfit_data = self._load_json(self.stage_files["outfits"])
            if outfit_data:
                loaded_data["outfits"] = outfit_data.get("outfits", [])
                print(f"  ✅ Loaded {len(loaded_data['outfits'])} outfits")
        
        # Load scene descriptions
        if os.path.exists(self.stage_files["scene_descriptions"]):
            print(f"📄 Loading scene descriptions from: {self.stage_files['scene_descriptions']}")
            scene_data = self._load_json(self.stage_files["scene_descriptions"])
            if scene_data:
               
                loaded_data["scene_descriptions"] = SceneDescription(**scene_data)
                print(f"  ✅ Loaded {len(loaded_data['scene_descriptions'].shots)} scene descriptions")
        
        print("\n" + "="*100)
        print("📂 DATA LOADING COMPLETE")
        print("="*100 + "\n")
        
        return loaded_data
    
    def run_complete_pipeline(
        self,
        ad_concept: Dict[str, Any],
        brand_info: Dict[str, Any],
        product_image_path: Optional[str] = None,
        target_duration: str = "30 seconds",
        aspect_ratio: str = "16:9",
        generate_character_images: bool = True,
        generate_location_images: bool = True,
        generate_outfit_images: bool = True,  # NEW
        generate_scene_images: bool = True,
        force_regenerate: bool = False
    ) -> Dict[str, Any]:
        """Run complete pipeline including outfit generation"""
        
        results = {
            "project_id": self.project_id,
            "directories": self.dirs,
            "files": {},
            "assets": {},
            "stages_executed": [],
            "stages_skipped": []
        }
        
        print("\n" + "="*100)
        print(f"🎬 STARTING COMPLETE AD PRODUCTION PIPELINE")
        print(f"Project: {self.project_id}")
        print(f"Force Regenerate: {force_regenerate}")
        print("="*100 + "\n")
        
        
        # ====================================================================
        # STEP 1: Generate Shot Script
        # ====================================================================
        print("\n" + "─"*100)
        print("STEP 1: SHOT SCRIPT GENERATION")
        print("─"*100)
        
        shot_script = None
        
        if force_regenerate or not self._stage_completed("shot_script"):
            print("🔄 Generating shot script...")
            
            
            
            shot_generator = ShotScriptGenerator()
            shot_script = shot_generator.generate_shot_script(
                ad_concept=ad_concept,
                brand_info=brand_info
            )
            
            # Save shot script
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script.json",
                self.dirs["scripts"]
            )
            
            
            results["files"]["shot_script_json"] = shot_script_json
          
            
            self._mark_stage_completed("shot_script")
            results["stages_executed"].append("shot_script_generation")
        else:
            print("✅ Shot script already exists, loading from file...")
            shot_script_data = self._load_json(self.stage_files["shot_script"])
            
            if shot_script_data:
            
                shot_script = ShotScript(**shot_script_data)
                results["files"]["shot_script_json"] = self.stage_files["shot_script"]
                results["stages_skipped"].append("shot_script_generation")
            else:
                print("⚠ Failed to load existing shot script, regenerating...")
                force_regenerate = True  # Force regeneration of subsequent stages
                return self.run_complete_pipeline(
                    ad_concept, brand_info, product_image_path, target_duration,
                    aspect_ratio, generate_character_images, generate_location_images,
                    generate_scene_images, force_regenerate=True
                )
        
        results["assets"]["shot_script"] = shot_script
        
        # ====================================================================
        # STEP 2: Generate Character Images
        # ====================================================================
        characters_with_images = None
        
        if generate_character_images and shot_script.characters_info:
            print("\n" + "─"*100)
            print("STEP 2: CHARACTER IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("characters"):
                print(f"🔄 Generating images for {len(shot_script.characters_info)} characters...")
                
                
                
                # Convert to FullCharacter format
                full_characters = []
                for idx, char in enumerate(shot_script.characters_info, 1):
                    full_char = FullCharacter(
                        name=char.name,
                        age=char.age,
                        role=char.role,
                        gender=char.gender,
                        overall_description=char.overall_description
                    )
                    full_characters.append(full_char)
                
                # Generate images
                char_generator = CharacterGenerator(output_dir=self.dirs["characters"])
                characters_with_images = char_generator.generate_images_for_all_characters(full_characters)
                
                # Save character info
                char_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_characters.json")
                char_generator.save_characters_with_images(characters_with_images, char_json_path)
                
                results["files"]["characters_json"] = char_json_path
                
                self._mark_stage_completed("characters")
                results["stages_executed"].append("character_image_generation")
            else:
                print("✅ Character images already exist, loading from file...")
                char_data = self._load_json(self.stage_files["characters"])
                
                if char_data:
                    
                    characters_with_images = [
                        FullCharacter(**char) for char in char_data.get("characters", [])
                    ]
                    results["files"]["characters_json"] = self.stage_files["characters"]
                    results["stages_skipped"].append("character_image_generation")
                else:
                    print("⚠ Failed to load existing character data")
            
            if characters_with_images:
                results["assets"]["characters"] = [char.model_dump() for char in characters_with_images]
                
                # Update shot script with image paths
                char_image_map = {char.name.lower(): char.image_path for char in characters_with_images}
                for char_info in shot_script.characters_info:
                    if char_info.name.lower() in char_image_map:
                        char_info.image_path = char_image_map[char_info.name.lower()]
        
        # ====================================================================
        # STEP 3: Generate Location Images
        # ====================================================================
        locations_with_images = None
        
        if generate_location_images and shot_script.location_info:
            print("\n" + "─"*100)
            print("STEP 3: LOCATION IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("locations"):
                print(f"🔄 Generating images for {len(shot_script.location_info)} locations...")
                
              
                
                # Convert to FullLocation format
                full_locations = []
                for loc in shot_script.location_info:
                    full_loc = FullLocation(
                        name=loc.name,
                        overall_description=loc.overall_description
                    )
                    full_locations.append(full_loc)
                
                # Generate images
                loc_generator = LocationGenerator(output_dir=self.dirs["locations"])
                locations_with_images = loc_generator.generate_images_for_all_locations(full_locations)
                
                # Save location info
                loc_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_locations.json")
                loc_generator.save_locations_with_images(locations_with_images, loc_json_path)
                
                results["files"]["locations_json"] = loc_json_path
                
                self._mark_stage_completed("locations")
                results["stages_executed"].append("location_image_generation")
            else:
                print("✅ Location images already exist, loading from file...")
                loc_data = self._load_json(self.stage_files["locations"])
                
                if loc_data:
               
                    locations_with_images = [
                        FullLocation(**loc) for loc in loc_data.get("locations", [])
                    ]
                    results["files"]["locations_json"] = self.stage_files["locations"]
                    results["stages_skipped"].append("location_image_generation")
                else:
                    print("⚠ Failed to load existing location data")
            
            if locations_with_images:
                results["assets"]["locations"] = [loc.model_dump() for loc in locations_with_images]
                
                # Update shot script with image paths
                loc_image_map = {loc.name.lower(): loc.image_path for loc in locations_with_images}
                for loc_info in shot_script.location_info:
                    if loc_info.name.lower() in loc_image_map:
                        loc_info.image_path = loc_image_map[loc_info.name.lower()]
        
        # Save updated shot script with all image paths
        if characters_with_images or locations_with_images:
       
            shot_generator = ShotScriptGenerator()
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script_complete.json",
                self.dirs["scripts"]
            )
            results["files"]["shot_script_complete"] = shot_script_json
            self._mark_stage_completed("shot_script_complete")

          # ====================================================================
        # STEP 4: Generate Outfit Images (NEW)
        # ====================================================================
        outfits_with_images = None
        
        if generate_outfit_images and shot_script.character_outfit_info:
            print("\n" + "─"*100)
            print("STEP 4: OUTFIT IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("outfits"):
                print(f"🔄 Generating images for {len(shot_script.character_outfit_info)} outfits...")
                
                
                
                # Convert to FullOutfit format
                full_outfits = []
                for outfit_info in shot_script.character_outfit_info:
                    full_outfit = FullOutfit(
                        outfit=outfit_info.outfit,
                        outfit_description=outfit_info.outfit_description
                    )
                    full_outfits.append(full_outfit)
                
                # Generate images
                outfit_generator = OutfitGenerator(output_dir=self.dirs["outfits"])
                outfits_with_images = outfit_generator.generate_images_for_all_outfits(full_outfits)
                
                # Save outfit info
                outfit_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_outfits.json")
                outfit_generator.save_outfits_with_images(outfits_with_images, outfit_json_path)
                
                results["files"]["outfits_json"] = outfit_json_path
                
                self._mark_stage_completed("outfits")
                results["stages_executed"].append("outfit_image_generation")
            else:
                print("✅ Outfit images already exist, loading from file...")
                outfit_data = self._load_json(self.stage_files["outfits"])
                
                if outfit_data:
                    
                    outfits_with_images = [
                        FullOutfit(**outfit) for outfit in outfit_data.get("outfits", [])
                    ]
                    results["files"]["outfits_json"] = self.stage_files["outfits"]
                    results["stages_skipped"].append("outfit_image_generation")
                else:
                    print("⚠ Failed to load existing outfit data")
            
            if outfits_with_images:
                results["assets"]["outfits"] = [outfit.model_dump() for outfit in outfits_with_images]
                
                # Update shot script with image paths
                outfit_image_map = {outfit.outfit.lower(): outfit.image_path for outfit in outfits_with_images}
                for outfit_info in shot_script.character_outfit_info:
                    if outfit_info.outfit.lower() in outfit_image_map:
                        outfit_info.image_path = outfit_image_map[outfit_info.outfit.lower()]
        
        # Save updated shot script
        if characters_with_images or locations_with_images or outfits_with_images:
            
            shot_generator = ShotScriptGenerator()
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script_complete.json",
                self.dirs["scripts"]
            )
            results["files"]["shot_script_complete"] = shot_script_json
            self._mark_stage_completed("shot_script_complete")
        
        # ====================================================================
        # STEP 5: Generate Scene Descriptions (Updated with outfits)
        # ====================================================================
        print("\n" + "─"*100)
        print("STEP 5: SCENE DESCRIPTION GENERATION")
        print("─"*100)
        
        scene_descriptions = None
        
        if force_regenerate or not self._stage_completed("scene_descriptions"):
            print("🔄 Generating scene descriptions and image prompts...")
            
           
            
            scene_desc_generator = SceneDescriptionGenerator()
            scene_descriptions = scene_desc_generator.generate_scene_descriptions(
                shots=shot_script.shots,
                brand_info=brand_info,
                ad_title=shot_script.ad_title,
                characters_info=[char.model_dump() for char in shot_script.characters_info],
                locations_info=[loc.model_dump() for loc in shot_script.location_info],
                outfits_info=results["assets"].get("outfits", []),  # NEW: Pass outfits
                ad_concept=ad_concept
            )
            
            # Save scene descriptions
            scene_desc_json = scene_desc_generator.save_scene_descriptions(
                scene_descriptions,
                f"{self.project_id}_scene_descriptions.json",
                self.dirs["prompts"]
            )
            scene_prompts_txt = scene_desc_generator.export_prompts_only(
                scene_descriptions,
                f"{self.project_id}_image_prompts.txt",
                self.dirs["prompts"]
            )
            
            results["files"]["scene_descriptions"] = scene_desc_json
            results["files"]["image_prompts"] = scene_prompts_txt
            
            self._mark_stage_completed("scene_descriptions")
            results["stages_executed"].append("scene_description_generation")
        else:
            print("✅ Scene descriptions already exist, loading from file...")
            scene_desc_data = self._load_json(self.stage_files["scene_descriptions"])
            
            if scene_desc_data:
        
                scene_descriptions = SceneDescription(**scene_desc_data)
                results["files"]["scene_descriptions"] = self.stage_files["scene_descriptions"]
                results["stages_skipped"].append("scene_description_generation")
        
        results["assets"]["scene_descriptions"] = scene_descriptions
        
        # ====================================================================
        # STEP 5: Generate Scene Images
        # ====================================================================
        # In CompleteAdProductionPipeline.run_complete_pipeline()

# STEP 6: Generate Scene Images (Updated)
        if generate_scene_images and scene_descriptions:
            print("\n" + "─"*100)
            print("STEP 6: SCENE IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("generation_report"):
                print("🔄 Generating scene images...")
                
               
                
                scene_image_generator = SceneImageGenerator(output_dir=self.dirs["scenes"])
                
                # Prepare product info
                product_info = None
                if product_image_path and os.path.exists(product_image_path):
                    product_info = {
                        "name": brand_info.get("product_name", "product"),
                        "image_path": product_image_path
                    }
                    print("Character_info")
                    print(results["assets"].get("characters", []))

                    print("Location Info")
                    print(results["assets"].get("locations", []))

                    print("Outfit Info")
                    print(results["assets"].get("outfits", []))
                
                # Generate all scene images with outfit references
                generation_results = scene_image_generator.generate_all_scene_images(
                    scene_description=scene_descriptions,
                    characters_info=results["assets"].get("characters", []),
                    locations_info=results["assets"].get("locations", []),
                    outfits_info=results["assets"].get("outfits", []),  # NEW: Pass outfits
                    product_info=product_info,
                    aspect_ratio=aspect_ratio,
                    project_id=self.project_id
                )
                
                # Save generation report
                report_path = scene_image_generator.save_generation_report(
                    f"{self.project_id}_scene_generation_report.json",
                    self.dirs["scripts"]
                )
                
                results["files"]["generation_report"] = report_path
                results["assets"]["scene_images"] = generation_results
                
                self._mark_stage_completed("generation_report")
                results["stages_executed"].append("scene_image_generation")
            else:
                print("✅ Scene images already generated, loading report...")
                report_data = self._load_json(self.stage_files["generation_report"])
                
                if report_data:
                    results["files"]["generation_report"] = self.stage_files["generation_report"]
                    results["assets"]["scene_images"] = report_data
                    results["stages_skipped"].append("scene_image_generation")
                else:
                    print("⚠ Failed to load existing generation report")
                
        # ====================================================================
        # Final Summary
        # ====================================================================
        print("\n" + "="*100)
        print("🎉 PIPELINE COMPLETE!")
        print("="*100)
        print(f"\n📁 Project Directory: {self.dirs['project']}")
        
        if results["stages_executed"]:
            print(f"\n✅ Stages Executed ({len(results['stages_executed'])}):")
            for stage in results["stages_executed"]:
                print(f"  - {stage}")
        
        if results["stages_skipped"]:
            print(f"\n⏭️  Stages Skipped ({len(results['stages_skipped'])}):")
            for stage in results["stages_skipped"]:
                print(f"  - {stage}")
        
        print(f"\n📄 Generated/Loaded Files:")
        for key, path in results["files"].items():
            print(f"  - {key}: {path}")
        
        return results
    
    def reset_stage(self, stage: str):
        """
        Reset a specific stage by deleting its output files
        
        Args:
            stage: Stage name (shot_script, characters, locations, scene_descriptions, generation_report)
        """
        if stage not in self.stage_files:
            print(f"⚠ Unknown stage: {stage}")
            return
        
        filepath = self.stage_files[stage]
        
        if os.path.exists(filepath):
            os.remove(filepath)
            print(f"🗑️  Deleted: {filepath}")
            self.completed_stages[stage] = False
        else:
            print(f"⚠ File not found: {filepath}")
    
    def reset_all_stages(self):
        """Reset all stages"""
        print("🗑️  Resetting all stages...")
        for stage in self.stage_files.keys():
            self.reset_stage(stage)
        print("✅ All stages reset")
    
    def get_pipeline_status(self) -> Dict[str, Any]:
        """Get current status of all pipeline stages"""
        self._check_existing_stages()
        
        status = {
            "project_id": self.project_id,
            "completed_stages": [],
            "pending_stages": [],
            "stage_details": {}
        }
        
        for stage, completed in self.completed_stages.items():
            if completed:
                status["completed_stages"].append(stage)
                status["stage_details"][stage] = {
                    "status": "completed",
                    "file": self.stage_files[stage]
                }
            else:
                status["pending_stages"].append(stage)
                status["stage_details"][stage] = {
                    "status": "pending",
                    "file": self.stage_files[stage]
                }
        
        return status


# Example usage
if __name__ == "__main__":
    # Example ad concept
    ad_concept = {
        "title": "The Captain's Pre-Match Ritual",
        "one_line_summary": "Dhoni's calm morning routine - applying sunscreen is as essential as checking his bat",
        "story": "MS Dhoni prepares for a crucial match with his signature calm demeanor.",
        "visual_flow": {
            "Opening": "Dhoni in hotel room, early morning light",
            "Sequence": "Checking bat → packing kit → Applying Deconstruct gel sunscreen calmly",
            "Stadium": "Walking out to toss under harsh sun"
        },
        "voice_over": "Champions prepare for everything. Even the sun.",
        "tagline": "Dhoni's choice. Captain Cool stays protected.",
        "key_message": "Preparation and attention to detail",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Example brand info
    brand_info = {
        "brand_name": "Deconstruct",
        "product_name": "Gel Sunscreen SPF 55",
        "product_description": "Lightweight, matte finish sunscreen",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Initialize pipeline
    pipeline = CompleteAdProductionPipeline(project_id="deconstruct_dhoni_004")
    
    # Check current status
    status = pipeline.get_pipeline_status()
    print("\n📊 Current Pipeline Status:")
    print(f"Completed: {status['completed_stages']}")
    print(f"Pending: {status['pending_stages']}")
    
    # Run pipeline (will skip completed stages automatically)
    results = pipeline.run_complete_pipeline(
        ad_concept=ad_concept,
        brand_info=brand_info,
        product_image_path="/Users/sanjail/Akaike/Internal_project/ads_poc/product_image.png",
        target_duration="45 seconds",
        aspect_ratio="16:9",
        generate_character_images=True,
        generate_location_images=True,
        generate_outfit_images=True,
        generate_scene_images=True,
        force_regenerate=False  # Set to True to regenerate everything
    )
    
    print("\n✅ Pipeline execution complete!")
    
    # Example: Reset a specific stage if you want to regenerate it
    # pipeline.reset_stage("scene_descriptions")
    
    # Example: Reset all stages
    # pipeline.reset_all_stages()

✅ Initialized project: deconstruct_dhoni_004

📋 Checking existing stages...
  shot_script: ✅ COMPLETED
  shot_script_complete: ✅ COMPLETED
  characters: ✅ COMPLETED
  locations: ✅ COMPLETED
  outfits: ✅ COMPLETED
  scene_descriptions: ✅ COMPLETED
  generation_report: ❌ PENDING

📋 Checking existing stages...
  shot_script: ✅ COMPLETED
  shot_script_complete: ✅ COMPLETED
  characters: ✅ COMPLETED
  locations: ✅ COMPLETED
  outfits: ✅ COMPLETED
  scene_descriptions: ✅ COMPLETED
  generation_report: ❌ PENDING

📊 Current Pipeline Status:
Completed: ['shot_script', 'shot_script_complete', 'characters', 'locations', 'outfits', 'scene_descriptions']
Pending: ['generation_report']

🎬 STARTING COMPLETE AD PRODUCTION PIPELINE
Project: deconstruct_dhoni_004
Force Regenerate: False


────────────────────────────────────────────────────────────────────────────────────────────────────
STEP 1: SHOT SCRIPT GENERATION
──────────────────────────────────────────────────────────────────────────────────────

In [ ]:
#

# Debug

In [ ]:
import os
import json
from typing import Dict, Any, Optional, List
import time


class Pipeline:
    """
    Complete end-to-end pipeline from ad concept to scene images with stage completion checks
    """
    def __init__(self, project_id: str, base_output_dir: str = "projects_data"):
        self.project_id = project_id
        self.base_output_dir = base_output_dir
        
        # Create directory structure
        self.dirs = {
            "base": base_output_dir,
            "project": os.path.join(base_output_dir, project_id),
            "characters": os.path.join(base_output_dir, project_id, "character_images"),
            "locations": os.path.join(base_output_dir, project_id, "location_images"),
            "products": os.path.join(base_output_dir, project_id, "product_images"),
            "scenes": os.path.join(base_output_dir, project_id, "scene_images"),
            "scripts": os.path.join(base_output_dir, project_id, "scripts"),
            "prompts": os.path.join(base_output_dir, project_id, "prompts")
        }
        
        for dir_path in self.dirs.values():
            os.makedirs(dir_path, exist_ok=True)
        
        # Define expected file paths for each stage
        self.stage_files = {
            "shot_script": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script.json"),
            "shot_script_complete": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script_complete.json"),
            "characters": os.path.join(self.dirs["scripts"], f"{project_id}_characters.json"),
            "locations": os.path.join(self.dirs["scripts"], f"{project_id}_locations.json"),
            "scene_descriptions": os.path.join(self.dirs["prompts"], f"{project_id}_scene_descriptions.json"),
            "generation_report": os.path.join(self.dirs["scripts"], f"{project_id}_scene_generation_report.json")
        }
        
        print(f"✅ Initialized project: {project_id}")
        self._check_existing_stages()
    
    def _check_existing_stages(self):
        """Check which stages have already been completed"""
        print("\n📋 Checking existing stages...")
        self.completed_stages = {}
        
        for stage, filepath in self.stage_files.items():
            exists = os.path.exists(filepath)
            self.completed_stages[stage] = exists
            status = "✅ COMPLETED" if exists else "❌ PENDING"
            print(f"  {stage}: {status}")
    
    def _load_json(self, filepath: str) -> Optional[Dict]:
        """Load JSON file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"⚠ Error loading {filepath}: {e}")
            return None
    
    def _stage_completed(self, stage: str) -> bool:
        """Check if a stage is completed"""
        return self.completed_stages.get(stage, False)
    
    def _mark_stage_completed(self, stage: str):
        """Mark a stage as completed"""
        self.completed_stages[stage] = True
    
    def load_existing_data(self) -> Dict[str, Any]:
        """
        Load all existing data from previous stages
        
        Returns:
            Dictionary with loaded data
        """
        print("\n" + "="*100)
        print("📂 LOADING EXISTING DATA")
        print("="*100 + "\n")
        
        loaded_data = {
            "shot_script": None,
            "characters": None,
            "locations": None,
            "scene_descriptions": None
        }
        
        # Load shot script
        shot_script_path = self.stage_files.get("shot_script_complete") or self.stage_files.get("shot_script")
        if os.path.exists(shot_script_path):
            print(f"📄 Loading shot script from: {shot_script_path}")
            shot_script_data = self._load_json(shot_script_path)
            if shot_script_data:
                
                loaded_data["shot_script"] = ShotScript(**shot_script_data)
                print(f"  ✅ Loaded {len(loaded_data['shot_script'].shots)} shots")
            else:
                print("  ❌ Failed to load shot script")
        else:
            print("  ⚠ Shot script not found")
        
        # Load characters
        if os.path.exists(self.stage_files["characters"]):
            print(f"📄 Loading characters from: {self.stage_files['characters']}")
            char_data = self._load_json(self.stage_files["characters"])
            if char_data:
                loaded_data["characters"] = char_data.get("characters", [])
                print(f"  ✅ Loaded {len(loaded_data['characters'])} characters")
            else:
                print("  ❌ Failed to load characters")
        else:
            print("  ⚠ Characters data not found")
        
        # Load locations
        if os.path.exists(self.stage_files["locations"]):
            print(f"📄 Loading locations from: {self.stage_files['locations']}")
            loc_data = self._load_json(self.stage_files["locations"])
            if loc_data:
                loaded_data["locations"] = loc_data.get("locations", [])
                print(f"  ✅ Loaded {len(loaded_data['locations'])} locations")
            else:
                print("  ❌ Failed to load locations")
        else:
            print("  ⚠ Locations data not found")
        
        # Load scene descriptions
        if os.path.exists(self.stage_files["scene_descriptions"]):
            print(f"📄 Loading scene descriptions from: {self.stage_files['scene_descriptions']}")
            scene_data = self._load_json(self.stage_files["scene_descriptions"])
            if scene_data:
                
                loaded_data["scene_descriptions"] = SceneDescription(**scene_data)
                print(f"  ✅ Loaded {len(loaded_data['scene_descriptions'].shots)} scene descriptions")
            else:
                print("  ❌ Failed to load scene descriptions")
        else:
            print("  ⚠ Scene descriptions not found")
        
        print("\n" + "="*100)
        print("📂 DATA LOADING COMPLETE")
        print("="*100 + "\n")
        
        return loaded_data
    
    # Add to CompleteAdProductionPipeline class

def debug_scene_description_generation(
    self,
    brand_info: Dict[str, Any],
    ad_concept: Optional[Dict[str, Any]] = None,
    force_regenerate: bool = False
) -> Dict[str, Any]:
    """Debug scene description generation with outfit support"""
    
    print("\n" + "="*100)
    print("🐛 DEBUG: SCENE DESCRIPTION GENERATION ONLY")
    print("="*100 + "\n")
    
    # Load existing data
    loaded_data = self.load_existing_data()
    
    if not loaded_data["shot_script"]:
        print("❌ ERROR: Shot script not found.")
        return {"error": "Shot script not found"}
    
    shot_script = loaded_data["shot_script"]
    characters_info = loaded_data["characters"] or []
    locations_info = loaded_data["locations"] or []
    outfits_info = loaded_data["outfits"] or []  # NEW
    
    print(f"✅ Loaded data:")
    print(f"   - Shots: {len(shot_script.shots)}")
    print(f"   - Characters: {len(characters_info)}")
    print(f"   - Locations: {len(locations_info)}")
    print(f"   - Outfits: {len(outfits_info)}")  # NEW
    
    # Generate scene descriptions
    print("\n" + "─"*100)
    print("GENERATING SCENE DESCRIPTIONS")
    print("─"*100)
    
    if force_regenerate or not self._stage_completed("scene_descriptions"):
        print("🔄 Generating scene descriptions and image prompts...")
        
        f
        
        scene_desc_generator = SceneDescriptionGenerator()
        scene_descriptions = scene_desc_generator.generate_scene_descriptions(
            shots=shot_script.shots,
            brand_info=brand_info,
            ad_title=shot_script.ad_title,
            characters_info=characters_info,
            locations_info=locations_info,
            outfits_info=outfits_info,  # NEW
            ad_concept=ad_concept
        )
        
        # Save scene descriptions
        scene_desc_json = scene_desc_generator.save_scene_descriptions(
            scene_descriptions,
            f"{self.project_id}_scene_descriptions.json",
            self.dirs["prompts"]
        )
        scene_prompts_txt = scene_desc_generator.export_prompts_only(
            scene_descriptions,
            f"{self.project_id}_image_prompts.txt",
            self.dirs["prompts"]
        )
        
        scene_desc_generator.display_scene_descriptions_summary(scene_descriptions)
        
        self._mark_stage_completed("scene_descriptions")
        
        print("\n✅ Scene descriptions generated successfully!")
        
        return {
            "scene_descriptions": scene_descriptions,
            "files": {
                "scene_descriptions_json": scene_desc_json,
                "image_prompts_txt": scene_prompts_txt
            }
        }
    else:
        print("✅ Scene descriptions already exist")
        scene_descriptions = loaded_data["scene_descriptions"]
        
        return {
            "scene_descriptions": scene_descriptions,
            "files": {
                "scene_descriptions_json": self.stage_files["scene_descriptions"]
            }
        }

def debug_scene_image_generation(
    self,
    brand_info: Dict[str, Any],
    product_image_path: Optional[str] = None,
    aspect_ratio: str = "16:9",
    shot_numbers: Optional[List[int]] = None,
    force_regenerate: bool = False,
    delay_between_shots: float = 2.0
) -> Dict[str, Any]:
    """Debug scene image generation with outfit support"""
    
    print("\n" + "="*100)
    print("🐛 DEBUG: SCENE IMAGE GENERATION ONLY")
    print("="*100 + "\n")
    
    # Load existing data
    loaded_data = self.load_existing_data()
    
    if not loaded_data["scene_descriptions"]:
        print("❌ ERROR: Scene descriptions not found.")
        return {"error": "Scene descriptions not found"}
    
    scene_descriptions = loaded_data["scene_descriptions"]
    characters_info = loaded_data["characters"] or []
    locations_info = loaded_data["locations"] or []
    outfits_info = loaded_data["outfits"] or []  # NEW
    
    print(f"✅ Loaded data:")
    print(f"   - Scene descriptions: {len(scene_descriptions.shots)}")
    print(f"   - Characters: {len(characters_info)}")
    print(f"   - Locations: {len(locations_info)}")
    print(f"   - Outfits: {len(outfits_info)}")  # NEW
    
    # Filter shots if specific shot numbers are provided
    shots_to_generate = scene_descriptions.shots
    if shot_numbers:
        shots_to_generate = [shot for shot in scene_descriptions.shots if shot.shot_no in shot_numbers]
        print(f"🎯 Filtering to specific shots: {shot_numbers}")
        print(f"   Found {len(shots_to_generate)} shots to generate")
    
    if not shots_to_generate:
        print("❌ ERROR: No shots found to generate")
        return {"error": "No shots found"}
    
    # Generate scene images
    print("\n" + "─"*100)
    print("GENERATING SCENE IMAGES")
    print("─"*100)
    
    
    
    scene_image_generator = SceneImageGenerator(output_dir=self.dirs["scenes"])
    
    # Prepare product info
    product_info = None
    if product_image_path and os.path.exists(product_image_path):
        product_info = {
            "name": brand_info.get("product_name", "product"),
            "image_path": product_image_path
        }
        print(f"📦 Product image loaded: {product_image_path}")
    
    # Create lookups
    char_lookup = {char['name'].lower(): char for char in characters_info}
    loc_lookup = {loc['name'].lower(): loc for loc in locations_info}
    outfit_lookup = {outfit['outfit'].lower(): outfit for outfit in outfits_info}  # NEW
    
    # Generate images for selected shots
    scene_image_generator.generation_progress["total_shots"] = len(shots_to_generate)
    
    for idx, shot in enumerate(shots_to_generate, 1):
        print(f"\n[{idx}/{len(shots_to_generate)}] Processing Shot {shot.shot_no}")
        
        # Get character references
        character_refs = []
        for char_name in shot.characters_involved:
            if char_name.lower() in char_lookup:
                character_refs.append(char_lookup[char_name.lower()])
        
        # Get location reference
        location_ref = None
        shot_location = shot.location.lower()
        for loc_name, loc_data in loc_lookup.items():
            if loc_name in shot_location:
                location_ref = loc_data
                break
        
        # NEW: Get outfit references based on mapping
        outfit_refs = []
        if hasattr(shot, 'outfit_character_mapping') and shot.outfit_character_mapping:
            for mapping in shot.outfit_character_mapping:
                outfit_name = mapping.outfit_name.lower()
                if outfit_name in outfit_lookup:
                    outfit_refs.append(outfit_lookup[outfit_name])
                    print(f"  👔 Using outfit: {outfit_name} for {mapping.character_name}")
        
        # Get product reference if needed
        product_ref = None
        if shot.product_image_required and product_info:
            product_ref = product_info
        
        # Generate scene image
        image_path = scene_image_generator.generate_scene_image(
            shot=shot.model_dump(),
            image_prompt=shot.image_prompt,
            character_refs=character_refs,
            location_ref=location_ref,
            product_ref=product_ref,
            outfit_refs=outfit_refs,  # NEW
            aspect_ratio=aspect_ratio,
            project_id=self.project_id
        )
        
        # Add delay
        if idx < len(shots_to_generate):
            print(f"⏳ Waiting {delay_between_shots}s before next generation...")
            time.sleep(delay_between_shots)
    
    # Save generation report
    report_path = scene_image_generator.save_generation_report(
        f"{self.project_id}_scene_generation_report_debug.json",
        self.dirs["scripts"]
    )
    
    # Print summary
    print("\n" + "="*100)
    print("GENERATION COMPLETE")
    print("="*100)
    print(f"✅ Successfully generated: {len(scene_image_generator.generation_progress['generated_images'])}/{scene_image_generator.generation_progress['total_shots']}")
    print(f"❌ Failed: {len(scene_image_generator.generation_progress['failed_generations'])}")
    
    return {
        "generation_progress": scene_image_generator.generation_progress,
        "files": {
            "generation_report": report_path
        }
    }
    
        

In [11]:
# ============================================================================
# DEBUG EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Example brand info
    brand_info = {
        "brand_name": "Deconstruct",
        "product_name": "Gel Sunscreen SPF 55",
        "product_description": "Lightweight, matte finish sunscreen",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Example ad concept (optional for scene description)
    ad_concept = {
        "title": "The Captain's Pre-Match Ritual",
        "one_line_summary": "Dhoni's calm morning routine - applying sunscreen is as essential as checking his bat",
        "story": "MS Dhoni prepares for a crucial match with his signature calm demeanor.",
        "voice_over": "Champions prepare for everything. Even the sun.",
        "tagline": "Dhoni's choice. Captain Cool stays protected.",
        "key_message": "Preparation and attention to detail",
    }
    
    # Initialize pipeline
    pipeline = Pipeline(project_id="deconstruct_dhoni_001")
    
    # Check current status
    # print("\n" + "="*100)
    # print("CHECKING PIPELINE STATUS")
    # print("="*100)
    # status = pipeline.get_pipeline_status()
    # print(f"\n✅ Completed: {status['completed_stages']}")
    # print(f"❌ Pending: {status['pending_stages']}")
    
    # ========================================================================
    # DEBUG OPTION 1: Test only scene description generation
    # ========================================================================
    print("\n\n" + "="*100)
    print("DEBUG OPTION 1: SCENE DESCRIPTION GENERATION ONLY")
    print("="*100)
    
    # This will load shot script, characters, and locations from existing files
    # and generate scene descriptions
    desc_results = pipeline.debug_scene_description_generation(
        brand_info=brand_info,
        ad_concept=ad_concept,
        force_regenerate=False  # Set to True to regenerate
    )
    
    if "error" not in desc_results:
        print("\n✅ Scene descriptions generated successfully!")
        print(f"Files: {desc_results['files']}")
    
    # ========================================================================
    # DEBUG OPTION 2: Test only scene image generation
    # ========================================================================
    print("\n\n" + "="*100)
    print("DEBUG OPTION 2: SCENE IMAGE GENERATION ONLY")
    print("="*100)
    
    # This will load scene descriptions, characters, and locations from existing files
    # and generate scene images
    
    # Option 2a: Generate all shots
    image_results = pipeline.debug_scene_image_generation(
        brand_info=brand_info,
        product_image_path="path/to/product/image.png",  # Optional
        aspect_ratio="16:9",
        shot_numbers=None,  # None = generate all shots
        force_regenerate=False,
        delay_between_shots=2.0
    )
    
    # # Option 2b: Generate only specific shots (useful for testing)
    # # image_results = pipeline.debug_scene_image_generation(
    # #     brand_info=brand_info,
    # #     product_image_path="path/to/product/image.png",
    # #     aspect_ratio="16:9",
    # #     shot_numbers=[1, 3, 5],  # Only generate shots 1, 3, and 5
    # #     force_regenerate=False,
    # #     delay_between_shots=2.0
    # # )
    
    if "error" not in image_results:
        print("\n✅ Scene images generated successfully!")
        print(f"Generated: {len(image_results['generation_progress']['generated_images'])} images")
        print(f"Failed: {len(image_results['generation_progress']['failed_generations'])} images")

✅ Initialized project: deconstruct_dhoni_001

📋 Checking existing stages...
  shot_script: ✅ COMPLETED
  shot_script_complete: ✅ COMPLETED
  characters: ✅ COMPLETED
  locations: ✅ COMPLETED
  scene_descriptions: ✅ COMPLETED
  generation_report: ✅ COMPLETED


DEBUG OPTION 1: SCENE DESCRIPTION GENERATION ONLY

🐛 DEBUG: SCENE DESCRIPTION GENERATION ONLY


📂 LOADING EXISTING DATA

📄 Loading shot script from: projects_data/deconstruct_dhoni_001/scripts/deconstruct_dhoni_001_shot_script_complete.json
  ✅ Loaded 9 shots
📄 Loading characters from: projects_data/deconstruct_dhoni_001/scripts/deconstruct_dhoni_001_characters.json
  ✅ Loaded 1 characters
📄 Loading locations from: projects_data/deconstruct_dhoni_001/scripts/deconstruct_dhoni_001_locations.json
  ✅ Loaded 2 locations
📄 Loading scene descriptions from: projects_data/deconstruct_dhoni_001/prompts/deconstruct_dhoni_001_scene_descriptions.json
  ✅ Loaded 9 scene descriptions

📂 DATA LOADING COMPLETE

✅ Loaded data:
   - Shots: 9
   - C

KeyboardInterrupt: 

# Video Describer

In [7]:
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
import json
import os
from dotenv import load_dotenv
from utils.llm import get_llm_model


load_dotenv()

llm_client = get_llm_model("gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))


class StandardVideoPrompt(BaseModel):
    """Standard video prompt for regular shots"""
    shot_no: int = Field(description="Shot number")
    camera_angle: str = Field(description="Detailed camera angle, movement, and lens specifications")
    scene_description: str = Field(description="Detailed description of action and how it should be performed")
    lighting: str = Field(description="Detailed lighting setup, color temperature, and mood")
    dialogue: Optional[str] = Field(default="", description="Exact dialogue if present")
    voice_over: Optional[str] = Field(default="", description="Voice over narration with voice type")
    additional_notes: str = Field(description="Music, sound design, audio SFX, animation notes, key focus")


class AnimatedProductShowcasePrompt(BaseModel):
    """Animated video prompt for product showcase with first and last frame"""
    shot_no: int = Field(description="Shot number")
    camera_angle: str = Field(description="Dynamic camera angle with movement description")
    scene_description: str = Field(description="Detailed animation from first frame to last frame transformation")
    lighting: str = Field(description="Dynamic lighting throughout animation")
    dialogue: Optional[str] = Field(default="", description="Exact dialogue if present")
    voice_over: Optional[str] = Field(default="", description="Voice over narration with voice type")
    additional_notes: str = Field(description="Animation details, text animations, effects, and transitions")
    requires_two_frames: bool = Field(default=True, description="Indicates this needs first and last frame images")


class VideoPrompt(BaseModel):
    """Container for either standard or animated video prompt"""
    shot_no: int
    prompt_type: str = Field(description="Either 'standard' or 'animated_showcase'")
    standard_prompt: Optional[StandardVideoPrompt] = None
    animated_prompt: Optional[AnimatedProductShowcasePrompt] = None


class VideoDescription(BaseModel):
    """Complete video description for all shots"""
    ad_title: str
    total_shots: int
    video_prompts: List[VideoPrompt]


class VideoDescriptionGenerator:
    def __init__(self):
        self.llm = llm_client
    
    def create_video_prompt_system_prompt(self) -> str:
        """Create system prompt for generating video prompts"""
        return """You are an expert video director and cinematographer specializing in commercial video production for AI video generation models like Google Veo 3.

Your task is to create HIGHLY DETAILED video generation prompts that can be used with image-to-video AI models.

There are TWO types of video prompts you need to generate:

═══════════════════════════════════════════════════════════════════════════════
TYPE 1: STANDARD VIDEO PROMPT (for regular shots)
═══════════════════════════════════════════════════════════════════════════════

Structure:
{
  "camera_angle": "Detailed camera specifications with movement",
  "scene_description": "Detailed action and performance description",
  "lighting": "Complete lighting setup with technical details",
  "dialogue": "Exact dialogue if present (empty string if none)",
  "voice_over": "Voice over with voice type specification (empty string if none)",
  "additional_notes": "Music, sound design, audio SFX, key focus"
}

Guidelines for Standard Prompts:
- Camera Angle: Specify shot type (wide/medium/close-up), camera movement (static/pan/tilt/dolly/tracking), lens (24mm/35mm/50mm/85mm), and any special techniques
- Scene Description: Describe character actions, movements, expressions, interactions in detail. Be specific about timing and pacing
- Lighting: Include light direction, intensity, color temperature (in Kelvin), shadow details, fill light, practical lights, mood
- Dialogue: Include exact spoken words with quotation marks. Leave empty if no dialogue
- Voice Over: Include VO text with voice type (e.g., "in Indian male voice", "in Indian female voice", "in calm narrator voice"). Leave empty if no VO
- Additional Notes: Background sounds, music type/mood, SFX (footsteps, door sounds, etc.), animation requirements, visual effects, key focus of the shot

Example Standard Prompt:
{
  "camera_angle": "Wide shot transitioning smoothly to medium close-up on the man, using a 50mm lens for shallow depth and background compression",
  "scene_description": "He walks confidently towards the pitch under bright, harsh sunlight. He squints briefly at the sun, smiles slightly, adjusts his cap, and says his line with calm confidence.",
  "lighting": "Direct bright sunlight creating strong highlights and subtle shadows on Dhoni's face; slight fill from reflector to maintain detail without flattening contrast; color temperature ~5600K to match daylight.",
  "dialogue": "I am protected.",
  "voice_over": "Champions prepare for everything. Even the sun. (in Indian female voice)",
  "additional_notes": "Stadium ambience with crowd murmurs and distant anthem; natural outdoor acoustics; emphasize confident walk and subtle smile; background should have soft stadium blur"
}

═══════════════════════════════════════════════════════════════════════════════
TYPE 2: ANIMATED PRODUCT SHOWCASE PROMPT (for final product shot)
═══════════════════════════════════════════════════════════════════════════════

This type is ONLY used when:
- It's the FINAL shot of the ad
- It's a product showcase shot
- Animation is requested
- Two frames will be provided (first frame and last frame)

Structure:
{
  "camera_angle": "Dynamic camera movement from start to end position",
  "scene_description": "Detailed animation transformation from first frame to last frame",
  "lighting": "Dynamic lighting changes throughout animation",
  "dialogue": "Exact dialogue if present (empty string if none)",
  "voice_over": "Voice over with voice type (empty string if none)",
  "additional_notes": "Detailed animation breakdown, text animations frame by frame, effects",
  "requires_two_frames": true
}

Guidelines for Animated Showcase Prompts:
- Camera Angle: Describe camera movement path from start to end (e.g., "low-to-mid angle starting from ground level moving upward")
- Scene Description: Describe the complete transformation - what's visible at the start, how it transitions, what's visible at the end. Be specific about animation flow
- Lighting: Describe how lighting evolves during the animation, including any dynamic effects like lens flares, highlights, shadows
- Additional Notes: CRITICAL - Must include:
  * Animation transition details (speed, easing, motion blur)
  * Text overlay animations IN DETAIL (each text element individually with animation type, timing, position)
  * Visual effects (lens flares, glows, reflections, particles)
  * Product highlighting techniques
  * Background animation if any

Example Animated Showcase Prompt:
{
  "camera_angle": "Dynamic low-to-mid angle starting from ground level moving upward and forward toward the product, using a 35mm lens for smooth perspective and natural depth",
  "scene_description": "Animation begins from the ground with a blurred stadium floor and a faint Deconstruct logo on the surface. The camera smoothly rises and transitions to reveal the Deconstruct Gel Sunscreen tube prominently on the center-right. The modern IPL stadium is bright with warm sunlight streaming from the top-left, creating subtle lens flares and golden highlights. Blurred cheering crowd and stadium details provide energetic context. Reflections under the product add realism, and the product remains the visual centerpiece throughout.",
  "lighting": "Dynamic sunlight from top-left creating warm highlights and lens flares; soft rim lighting on product edges; subtle shadows under the product for grounding; color temperature ~5600K for natural sunlight; gradient highlights in yellow, orange, and soft white building in intensity as camera rises.",
  "dialogue": "",
  "voice_over": "Stay Protected with Deconstruct Gel Sunscreen. (in Indian male voice)",
  "additional_notes": "Transition animation from ground to product showcase over 3 seconds with golden highlights and subtle motion blur; animate each text overlay individually: 
1. 'IPL Season Begins.' – bold energetic yellow font, slightly italicized, appearing left of bottle with slide-in from left + fade effect (0.5s delay from start)
2. 'Stay Protected.' – bold black modern sans serif with soft shadow, appearing right of bottle with upward fade-in (1.0s delay)
3. 'Deconstruct' – black thin serif (Didot/Bodoni), spaced wide, animating from top downward to slightly above product (1.5s delay)
4. 'Gel Sunscreen' – modern sans serif light gray with subtle underline, sliding up from below brand name (2.0s delay)
5. 'SPF 55+ | PA+++' – monospaced metallic silver, bottom-right alignment, fading in with gentle glow (2.5s delay)
Include subtle product reflections and golden rim lighting throughout; background maintains modern IPL editorial vibe with slight atmospheric blur; add subtle sparkle effects on product surface"
}

═══════════════════════════════════════════════════════════════════════════════
IMPORTANT RULES:
═══════════════════════════════════════════════════════════════════════════════

1. Choose the correct prompt type:
   - Use STANDARD for all regular shots (99% of shots)
   - Use ANIMATED SHOWCASE only for final product showcase shots when animation is requested

2. Do not include celebrity name this is the main requirement and write a safe prompt avoid unsafe words

2. Be extremely specific and detailed in all descriptions
3. Include technical specifications (focal length, color temperature, etc.)
4. Describe timing and pacing clearly
5. For animated prompts, break down the animation step-by-step
6. Always specify voice type for voice overs (Indian male/female/neutral)
7. Leave dialogue and voice_over as empty strings if not present (not "None" or null)
8. Additional notes should be comprehensive - include ALL audio, visual effects, and timing details

Your prompts will be used directly by AI video generation models, so clarity and detail are critical.
"""

    def generate_video_prompt_for_shot(
        self,
        shot: Dict[str, Any],
        is_final_shot: bool = False,
        enable_animation: bool = False
    ) -> VideoPrompt:
        """
        Generate video prompt for a single shot
        
        Args:
            shot: Shot dictionary with all shot information
            is_final_shot: Whether this is the final shot
            enable_animation: Whether to generate animated showcase (only for final product shots)
            
        Returns:
            VideoPrompt object
        """
        shot_no = shot.get('shot_no', 0)
        
        # Determine prompt type
        is_product_showcase = (
            is_final_shot and 
            shot.get('product_image_required', False) and 
            shot.get('text_overlay') and 
            shot.get('text_overlay') != "None"
        )
        
        prompt_type = "animated_showcase" if (is_product_showcase and enable_animation) else "standard"
        
        system_prompt = self.create_video_prompt_system_prompt()
        
        # Extract shot information
        shot_info = f"""
Shot Number: {shot.get('shot_no', 'N/A')}
Duration: {shot.get('duration', 'N/A')}
Time Stamp: {shot.get('time_stamp', 'N/A')}
Location: {shot.get('location', 'N/A')}
Camera Angle: {shot.get('camera_angle', 'N/A')}
Visual Description: {shot.get('visual_description', 'N/A')}
Action: {shot.get('action', 'N/A')}
Objects/Props Involved: {shot.get('objects_props_involved', 'N/A')}
Audio/SFX: {shot.get('audio_sfx', 'N/A')}
Dialogue: {shot.get('dialogue', 'None')}
Voice Over: {shot.get('voice_over', 'None')}
Text Overlay: {shot.get('text_overlay', 'None')}
Key Focus: {shot.get('key_focus', 'N/A')}
Product Image Required: {shot.get('product_image_required', False)}
Characters Involved: {', '.join(shot.get('characters_involved', []))}
"""

        if prompt_type == "standard":
            user_prompt = f"""Generate a STANDARD video prompt for this shot:

{shot_info}

Create a detailed standard video prompt that includes:
1. Camera angle with technical specifications
2. Detailed scene description with actions and performances
3. Complete lighting setup
4. Exact dialogue (empty string if none)
5. Voice over with voice type (empty string if none)
6. Comprehensive additional notes with audio, SFX, and key focus

Return in StandardVideoPrompt format."""

            try:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
                
                
                structured_llm = self.llm.with_structured_output(StandardVideoPrompt)
                standard_prompt = structured_llm.invoke(messages)
                
                return VideoPrompt(
                    shot_no=shot_no,
                    prompt_type="standard",
                    standard_prompt=standard_prompt,
                    animated_prompt=None
                )
                
            except Exception as e:
                print(f"Error generating standard video prompt for shot {shot_no}: {e}")
                raise
        
        else:  # animated_showcase
            user_prompt = f"""Generate an ANIMATED PRODUCT SHOWCASE video prompt for this FINAL shot:

{shot_info}

This is a product showcase shot that requires animation between two frames (first frame and last frame).

Create a detailed animated showcase prompt that includes:
1. Dynamic camera movement from start to end
2. Complete animation transformation description (what happens from first frame to last frame)
3. Dynamic lighting evolution during animation
4. Exact dialogue (empty string if none)
5. Voice over with voice type (empty string if none)
6. DETAILED additional notes including:
   - Animation transition details (timing, easing, effects)
   - Individual text overlay animations (each text element with specific animation type, timing, position)
   - Visual effects (lens flares, glows, reflections, highlights)
   - Product highlighting techniques
   - Background animation details

CRITICAL: Text overlay animations must be broken down individually for each text element from the text_overlay field.

Return in AnimatedProductShowcasePrompt format."""

            try:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
                
                structured_llm= self.llm.with_structured_output(AnimatedProductShowcasePrompt)
                animated_prompt = structured_llm.invoke(messages)
                
                return VideoPrompt(
                    shot_no=shot_no,
                    prompt_type="animated_showcase",
                    standard_prompt=None,
                    animated_prompt=animated_prompt
                )
                
            except Exception as e:
                print(f"Error generating animated video prompt for shot {shot_no}: {e}")
                raise
    
    def generate_video_descriptions(
        self,
        shots: List[Dict[str, Any]],
        ad_title: str,
        enable_animation_for_finale: bool = True
    ) -> VideoDescription:
        """
        Generate video prompts for all shots
        
        Args:
            shots: List of shot dictionaries
            ad_title: Title of the ad
            enable_animation_for_finale: Whether to enable animation for final product showcase
            
        Returns:
            VideoDescription with all video prompts
        """
        print("\n" + "="*100)
        print(f"GENERATING VIDEO DESCRIPTIONS - {ad_title}")
        print("="*100 + "\n")
        
        video_prompts = []
        total_shots = len(shots)
        
        for idx, shot in enumerate(shots, 1):
            shot_no = shot.get('shot_no', idx)
            is_final = (idx == total_shots)
            
            print(f"\n[Shot {shot_no}/{total_shots}] Generating video prompt...")
            
            video_prompt = self.generate_video_prompt_for_shot(
                shot=shot,
                is_final_shot=is_final,
                enable_animation=enable_animation_for_finale
            )
            
            video_prompts.append(video_prompt)
            
            prompt_type_label = "ANIMATED SHOWCASE" if video_prompt.prompt_type == "animated_showcase" else "STANDARD"
            print(f"✓ Generated {prompt_type_label} video prompt for shot {shot_no}")
        
        video_description = VideoDescription(
            ad_title=ad_title,
            total_shots=total_shots,
            video_prompts=video_prompts
        )
        
        print("\n" + "="*100)
        print(f"✓ Video descriptions complete for all {total_shots} shots!")
        print("="*100 + "\n")
        
        return video_description
    
    def save_video_descriptions(
        self,
        video_description: VideoDescription,
        output_file: str,
        output_dir: str = "projects_data"
    ):
        """Save video descriptions to JSON file"""
        video_dict = video_description.model_dump()
        
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(video_dict, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Video descriptions saved to {file_path}")
        return file_path
    
    def export_video_prompts_readable(
        self,
        video_description: VideoDescription,
        output_file: str,
        output_dir: str = "projects_data"
    ):
        """Export video prompts in human-readable format"""
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(f"VIDEO PROMPTS FOR: {video_description.ad_title}\n")
            f.write("="*100 + "\n\n")
            
            for video_prompt in video_description.video_prompts:
                f.write(f"{'═'*100}\n")
                f.write(f"SHOT {video_prompt.shot_no} - {video_prompt.prompt_type.upper()}\n")
                f.write(f"{'═'*100}\n\n")
                
                if video_prompt.prompt_type == "standard" and video_prompt.standard_prompt:
                    prompt = video_prompt.standard_prompt
                    f.write(f"Camera Angle:\n{prompt.camera_angle}\n\n")
                    f.write(f"Scene Description:\n{prompt.scene_description}\n\n")
                    f.write(f"Lighting:\n{prompt.lighting}\n\n")
                    
                    if prompt.dialogue:
                        f.write(f"Dialogue:\n\"{prompt.dialogue}\"\n\n")
                    
                    if prompt.voice_over:
                        f.write(f"Voice Over:\n{prompt.voice_over}\n\n")
                    
                    f.write(f"Additional Notes:\n{prompt.additional_notes}\n\n")
                
                elif video_prompt.prompt_type == "animated_showcase" and video_prompt.animated_prompt:
                    prompt = video_prompt.animated_prompt
                    f.write(f"⚠️ REQUIRES TWO FRAMES: First Frame + Last Frame\n\n")
                    f.write(f"Camera Angle (Dynamic):\n{prompt.camera_angle}\n\n")
                    f.write(f"Scene Description (Animation):\n{prompt.scene_description}\n\n")
                    f.write(f"Lighting (Dynamic):\n{prompt.lighting}\n\n")
                    
                    if prompt.dialogue:
                        f.write(f"Dialogue:\n\"{prompt.dialogue}\"\n\n")
                    
                    if prompt.voice_over:
                        f.write(f"Voice Over:\n{prompt.voice_over}\n\n")
                    
                    f.write(f"Additional Notes (Animation Details):\n{prompt.additional_notes}\n\n")
                
                f.write("\n")
        
        print(f"✓ Readable video prompts exported to {file_path}")
        return file_path
    
    def display_video_descriptions_summary(self, video_description: VideoDescription):
        """Display summary of video descriptions"""
        print("\n" + "="*100)
        print(f"VIDEO DESCRIPTIONS SUMMARY - {video_description.ad_title}")
        print("="*100 + "\n")
        
        print(f"Total Shots: {video_description.total_shots}\n")
        
        standard_count = sum(1 for vp in video_description.video_prompts if vp.prompt_type == "standard")
        animated_count = sum(1 for vp in video_description.video_prompts if vp.prompt_type == "animated_showcase")
        
        print(f"Standard Prompts: {standard_count}")
        print(f"Animated Showcase Prompts: {animated_count}\n")
        
        for video_prompt in video_description.video_prompts:
            print(f"\n{'─'*100}")
            print(f"SHOT {video_prompt.shot_no} - {video_prompt.prompt_type.upper()}")
            print(f"{'─'*100}")
            
            if video_prompt.prompt_type == "standard" and video_prompt.standard_prompt:
                prompt = video_prompt.standard_prompt
                print(f"Camera: {prompt.camera_angle[:80]}...")
                print(f"Scene: {prompt.scene_description[:80]}...")
                if prompt.dialogue:
                    print(f"Dialogue: \"{prompt.dialogue}\"")
                if prompt.voice_over:
                    print(f"VO: {prompt.voice_over[:60]}...")
            
            elif video_prompt.prompt_type == "animated_showcase" and video_prompt.animated_prompt:
                prompt = video_prompt.animated_prompt
                print(f"⚠️ ANIMATED - Requires 2 frames")
                print(f"Camera: {prompt.camera_angle[:80]}...")
                print(f"Animation: {prompt.scene_description[:80]}...")
                if prompt.voice_over:
                    print(f"VO: {prompt.voice_over[:60]}...")
        
        print("\n" + "="*100 + "\n")
    
    def load_video_descriptions(self, file_path: str) -> VideoDescription:
        """Load video descriptions from JSON file"""
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        return VideoDescription(**data)


# # ============================================================================
# # STANDALONE TESTING
# # ============================================================================

# if __name__ == "__main__":
#     print("\n" + "="*100)
#     print("VIDEO DESCRIPTION GENERATOR - STANDALONE TEST")
#     print("="*100)
    
#     # Example shots data (you can load from your shot script JSON)
#     example_shots = [
#         {
#             "shot_no": 1,
#             "duration": "3 seconds",
#             "time_stamp": "00:00-00:03",
#             "location": "Hotel room, morning light",
#             "location_name": "hotel room",
#             "camera_angle": "Medium shot",
#             "visual_description": "Dhoni sits on bed, checking his cricket bat carefully",
#             "action": "Dhoni examines bat, running fingers along the edge, checking balance",
#             "objects_props_involved": "Cricket bat (worn leather grip, slight scratches), cricket kit bag (CSK blue and yellow), water bottle on nightstand",
#             "audio_sfx": "Soft morning ambience, distant birds chirping",
#             "dialogue": "None",
#             "voice_over": "None",
#             "text_overlay": "None",
#             "key_focus": "Establish Dhoni's meticulous preparation routine",
#             "product_image_required": False,
#             "characters_involved": ["dhoni"],
#             "outfit_character_mapping": []
#         },
        
#         {
#             "shot_no": 8,
#             "duration": "5 seconds",
#             "time_stamp": "00:25-00:30",
#             "location": "Modern studio background with IPL stadium blur",
#             "location_name": "stadium",
#             "camera_angle": "Center product shot",
#             "visual_description": "Deconstruct Gel Sunscreen showcased with text overlays and IPL branding",
#             "action": "Static product display with dynamic lighting and text animations",
#             "objects_props_involved": "Deconstruct Gel Sunscreen tube prominently displayed",
#             "audio_sfx": "Upbeat music crescendo, stadium crowd cheers fading in",
#             "dialogue": "None",
#             "voice_over": "Even the sun.",
#             "text_overlay": "IPL Season Begins. Stay Protected. | Deconstruct | Gel Sunscreen | SPF 55+ | PA+++",
#             "key_focus": "Final product showcase with brand message",
#             "product_image_required": True,
#             "characters_involved": [],
#             "outfit_character_mapping": []
#         }
#     ]
    
#     # Initialize generator
#     generator = VideoDescriptionGenerator()
    
#     # Test 1: Generate video descriptions with animation enabled
#     print("\n" + "─"*100)
#     print("TEST 1: Generating video descriptions with animation for finale")
#     print("─"*100)
    
#     video_descriptions = generator.generate_video_descriptions(
#         shots=example_shots,
#         ad_title="The Captain's Pre-Match Ritual",
#         enable_animation_for_finale=True
#     )
    
#     # Display summary
#     generator.display_video_descriptions_summary(video_descriptions)
    
#     # Save to files
#     json_path = generator.save_video_descriptions(
#         video_descriptions,
#         "video_descriptions_test.json",
#         "test_output"
#     )
    
#     readable_path = generator.export_video_prompts_readable(
#         video_descriptions,
#         "video_prompts_readable_test.txt",
#         "test_output"
#     )
    
#     print("\n" + "─"*100)
#     print("TEST 2: Generating video descriptions WITHOUT animation")
#     print("─"*100)
    
#     video_descriptions_no_anim = generator.generate_video_descriptions(
#         shots=example_shots,
#         ad_title="The Captain's Pre-Match Ritual",
#         enable_animation_for_finale=False
#     )
    
#     generator.display_video_descriptions_summary(video_descriptions_no_anim)
    
#     # Save without animation
#     generator.save_video_descriptions(
#         video_descriptions_no_anim,
#         "video_descriptions_no_animation_test.json",
#         "test_output"
#     )
    
#     generator.export_video_prompts_readable(
#         video_descriptions_no_anim,
#         "video_prompts_no_animation_readable_test.txt",
#         "test_output"
#     )
    
#     print("\n" + "="*100)
#     print("✅ VIDEO DESCRIPTION GENERATOR TEST COMPLETE!")
#     print("="*100)
#     print(f"\nGenerated files:")
#     print(f"  - {json_path}")
#     print(f"  - {readable_path}")
#     print(f"\nCheck the 'test_output' directory for results.")

# Video Generator Class

In [8]:
import time
import os
from typing import List, Optional, Dict, Any
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import json
from dotenv import load_dotenv


load_dotenv()


class VideoGenerationResult(BaseModel):
    """Result of video generation for a single shot"""
    shot_no: int
    video_path: str
    prompt_type: str  # "standard" or "animated_showcase"
    duration_seconds: int
    status: str  # "success" or "failed"
    error_message: Optional[str] = None


class VideoGenerationProgress(BaseModel):
    """Progress tracking for video generation"""
    ad_title: str
    total_shots: int
    generated_videos: List[VideoGenerationResult] = Field(default_factory=list)
    failed_generations: List[Dict[str, Any]] = Field(default_factory=list)
    current_shot: Optional[int] = None


class VideoGenerator:
    """Generate videos using Google Veo 3.1 from scene images and video prompts"""
    
    def __init__(self, output_dir: str = "generated_videos", aspect_ratio: str = "16:9"):
        """
        Initialize Video Generator
        
        Args:
            output_dir: Directory to store generated videos
            aspect_ratio: Video aspect ratio (default: "16:9")
        """
        self.client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.output_dir = output_dir
        self.aspect_ratio = aspect_ratio
        os.makedirs(output_dir, exist_ok=True)
        
        self.progress = VideoGenerationProgress(
            ad_title="",
            total_shots=0,
            generated_videos=[],
            failed_generations=[]
        )
    
    def determine_video_duration(
        self,
        shot: Dict[str, Any],
        video_prompt: Dict[str, Any]
    ) -> int:
        """
        Determine optimal video duration based on shot content
        Available durations: 4, 6, 8 seconds
        
        Args:
            shot: Shot information dictionary
            video_prompt: Video prompt information
            
        Returns:
            Duration in seconds (4, 6, or 8)
        """
        # Get shot duration from shot info
        shot_duration_str = shot.get('duration', '4 seconds')
        
        # Parse duration
        try:
            shot_duration = int(shot_duration_str.split()[0])
        except:
            shot_duration = 4
        
        # Check for dialogue
        has_dialogue = shot.get('dialogue') and shot.get('dialogue') != "None"
        
        # Check for voice over
        has_voice_over = shot.get('voice_over') and shot.get('voice_over') != "None"
        
        # Check for complex action
        action_text = shot.get('action', '')
        is_complex_action = len(action_text) > 100  # Long action description
        
        # Determine duration
        if shot_duration <= 3:
            duration = 4
        elif shot_duration <= 5:
            if has_dialogue or has_voice_over:
                duration = 6
            else:
                duration = 4
        else:  # shot_duration > 5
            if has_dialogue or has_voice_over or is_complex_action:
                duration = 8
            else:
                duration = 6
        
        print(f"  📏 Determined duration: {duration}s (shot duration: {shot_duration_str}, dialogue: {has_dialogue}, VO: {has_voice_over})")
        
        return duration
    
    def generate_standard_video(
        self,
        shot_no: int,
        first_frame_path: str,
        video_prompt: str,
        duration_seconds: int,
        project_id: str = "project"
    ) -> Optional[str]:
        """
        Generate video from first frame image (standard shot)
        
        Args:
            shot_no: Shot number
            first_frame_path: Path to the first frame image (scene image)
            video_prompt: Detailed video generation prompt
            duration_seconds: Duration in seconds (4, 6, or 8)
            project_id: Project identifier
            
        Returns:
            Path to generated video or None if failed
        """
        print(f"\n{'─'*80}")
        print(f"🎬 Generating STANDARD video for Shot {shot_no}")
        print(f"{'─'*80}")
        print(f"  📸 First frame: {first_frame_path}")
        print(f"  ⏱️  Duration: {duration_seconds}s")
        print(f"  📝 Prompt: {video_prompt[:100]}...")
        
        try:
            # Load first frame image
            if not os.path.exists(first_frame_path):
                print(f"  ❌ First frame image not found: {first_frame_path}")
                return None
            
            first_image = types.Image.from_file(location=first_frame_path)
            print(f"  ✅ Loaded first frame image")
            
            # Generate video
            print(f"  🎬 Starting video generation...")
            operation = self.client.models.generate_videos(
                model="veo-3.1-generate-preview",
                prompt=video_prompt,
                image=first_image,
                config=types.GenerateVideosConfig(
                    aspect_ratio=self.aspect_ratio,
                    duration_seconds=duration_seconds
                )
            )
            
            # Poll operation status
            print(f"  ⏳ Polling for completion...")
            poll_count = 0
            while not operation.done:
                poll_count += 1
                print(f"     Polling attempt {poll_count}... (waiting 10s)")
                time.sleep(10)
                operation = self.client.operations.get(operation)
            
            print(f"  ✅ Video generation complete after {poll_count} polls")
            
            # Download generated video
            generated_video = operation.response.generated_videos[0]
            
            # Save video
            filename = f"{project_id}_shot_{shot_no:03d}_video.mp4"
            video_path = os.path.join(self.output_dir, filename)
            
            self.client.files.download(file=generated_video.video)
            generated_video.video.save(video_path)
            
            print(f"  💾 Video saved: {video_path}")
            
            return video_path
            
        except Exception as e:
            print(f"  ❌ Error generating standard video for shot {shot_no}: {e}")
            return None
    
    def generate_animated_showcase_video(
        self,
        shot_no: int,
        first_frame_path: str,
        last_frame_path: str,
        video_prompt: str,
        project_id: str = "project"
    ) -> Optional[str]:
        """
        Generate animated video from first and last frame images (product showcase)
        Duration is fixed at 8 seconds for animated showcase
        
        Args:
            shot_no: Shot number
            first_frame_path: Path to the first frame image
            last_frame_path: Path to the last frame image
            video_prompt: Detailed video generation prompt
            project_id: Project identifier
            
        Returns:
            Path to generated video or None if failed
        """
        print(f"\n{'─'*80}")
        print(f"🎬 Generating ANIMATED SHOWCASE video for Shot {shot_no}")
        print(f"{'─'*80}")
        print(f"  📸 First frame: {first_frame_path}")
        print(f"  📸 Last frame: {last_frame_path}")
        print(f"  ⏱️  Duration: 8s (fixed for animated showcase)")
        print(f"  📝 Prompt: {video_prompt[:100]}...")
        
        try:
            # Load images
            if not os.path.exists(first_frame_path):
                print(f"  ❌ First frame image not found: {first_frame_path}")
                return None
            
            if not os.path.exists(last_frame_path):
                print(f"  ❌ Last frame image not found: {last_frame_path}")
                return None
            
            first_image = types.Image.from_file(location=first_frame_path)
            last_image = types.Image.from_file(location=last_frame_path)
            print(f"  ✅ Loaded both frame images")
            
            # Generate video with first and last frame
            print(f"  🎬 Starting animated video generation...")
            operation = self.client.models.generate_videos(
                model="veo-3.1-generate-preview",
                prompt=video_prompt,
                image=first_image,
                config=types.GenerateVideosConfig(
                    aspect_ratio=self.aspect_ratio,
                    last_frame=last_image
                )
            )
            
            # Poll operation status
            print(f"  ⏳ Polling for completion...")
            poll_count = 0
            while not operation.done:
                poll_count += 1
                print(f"     Polling attempt {poll_count}... (waiting 10s)")
                time.sleep(10)
                operation = self.client.operations.get(operation)
            
            print(f"  ✅ Video generation complete after {poll_count} polls")
            
            # Download generated video
            generated_video = operation.response.generated_videos[0]
            
            # Save video
            filename = f"{project_id}_shot_{shot_no:03d}_video_animated.mp4"
            video_path = os.path.join(self.output_dir, filename)
            
            self.client.files.download(file=generated_video.video)
            generated_video.video.save(video_path)
            
            print(f"  💾 Video saved: {video_path}")
            
            return video_path
            
        except Exception as e:
            print(f"  ❌ Error generating animated showcase video for shot {shot_no}: {e}")
            return None
    
    def generate_all_videos(
        self,
        shots: List[Dict[str, Any]],
        video_prompts: List[Dict[str, Any]],
        scene_images: Dict[int, str],  # Maps shot_no to scene image path
        ad_title: str,
        project_id: str = "project",
        delay_between_videos: float = 5.0
    ) -> VideoGenerationProgress:
        """
        Generate videos for all shots
        
        Args:
            shots: List of shot dictionaries
            video_prompts: List of video prompt dictionaries
            scene_images: Dictionary mapping shot_no to scene image path
            ad_title: Ad title
            project_id: Project identifier
            delay_between_videos: Delay between video generations (seconds)
            
        Returns:
            VideoGenerationProgress with results
        """
        print("\n" + "="*100)
        print(f"🎬 GENERATING VIDEOS - {ad_title}")
        print(f"Aspect Ratio: {self.aspect_ratio}")
        print("="*100 + "\n")
        
        self.progress.ad_title = ad_title
        self.progress.total_shots = len(shots)
        
        # Create shot_no to video_prompt mapping
        prompt_map = {vp['shot_no']: vp for vp in video_prompts}
        
        for idx, shot in enumerate(shots, 1):
            shot_no = shot.get('shot_no', idx)
            self.progress.current_shot = shot_no
            
            print(f"\n{'='*100}")
            print(f"[{idx}/{len(shots)}] Processing Shot {shot_no}")
            print(f"{'='*100}")
            
            # Get scene image
            scene_image_path = scene_images.get(shot_no)
            if not scene_image_path:
                print(f"  ❌ No scene image found for shot {shot_no}")
                self.progress.failed_generations.append({
                    "shot_no": shot_no,
                    "error": "Scene image not found"
                })
                continue
            
            # Get video prompt
            video_prompt_data = prompt_map.get(shot_no)
            if not video_prompt_data:
                print(f"  ❌ No video prompt found for shot {shot_no}")
                self.progress.failed_generations.append({
                    "shot_no": shot_no,
                    "error": "Video prompt not found"
                })
                continue
            
            prompt_type = video_prompt_data.get('prompt_type', 'standard')
            
            video_path = None
            duration = 0
            
            if prompt_type == "standard":
                # Standard video generation
                standard_prompt_data = video_prompt_data.get('standard_prompt')
                if not standard_prompt_data:
                    print(f"  ❌ No standard prompt data for shot {shot_no}")
                    continue
                
                # Build full prompt
                full_prompt = self._build_full_prompt(standard_prompt_data)
                
                # Determine duration
                duration = self.determine_video_duration(shot, video_prompt_data)
                
                # Generate video
                video_path = self.generate_standard_video(
                    shot_no=shot_no,
                    first_frame_path=scene_image_path,
                    video_prompt=full_prompt,
                    duration_seconds=duration,
                    project_id=project_id
                )
                
            elif prompt_type == "animated_showcase":
                # Animated showcase video generation
                animated_prompt_data = video_prompt_data.get('animated_prompt')
                if not animated_prompt_data:
                    print(f"  ❌ No animated prompt data for shot {shot_no}")
                    continue
                
                # Build full prompt
                full_prompt = self._build_full_prompt(animated_prompt_data)
                
                # Get last frame (use previous shot's scene image or same image)
                if idx > 1:
                    previous_shot_no = shots[idx-2].get('shot_no', idx-1)
                    last_frame_path = scene_images.get(previous_shot_no, scene_image_path)
                else:
                    last_frame_path = scene_image_path
                
                duration = 8  # Fixed for animated showcase
                
                # Generate video
                video_path = self.generate_animated_showcase_video(
                    shot_no=shot_no,
                    first_frame_path=last_frame_path,
                    last_frame_path=scene_image_path,
                    video_prompt=full_prompt,
                    project_id=project_id
                )
            
            # Record result
            if video_path:
                result = VideoGenerationResult(
                    shot_no=shot_no,
                    video_path=video_path,
                    prompt_type=prompt_type,
                    duration_seconds=duration,
                    status="success"
                )
                self.progress.generated_videos.append(result)
                print(f"  ✅ Successfully generated video for shot {shot_no}")
            else:
                self.progress.failed_generations.append({
                    "shot_no": shot_no,
                    "error": "Video generation failed"
                })
                print(f"  ❌ Failed to generate video for shot {shot_no}")
            
            # Delay before next video
            if idx < len(shots):
                print(f"\n  ⏳ Waiting {delay_between_videos}s before next video generation...")
                time.sleep(delay_between_videos)
        
        # Print summary
        print("\n" + "="*100)
        print("VIDEO GENERATION COMPLETE")
        print("="*100)
        print(f"✅ Successfully generated: {len(self.progress.generated_videos)}/{self.progress.total_shots}")
        print(f"❌ Failed: {len(self.progress.failed_generations)}")
        
        if self.progress.failed_generations:
            print("\nFailed videos:")
            for failed in self.progress.failed_generations:
                print(f"  - Shot {failed['shot_no']}: {failed['error']}")
        
        return self.progress
    
    def _build_full_prompt(self, prompt_data: Dict[str, Any]) -> str:
        """Build full video prompt from prompt data"""
        parts = []
        
        if prompt_data.get('camera_angle'):
            parts.append(f"Camera Angle: {prompt_data['camera_angle']}")
        
        if prompt_data.get('scene_description'):
            parts.append(f"Scene Description: {prompt_data['scene_description']}")
        
        if prompt_data.get('lighting'):
            parts.append(f"Lighting: {prompt_data['lighting']}")
        
        if prompt_data.get('dialogue'):
            parts.append(f"Dialogue: \"{prompt_data['dialogue']}\"")
        
        if prompt_data.get('voice_over'):
            parts.append(f"Voice Over: {prompt_data['voice_over']}")
        
        if prompt_data.get('additional_notes'):
            parts.append(f"Additional Notes: {prompt_data['additional_notes']}")
        
        return "\n".join(parts)
    
    def save_generation_report(
        self,
        output_file: str,
        output_dir: str = "projects_data"
    ) -> str:
        """Save video generation report to JSON"""
        file_path = os.path.join(output_dir, output_file)
        os.makedirs(output_dir, exist_ok=True)
        
        report_dict = self.progress.model_dump()
        
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(report_dict, f, indent=2, ensure_ascii=False)
        
        print(f"📊 Video generation report saved to {file_path}")
        return file_path


# ============================================================================
# STANDALONE TESTING
# ============================================================================

# if __name__ == "__main__":
#     print("\n" + "="*100)
#     print("VIDEO GENERATOR - STANDALONE TEST")
#     print("="*100)
    
#     # Example: Test with mock data
#     example_shots = [
#         {
#             "shot_no": 1,
#             "duration": "3 seconds",
#             "dialogue": "None",
#             "voice_over": "Champions prepare for everything.",
#             "action": "the man checks his bat carefully"
#         },
#         {
#             "shot_no": 2,
#             "duration": "5 seconds",
#             "dialogue": "I am protected.",
#             "voice_over": "None",
#             "action": "the man walks confidently towards the pitch"
#         }
#     ]
    
#     example_video_prompts = [
#         {
#             "shot_no": 1,
#             "prompt_type": "standard",
#             "standard_prompt": {
#                 "camera_angle": "Medium shot, 50mm lens",
#                 "scene_description": "the man examines his cricket bat carefully",
#                 "lighting": "Soft morning light, 5600K",
#                 "dialogue": "",
#                 "voice_over": "Champions prepare for everything. (in Indian female voice)",
#                 "additional_notes": "Morning ambience, subtle sounds"
#             }
#         },
#         {
#             "shot_no": 2,
#             "prompt_type": "standard",
#             "standard_prompt": {
#                 "camera_angle": "Wide to medium close-up, 50mm lens",
#                 "scene_description": "the man walks confidently towards pitch",
#                 "lighting": "Bright sunlight, 5600K",
#                 "dialogue": "I am protected.",
#                 "voice_over": "",
#                 "additional_notes": "Stadium crowd ambience"
#             }
#         }
#     ]
    
#     # Mock scene images (replace with actual paths)
#     scene_images = {
#         1: "/Users/sanjail/Akaike/Internal_project/ads_poc/projects_data/deconstruct_dhoni_004/scene_images/deconstruct_dhoni_004_shot_002_scene.png",
#         2: "/Users/sanjail/Akaike/Internal_project/ads_poc/projects_data/deconstruct_dhoni_004/scene_images/deconstruct_dhoni_004_shot_007_scene.png"
#     }
    
#     # Initialize generator
#     generator = VideoGenerator(
#         output_dir="test_videos",
#         aspect_ratio="16:9"
#     )
    
#     # Generate videos
#     progress = generator.generate_all_videos(
#         shots=example_shots,
#         video_prompts=example_video_prompts,
#         scene_images=scene_images,
#         ad_title="Test Ad",
#         project_id="test_project",
#         delay_between_videos=5.0
#     )
    
#     # Save report
#     generator.save_generation_report(
#         "video_generation_report_test.json",
#         "test_output"
#     )
    
#     print("\n✅ Video generation test complete!")

# Full Pipeline Testing

In [9]:
import os
import json
from typing import Dict, Any, Optional, List
import time


class CompleteAdProductionPipeline:
    """Complete end-to-end pipeline with outfit generation"""
    
    def __init__(self, project_id: str, base_output_dir: str = "projects_data"):
        self.project_id = project_id
        self.base_output_dir = base_output_dir
        
        # Create directory structure
        self.dirs = {
            "base": base_output_dir,
            "project": os.path.join(base_output_dir, project_id),
            "characters": os.path.join(base_output_dir, project_id, "character_images"),
            "locations": os.path.join(base_output_dir, project_id, "location_images"),
            "outfits": os.path.join(base_output_dir, project_id, "outfit_images"),  # NEW
            "products": os.path.join(base_output_dir, project_id, "product_images"),
            "scenes": os.path.join(base_output_dir, project_id, "scene_images"),
            "scripts": os.path.join(base_output_dir, project_id, "scripts"),
            "prompts": os.path.join(base_output_dir, project_id, "prompts"),
            "videos": os.path.join(base_output_dir, project_id, "generated_videos")
        }
        
        for dir_path in self.dirs.values():
            os.makedirs(dir_path, exist_ok=True)
        
        # Define expected file paths for each stage
        self.stage_files = {
            "shot_script": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script.json"),
            "shot_script_complete": os.path.join(self.dirs["scripts"], f"{project_id}_shot_script_complete.json"),
            "characters": os.path.join(self.dirs["scripts"], f"{project_id}_characters.json"),
            "locations": os.path.join(self.dirs["scripts"], f"{project_id}_locations.json"),
            "outfits": os.path.join(self.dirs["scripts"], f"{project_id}_outfits.json"),  # NEW
            "scene_descriptions": os.path.join(self.dirs["prompts"], f"{project_id}_scene_descriptions.json"),
            "generation_report": os.path.join(self.dirs["scripts"], f"{project_id}_scene_generation_report.json"),
            "video_descriptions": os.path.join(self.dirs["prompts"], f"{project_id}_video_descriptions.json"),  # NEW
            "video_generation_report": os.path.join(self.dirs["scripts"], f"{project_id}_video_generation_report.json"),
        }
        
        print(f"✅ Initialized project: {project_id}")
        self._check_existing_stages()
        
    def _check_existing_stages(self):
        """Check which stages have already been completed"""
        print("\n📋 Checking existing stages...")
        self.completed_stages = {}
        
        for stage, filepath in self.stage_files.items():
            exists = os.path.exists(filepath)
            self.completed_stages[stage] = exists
            status = "✅ COMPLETED" if exists else "❌ PENDING"
            print(f"  {stage}: {status}")
    def _stage_completed(self, stage: str) -> bool:
        """Check if a stage is completed"""
        return self.completed_stages.get(stage, False)
    
    def _mark_stage_completed(self, stage: str):
        """Mark a stage as completed"""
        self.completed_stages[stage] = True
    
    def _load_json(self, filepath: str) -> Optional[Dict]:
        """Load JSON file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"⚠ Error loading {filepath}: {e}")
            return None
    
    def load_existing_data(self) -> Dict[str, Any]:
        """Load all existing data from previous stages"""
        print("\n" + "="*100)
        print("📂 LOADING EXISTING DATA")
        print("="*100 + "\n")
        
        loaded_data = {
            "shot_script": None,
            "characters": None,
            "locations": None,
            "outfits": None,  # NEW
            "scene_descriptions": None
        }
        
        # Load shot script
        shot_script_path = self.stage_files.get("shot_script_complete") or self.stage_files.get("shot_script")
        if os.path.exists(shot_script_path):
            print(f"📄 Loading shot script from: {shot_script_path}")
            shot_script_data = self._load_json(shot_script_path)
            if shot_script_data:
            
                loaded_data["shot_script"] = ShotScript(**shot_script_data)
                print(f"  ✅ Loaded {len(loaded_data['shot_script'].shots)} shots")
        
        # Load characters
        if os.path.exists(self.stage_files["characters"]):
            print(f"📄 Loading characters from: {self.stage_files['characters']}")
            char_data = self._load_json(self.stage_files["characters"])
            if char_data:
                loaded_data["characters"] = char_data.get("characters", [])
                print(f"  ✅ Loaded {len(loaded_data['characters'])} characters")
        
        # Load locations
        if os.path.exists(self.stage_files["locations"]):
            print(f"📄 Loading locations from: {self.stage_files['locations']}")
            loc_data = self._load_json(self.stage_files["locations"])
            if loc_data:
                loaded_data["locations"] = loc_data.get("locations", [])
                print(f"  ✅ Loaded {len(loaded_data['locations'])} locations")
        
        # NEW: Load outfits
        if os.path.exists(self.stage_files["outfits"]):
            print(f"📄 Loading outfits from: {self.stage_files['outfits']}")
            outfit_data = self._load_json(self.stage_files["outfits"])
            if outfit_data:
                loaded_data["outfits"] = outfit_data.get("outfits", [])
                print(f"  ✅ Loaded {len(loaded_data['outfits'])} outfits")
        
        # Load scene descriptions
        if os.path.exists(self.stage_files["scene_descriptions"]):
            print(f"📄 Loading scene descriptions from: {self.stage_files['scene_descriptions']}")
            scene_data = self._load_json(self.stage_files["scene_descriptions"])
            if scene_data:
               
                loaded_data["scene_descriptions"] = SceneDescription(**scene_data)
                print(f"  ✅ Loaded {len(loaded_data['scene_descriptions'].shots)} scene descriptions")
        
        print("\n" + "="*100)
        print("📂 DATA LOADING COMPLETE")
        print("="*100 + "\n")
        
        return loaded_data
    
    def run_complete_pipeline(
        self,
        ad_concept: Dict[str, Any],
        brand_info: Dict[str, Any],
        product_image_path: Optional[str] = None,
        target_duration: str = "30 seconds",
        aspect_ratio: str = "16:9",
        generate_character_images: bool = True,
        generate_location_images: bool = True,
        generate_outfit_images: bool = True,  # NEW
        generate_scene_images: bool = True,
        generate_videos:bool=True,
        force_regenerate: bool = False
    ) -> Dict[str, Any]:
        """Run complete pipeline including outfit generation"""
        
        results = {
            "project_id": self.project_id,
            "directories": self.dirs,
            "files": {},
            "assets": {},
            "stages_executed": [],
            "stages_skipped": []
        }
        
        print("\n" + "="*100)
        print(f"🎬 STARTING COMPLETE AD PRODUCTION PIPELINE")
        print(f"Project: {self.project_id}")
        print(f"Force Regenerate: {force_regenerate}")
        print("="*100 + "\n")
        
        
        # ====================================================================
        # STEP 1: Generate Shot Script
        # ====================================================================
        print("\n" + "─"*100)
        print("STEP 1: SHOT SCRIPT GENERATION")
        print("─"*100)
        
        shot_script = None
        
        if force_regenerate or not self._stage_completed("shot_script"):
            print("🔄 Generating shot script...")
            
            
            
            shot_generator = ShotScriptGenerator()
            shot_script = shot_generator.generate_shot_script(
                ad_concept=ad_concept,
                brand_info=brand_info
            )
            
            # Save shot script
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script.json",
                self.dirs["scripts"]
            )
            
            
            results["files"]["shot_script_json"] = shot_script_json
          
            
            self._mark_stage_completed("shot_script")
            results["stages_executed"].append("shot_script_generation")
        else:
            print("✅ Shot script already exists, loading from file...")
            shot_script_data = self._load_json(self.stage_files["shot_script"])
            
            if shot_script_data:
            
                shot_script = ShotScript(**shot_script_data)
                results["files"]["shot_script_json"] = self.stage_files["shot_script"]
                results["stages_skipped"].append("shot_script_generation")
            else:
                print("⚠ Failed to load existing shot script, regenerating...")
                force_regenerate = True  # Force regeneration of subsequent stages
                return self.run_complete_pipeline(
                    ad_concept, brand_info, product_image_path, target_duration,
                    aspect_ratio, generate_character_images, generate_location_images,
                    generate_scene_images, force_regenerate=True
                )
        
        results["assets"]["shot_script"] = shot_script
        
        # ====================================================================
        # STEP 2: Generate Character Images
        # ====================================================================
        characters_with_images = None
        
        if generate_character_images and shot_script.characters_info:
            print("\n" + "─"*100)
            print("STEP 2: CHARACTER IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("characters"):
                print(f"🔄 Generating images for {len(shot_script.characters_info)} characters...")
                
                
                
                # Convert to FullCharacter format
                full_characters = []
                for idx, char in enumerate(shot_script.characters_info, 1):
                    full_char = FullCharacter(
                        name=char.name,
                        age=char.age,
                        role=char.role,
                        gender=char.gender,
                        overall_description=char.overall_description
                    )
                    full_characters.append(full_char)
                
                # Generate images
                char_generator = CharacterGenerator(output_dir=self.dirs["characters"])
                characters_with_images = char_generator.generate_images_for_all_characters(full_characters)
                
                # Save character info
                char_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_characters.json")
                char_generator.save_characters_with_images(characters_with_images, char_json_path)
                
                results["files"]["characters_json"] = char_json_path
                
                self._mark_stage_completed("characters")
                results["stages_executed"].append("character_image_generation")
            else:
                print("✅ Character images already exist, loading from file...")
                char_data = self._load_json(self.stage_files["characters"])
                
                if char_data:
                    
                    characters_with_images = [
                        FullCharacter(**char) for char in char_data.get("characters", [])
                    ]
                    results["files"]["characters_json"] = self.stage_files["characters"]
                    results["stages_skipped"].append("character_image_generation")
                else:
                    print("⚠ Failed to load existing character data")
            
            if characters_with_images:
                results["assets"]["characters"] = [char.model_dump() for char in characters_with_images]
                
                # Update shot script with image paths AND reference descriptions
                char_image_map = {char.name.lower(): char.image_path for char in characters_with_images}
                char_ref_desc_map = {char.name.lower(): char.reference_description for char in characters_with_images}  # NEW
                
                for char_info in shot_script.characters_info:
                    char_name_lower = char_info.name.lower()
                    if char_name_lower in char_image_map:
                        char_info.image_path = char_image_map[char_name_lower]
                    if char_name_lower in char_ref_desc_map:  # NEW
                        char_info.reference_description = char_ref_desc_map[char_name_lower]
        
        # ====================================================================
        # STEP 3: Generate Location Images
        # ====================================================================
        locations_with_images = None
        
        if generate_location_images and shot_script.location_info:
            print("\n" + "─"*100)
            print("STEP 3: LOCATION IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("locations"):
                print(f"🔄 Generating images for {len(shot_script.location_info)} locations...")
                
              
                
                # Convert to FullLocation format
                full_locations = []
                for loc in shot_script.location_info:
                    full_loc = FullLocation(
                        name=loc.name,
                        overall_description=loc.overall_description
                    )
                    full_locations.append(full_loc)
                
                # Generate images
                loc_generator = LocationGenerator(output_dir=self.dirs["locations"])
                locations_with_images = loc_generator.generate_images_for_all_locations(full_locations)
                
                # Save location info
                loc_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_locations.json")
                loc_generator.save_locations_with_images(locations_with_images, loc_json_path)
                
                results["files"]["locations_json"] = loc_json_path
                
                self._mark_stage_completed("locations")
                results["stages_executed"].append("location_image_generation")
            else:
                print("✅ Location images already exist, loading from file...")
                loc_data = self._load_json(self.stage_files["locations"])
                
                if loc_data:
               
                    locations_with_images = [
                        FullLocation(**loc) for loc in loc_data.get("locations", [])
                    ]
                    results["files"]["locations_json"] = self.stage_files["locations"]
                    results["stages_skipped"].append("location_image_generation")
                else:
                    print("⚠ Failed to load existing location data")
            
            if locations_with_images:
                results["assets"]["locations"] = [loc.model_dump() for loc in locations_with_images]
                
                # Update shot script with image paths
                loc_image_map = {loc.name.lower(): loc.image_path for loc in locations_with_images}
                for loc_info in shot_script.location_info:
                    if loc_info.name.lower() in loc_image_map:
                        loc_info.image_path = loc_image_map[loc_info.name.lower()]
        
        # Save updated shot script with all image paths
        if characters_with_images or locations_with_images:
       
            shot_generator = ShotScriptGenerator()
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script_complete.json",
                self.dirs["scripts"]
            )
            results["files"]["shot_script_complete"] = shot_script_json
            self._mark_stage_completed("shot_script_complete")

          # ====================================================================
        # STEP 4: Generate Outfit Images (NEW)
        # ====================================================================
        outfits_with_images = None
        
        if generate_outfit_images and shot_script.character_outfit_info:
            print("\n" + "─"*100)
            print("STEP 4: OUTFIT IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("outfits"):
                print(f"🔄 Generating images for {len(shot_script.character_outfit_info)} outfits...")
                
                
                
                # Convert to FullOutfit format
                full_outfits = []
                for outfit_info in shot_script.character_outfit_info:
                    full_outfit = FullOutfit(
                        outfit=outfit_info.outfit,
                        outfit_description=outfit_info.outfit_description
                    )
                    full_outfits.append(full_outfit)
                
                # Generate images
                outfit_generator = OutfitGenerator(output_dir=self.dirs["outfits"])
                outfits_with_images = outfit_generator.generate_images_for_all_outfits(full_outfits)
                
                # Save outfit info
                outfit_json_path = os.path.join(self.dirs["scripts"], f"{self.project_id}_outfits.json")
                outfit_generator.save_outfits_with_images(outfits_with_images, outfit_json_path)
                
                results["files"]["outfits_json"] = outfit_json_path
                
                self._mark_stage_completed("outfits")
                results["stages_executed"].append("outfit_image_generation")
            else:
                print("✅ Outfit images already exist, loading from file...")
                outfit_data = self._load_json(self.stage_files["outfits"])
                
                if outfit_data:
                    
                    outfits_with_images = [
                        FullOutfit(**outfit) for outfit in outfit_data.get("outfits", [])
                    ]
                    results["files"]["outfits_json"] = self.stage_files["outfits"]
                    results["stages_skipped"].append("outfit_image_generation")
                else:
                    print("⚠ Failed to load existing outfit data")
            
            if outfits_with_images:
                results["assets"]["outfits"] = [outfit.model_dump() for outfit in outfits_with_images]
                
                # Update shot script with image paths
                outfit_image_map = {outfit.outfit.lower(): outfit.image_path for outfit in outfits_with_images}
                for outfit_info in shot_script.character_outfit_info:
                    if outfit_info.outfit.lower() in outfit_image_map:
                        outfit_info.image_path = outfit_image_map[outfit_info.outfit.lower()]
        
        # Save updated shot script
        if characters_with_images or locations_with_images or outfits_with_images:
            
            shot_generator = ShotScriptGenerator()
            shot_script_json = shot_generator.save_shot_script_json(
                shot_script,
                f"{self.project_id}_shot_script_complete.json",
                self.dirs["scripts"]
            )
            results["files"]["shot_script_complete"] = shot_script_json
            self._mark_stage_completed("shot_script_complete")
        
        # ====================================================================
        # STEP 5: Generate Scene Descriptions (Updated with outfits)
        # ====================================================================
        print("\n" + "─"*100)
        print("STEP 5: SCENE DESCRIPTION GENERATION")
        print("─"*100)
        
        scene_descriptions = None
        
        if force_regenerate or not self._stage_completed("scene_descriptions"):
            print("🔄 Generating scene descriptions and image prompts...")
            
           
            
            scene_desc_generator = SceneDescriptionGenerator()
            scene_descriptions = scene_desc_generator.generate_scene_descriptions(
                shots=shot_script.shots,
                brand_info=brand_info,
                ad_title=shot_script.ad_title,
                characters_info=[char.model_dump() for char in shot_script.characters_info],
                locations_info=[loc.model_dump() for loc in shot_script.location_info],
                outfits_info=results["assets"].get("outfits", []),  # NEW: Pass outfits
                ad_concept=ad_concept
            )
            
            # Save scene descriptions
            scene_desc_json = scene_desc_generator.save_scene_descriptions(
                scene_descriptions,
                f"{self.project_id}_scene_descriptions.json",
                self.dirs["prompts"]
            )
            scene_prompts_txt = scene_desc_generator.export_prompts_only(
                scene_descriptions,
                f"{self.project_id}_image_prompts.txt",
                self.dirs["prompts"]
            )
            
            results["files"]["scene_descriptions"] = scene_desc_json
            results["files"]["image_prompts"] = scene_prompts_txt
            
            self._mark_stage_completed("scene_descriptions")
            results["stages_executed"].append("scene_description_generation")
        else:
            print("✅ Scene descriptions already exist, loading from file...")
            scene_desc_data = self._load_json(self.stage_files["scene_descriptions"])
            
            if scene_desc_data:
        
                scene_descriptions = SceneDescription(**scene_desc_data)
                results["files"]["scene_descriptions"] = self.stage_files["scene_descriptions"]
                results["stages_skipped"].append("scene_description_generation")
        
        results["assets"]["scene_descriptions"] = scene_descriptions
        
        # ====================================================================
        # STEP 5: Generate Scene Images
        # ====================================================================
        # In CompleteAdProductionPipeline.run_complete_pipeline()

# STEP 6: Generate Scene Images (Updated)
        if generate_scene_images and scene_descriptions:
            print("\n" + "─"*100)
            print("STEP 6: SCENE IMAGE GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("generation_report"):
                print("🔄 Generating scene images...")
                
               
                
                scene_image_generator = SceneImageGenerator(output_dir=self.dirs["scenes"])
                
                # Prepare product info
                product_info = None
                if product_image_path and os.path.exists(product_image_path):
                    product_info = {
                        "name": brand_info.get("product_name", "product"),
                        "image_path": product_image_path
                    }
                    print("Character_info")
                    print(results["assets"].get("characters", []))

                    print("Location Info")
                    print(results["assets"].get("locations", []))

                    print("Outfit Info")
                    print(results["assets"].get("outfits", []))
                
                # Generate all scene images with outfit references
                generation_results = scene_image_generator.generate_all_scene_images(
                    scene_description=scene_descriptions,
                    characters_info=results["assets"].get("characters", []),
                    locations_info=results["assets"].get("locations", []),
                    outfits_info=results["assets"].get("outfits", []),  # NEW: Pass outfits
                    product_info=product_info,
                    aspect_ratio=aspect_ratio,
                    project_id=self.project_id
                )
                
                # Save generation report
                report_path = scene_image_generator.save_generation_report(
                    f"{self.project_id}_scene_generation_report.json",
                    self.dirs["scripts"]
                )
                
                results["files"]["generation_report"] = report_path
                results["assets"]["scene_images"] = generation_results
                
                self._mark_stage_completed("generation_report")
                results["stages_executed"].append("scene_image_generation")
            else:
                print("✅ Scene images already generated, loading report...")
                report_data = self._load_json(self.stage_files["generation_report"])
                
                if report_data:
                    results["files"]["generation_report"] = self.stage_files["generation_report"]
                    results["assets"]["scene_images"] = report_data
                    results["stages_skipped"].append("scene_image_generation")
                else:
                    print("⚠ Failed to load existing generation report")

        # ====================================================================
        # STEP 7: Generate Video Descriptions (NEW)
        # ====================================================================
        print("\n" + "─"*100)
        print("STEP 7: VIDEO DESCRIPTION GENERATION")
        print("─"*100)

        video_descriptions = None

        if force_regenerate or not self._stage_completed("video_descriptions"):
            print("🔄 Generating video descriptions...")
            

            
            video_desc_generator = VideoDescriptionGenerator()
            video_descriptions = video_desc_generator.generate_video_descriptions(
                shots=[shot.model_dump() for shot in shot_script.shots],
                ad_title=shot_script.ad_title,
                enable_animation_for_finale=True  # Can be parameterized
            )
            
            # Save video descriptions
            video_desc_json = video_desc_generator.save_video_descriptions(
                video_descriptions,
                f"{self.project_id}_video_descriptions.json",
                self.dirs["prompts"]
            )
            video_desc_txt = video_desc_generator.export_video_prompts_readable(
                video_descriptions,
                f"{self.project_id}_video_prompts.txt",
                self.dirs["prompts"]
            )
            
            results["files"]["video_descriptions"] = video_desc_json
            results["files"]["video_prompts_txt"] = video_desc_txt
            
            self._mark_stage_completed("video_descriptions")
            results["stages_executed"].append("video_description_generation")
        else:
            print("✅ Video descriptions already exist, loading...")
            # Load existing video descriptions
            video_desc_data = self._load_json(self.stage_files["video_descriptions"])
            if video_desc_data:

                video_descriptions = VideoDescription(**video_desc_data)
                results["stages_skipped"].append("video_description_generation")

        results["assets"]["video_descriptions"] = video_descriptions


        # ====================================================================
        # STEP 8: Generate Videos (NEW)
        # ====================================================================
        if generate_videos and video_descriptions:  # Add generate_videos parameter
            print("\n" + "─"*100)
            print("STEP 8: VIDEO GENERATION")
            print("─"*100)
            
            if force_regenerate or not self._stage_completed("video_generation_report"):
                print("🔄 Generating videos...")
                
                
                
                video_generator = VideoGenerator(
                    output_dir=self.dirs["videos"],
                    aspect_ratio=aspect_ratio
                )
                
                # Build scene images mapping from generation report
                scene_images = {}
                if results["assets"].get("scene_images"):
                    for shot_id, img_data in results["assets"]["scene_images"]["generated_images"].items():
                        shot_no = img_data.get("shot_no")
                        filepath = img_data.get("filepath")
                        if shot_no and filepath:
                            scene_images[shot_no] = filepath
                
                # Generate all videos
                video_progress = video_generator.generate_all_videos(
                    shots=[shot.model_dump() for shot in shot_script.shots],
                    video_prompts=[vp.model_dump() for vp in video_descriptions.video_prompts],
                    scene_images=scene_images,
                    ad_title=shot_script.ad_title,
                    project_id=self.project_id,
                    delay_between_videos=5.0
                )
                
                # Save video generation report
                video_report_path = video_generator.save_generation_report(
                    f"{self.project_id}_video_generation_report.json",
                    self.dirs["scripts"]
                )
                
                results["files"]["video_generation_report"] = video_report_path
                results["assets"]["generated_videos"] = video_progress
                
                self._mark_stage_completed("video_generation_report")
                results["stages_executed"].append("video_generation")
            else:
                print("✅ Videos already generated")
                results["stages_skipped"].append("video_generation")
                
        # ====================================================================
        # Final Summary
        # ====================================================================
        print("\n" + "="*100)
        print("🎉 PIPELINE COMPLETE!")
        print("="*100)
        print(f"\n📁 Project Directory: {self.dirs['project']}")
        
        if results["stages_executed"]:
            print(f"\n✅ Stages Executed ({len(results['stages_executed'])}):")
            for stage in results["stages_executed"]:
                print(f"  - {stage}")
        
        if results["stages_skipped"]:
            print(f"\n⏭️  Stages Skipped ({len(results['stages_skipped'])}):")
            for stage in results["stages_skipped"]:
                print(f"  - {stage}")
        
        print(f"\n📄 Generated/Loaded Files:")
        for key, path in results["files"].items():
            print(f"  - {key}: {path}")
        
        return results
    
    def reset_stage(self, stage: str):
        """
        Reset a specific stage by deleting its output files
        
        Args:
            stage: Stage name (shot_script, characters, locations, scene_descriptions, generation_report)
        """
        if stage not in self.stage_files:
            print(f"⚠ Unknown stage: {stage}")
            return
        
        filepath = self.stage_files[stage]
        
        if os.path.exists(filepath):
            os.remove(filepath)
            print(f"🗑️  Deleted: {filepath}")
            self.completed_stages[stage] = False
        else:
            print(f"⚠ File not found: {filepath}")
    
    def reset_all_stages(self):
        """Reset all stages"""
        print("🗑️  Resetting all stages...")
        for stage in self.stage_files.keys():
            self.reset_stage(stage)
        print("✅ All stages reset")
    
    def get_pipeline_status(self) -> Dict[str, Any]:
        """Get current status of all pipeline stages"""
        self._check_existing_stages()
        
        status = {
            "project_id": self.project_id,
            "completed_stages": [],
            "pending_stages": [],
            "stage_details": {}
        }
        
        for stage, completed in self.completed_stages.items():
            if completed:
                status["completed_stages"].append(stage)
                status["stage_details"][stage] = {
                    "status": "completed",
                    "file": self.stage_files[stage]
                }
            else:
                status["pending_stages"].append(stage)
                status["stage_details"][stage] = {
                    "status": "pending",
                    "file": self.stage_files[stage]
                }
        
        return status


# Example usage
if __name__ == "__main__":
    # Example ad concept
    ad_concept = {
    "title": "Did I Apply Sunscreen?",
    "one_line_summary": "A playful 15s routine where a woman 25-26 yrs old keeps re-checking whether she applied sunscreen because it's so weightless and invisible.",
    "story": "A young woman moves through her day under strong sunlight. Deconstruct’s gel sunscreen is so light that she keeps forgetting she already applied it. Her internal monologue becomes the comedic hook.",
    "visual_flow": {
        "Opening": "Morning street scene – she steps out, touches her face, thinks: 'Did I apply sunscreen?' A quick visual flashback shows her applying it earlier.",
        "Sequence": "Afternoon beach – sun is harsher. She pauses mid-activity, touches her cheek again, internal thought repeats: 'Wait… did I apply it?' Another quick flashback.",
        "Evening": "Outdoor café at sunset – warm glow. She smiles to herself this time before the thought arises again. Final humorous freeze: She *knows* she applied it, but still checks."
    },
    "voice_over": "So light, you might forget you applied it.",
    "tagline": "So light, you’ll forget.",
    "key_message": "Invisible protection that never feels heavy.",
    "key_features": ["SPF 55+", "PA+++", "Weightless gel texture", "No white cast", "Sweat-resistant"]
}

    
    # Example brand info
    brand_info = {
        "brand_name": "Deconstruct",
        "product_name": "Gel Sunscreen SPF 55",
        "product_description": "Lightweight, matte finish sunscreen",
        "key_features": ["SPF 55+", "PA+++", "Matte finish", "Sweat-resistant"]
    }
    
    # Initialize pipeline
    pipeline = CompleteAdProductionPipeline(project_id="deconstruct_light_weight_single_character_004")
    
    # Check current status
    status = pipeline.get_pipeline_status()
    print("\n📊 Current Pipeline Status:")
    print(f"Completed: {status['completed_stages']}")
    print(f"Pending: {status['pending_stages']}")
    
    # Run pipeline (will skip completed stages automatically)
    results = pipeline.run_complete_pipeline(
        ad_concept=ad_concept,
        brand_info=brand_info,
        product_image_path="/Users/sanjail/Akaike/Internal_project/ads_poc/product_image.png",
        target_duration="45 seconds",
        aspect_ratio="16:9",
        generate_character_images=True,
        generate_location_images=True,
        generate_outfit_images=True,
        generate_scene_images=True,
        generate_videos=True,
        force_regenerate=False  # Set to True to regenerate everything
    )
    
    print("\n✅ Pipeline execution complete!")
    
    # Example: Reset a specific stage if you want to regenerate it
    # pipeline.reset_stage("scene_descriptions")
    
    # Example: Reset all stages
    # pipeline.reset_all_stages()

✅ Initialized project: deconstruct_light_weight_single_character_004

📋 Checking existing stages...
  shot_script: ❌ PENDING
  shot_script_complete: ❌ PENDING
  characters: ❌ PENDING
  locations: ❌ PENDING
  outfits: ❌ PENDING
  scene_descriptions: ❌ PENDING
  generation_report: ❌ PENDING
  video_descriptions: ❌ PENDING
  video_generation_report: ❌ PENDING

📋 Checking existing stages...
  shot_script: ❌ PENDING
  shot_script_complete: ❌ PENDING
  characters: ❌ PENDING
  locations: ❌ PENDING
  outfits: ❌ PENDING
  scene_descriptions: ❌ PENDING
  generation_report: ❌ PENDING
  video_descriptions: ❌ PENDING
  video_generation_report: ❌ PENDING

📊 Current Pipeline Status:
Completed: []
Pending: ['shot_script', 'shot_script_complete', 'characters', 'locations', 'outfits', 'scene_descriptions', 'generation_report', 'video_descriptions', 'video_generation_report']

🎬 STARTING COMPLETE AD PRODUCTION PIPELINE
Project: deconstruct_light_weight_single_character_004
Force Regenerate: False


─────

KeyboardInterrupt: 